#  n_gauss=3

### 1. 筛选3sigma

In [1]:
import numpy as np
from astropy.io import fits
import matplotlib.pyplot as plt
from ROHSApy import ROHSA
from scipy import ndimage
from mpl_toolkits.axes_grid1 import make_axes_locatable
import os
import warnings
warnings.filterwarnings('ignore')

fitsname = "CRAFTS_-4.7_-350_-150_Original.fits"
hdu = fits.open(fitsname)
hdr = hdu[0].header
cube = hdu[0].data

In [2]:
core = ROHSA(cube)    
core.hdr = hdr

In [36]:
from scipy import ndimage
from mpl_toolkits.axes_grid1 import make_axes_locatable
import os
import warnings
warnings.filterwarnings('ignore')



# ==================== 1. 读取ROHSA结果（同时保留像素单位和物理单位） ====================
def read_rohsa_results(gaussian_file):
    """
    读取ROHSA的输出结果(.dat格式)
    返回像素单位和物理单位的结果
    """
    print("\n" + "="*60)
    print("READING ROHSA RESULTS")
    print("="*60)
    
    # 使用core模块读取像素单位的gaussian
    print("Reading Gaussian parameters (pixel units)...")
    gaussian_pixel = core.read_gaussian(gaussian_file)
    
    print(f"Gaussian pixel array shape: {gaussian_pixel.shape}")
    print(f"Data type: {gaussian_pixel.dtype}")
    model = core.return_result_cube(gaussian=gaussian_pixel)
    
    # 转换为物理单位
    print("Converting to physical units...")
    gaussian_physical = core.physical_gaussian(gaussian_pixel)
    
    # 解析维度: (n_parameters * n_components, n_y, n_x)
    n_params_times_comp, n_y, n_x = gaussian_pixel.shape
    n_components = n_params_times_comp // 3
    
    print(f"Spatial dimensions: {n_x} × {n_y}")
    print(f"Number of Gaussian components: {n_components}")
    
    # 提取像素单位的参数
    amplitude_pixel = gaussian_pixel[0::3].transpose(2, 1, 0)  # [n_x, n_y, n_comp]
    position_pixel = gaussian_pixel[1::3].transpose(2, 1, 0)   # [n_x, n_y, n_comp]
    dispersion_pixel = gaussian_pixel[2::3].transpose(2, 1, 0) # [n_x, n_y, n_comp]
    
    # 提取物理单位的参数
    amplitude_phys = gaussian_physical[0::3].transpose(2, 1, 0)  # [n_x, n_y, n_comp]
    position_phys = gaussian_physical[1::3].transpose(2, 1, 0)   # [n_x, n_y, n_comp]
    dispersion_phys = gaussian_physical[2::3].transpose(2, 1, 0) # [n_x, n_y, n_comp]
    
    # 清理无效值
    amplitude_pixel = np.nan_to_num(amplitude_pixel)
    position_pixel = np.nan_to_num(position_pixel)
    dispersion_pixel = np.nan_to_num(np.abs(dispersion_pixel))
    
    amplitude_phys = np.nan_to_num(amplitude_phys)
    position_phys = np.nan_to_num(position_phys)
    dispersion_phys = np.nan_to_num(np.abs(dispersion_phys))
    
    print(f"\nPixel units statistics:")
    for comp in range(n_components):
        amp_comp = amplitude_pixel[:, :, comp]
        amp_nonzero = amp_comp[amp_comp > 0]
        if len(amp_nonzero) > 0:
            print(f"  Component {comp+1} amplitude (pixel): range [{np.min(amp_nonzero):.4f}, {np.max(amp_comp):.4f}]")
    
    print(f"\nPhysical units statistics:")
    for comp in range(n_components):
        amp_comp = amplitude_phys[:, :, comp]
        amp_nonzero = amp_comp[amp_comp > 0]
        if len(amp_nonzero) > 0:
            print(f"  Component {comp+1} amplitude (K): range [{np.min(amp_nonzero):.4f}, {np.max(amp_comp):.4f}]")
    
    return {
        'pixel': {
            'amplitude': amplitude_pixel,
            'position': position_pixel,
            'dispersion': dispersion_pixel,
        },
        'physical': {
            'amplitude': amplitude_phys,
            'position': position_phys,
            'dispersion': dispersion_phys,
        },
        'n_components': n_components,
        'shape': (n_x, n_y),
        'original_pixel': gaussian_pixel,
        'original_physical': gaussian_physical
    }

# ==================== 2. 读取原始FITS数据 ====================
def read_original_fits(fits_file):
    """读取原始FITS文件"""
    print("\n" + "="*60)
    print("READING ORIGINAL FITS")
    print("="*60)
    
    try:
        hdu = fits.open(fits_file)[0]
        cube = hdu.data
        header = hdu.header
        
        print(f"Original cube shape: {cube.shape}")
        print(f"Data range: [{np.min(cube):.4f}, {np.max(cube):.4f}]")
        
        # 获取速度信息（如果有）
        if 'CRVAL3' in header and 'CDELT3' in header and 'NAXIS3' in header:
            v0 = header['CRVAL3']
            dv = header['CDELT3']
            n_channels = header['NAXIS3']
            velocities = v0 + np.arange(n_channels) * dv
            print(f"Velocity range: [{velocities[0]:.2f}, {velocities[-1]:.2f}] km/s")
        else:
            velocities = None
            print("No velocity information in header")
        
        return cube, header, velocities
        
    except Exception as e:
        print(f"Error reading FITS file: {e}")
        return None, None, None

# ==================== 3. 从数据计算每个像素的噪声 ====================
def calculate_per_pixel_noise(original_cube, output_dir, noise_channels=None, velocities=None):
    """
    从数据立方体中计算每个像素的噪声
    """
    print("\n" + "="*60)
    print("CALCULATING PER-PIXEL NOISE FROM DATA")
    print("="*60)
    
    if original_cube is None:
        print("No original cube data available")
        return None, None
    
    n_channels, n_y, n_x = original_cube.shape
    
    if noise_channels is None:
        # 自动选择：前10%和后10%的通道作为噪声
        n_noise = max(10, int(n_channels * 0.1))
        noise_channels_indices = [0, n_noise, n_channels - n_noise, n_channels]
        print(f"Auto-selected noise channels: indices {noise_channels_indices}")
    else:
        noise_channels_indices = noise_channels
        print(f"Using specified noise channels: {noise_channels_indices}")
    
    # 提取噪声通道的数据
    start1, end1, start2, end2 = noise_channels_indices
    noise_data1 = original_cube[start1:end1, :, :]
    noise_data2 = original_cube[start2:end2, :, :]
    noise_data = np.concatenate([noise_data1, noise_data2], axis=0)
    
    print(f"Using {len(noise_data)} channels for noise calculation")
    
    # 计算每个像素的RMS
    noise_rms_map = np.std(noise_data, axis=0)  # [n_y, n_x]
    global_noise_rms = np.mean(noise_rms_map)
    
    print(f"\nPer-pixel noise statistics (K):")
    print(f"  Global mean RMS: {global_noise_rms:.4f}")
    print(f"  RMS range: [{np.min(noise_rms_map):.4f}, {np.max(noise_rms_map):.4f}]")
    print(f"  RMS median: {np.median(noise_rms_map):.4f}")
    
    # 显示噪声分布的直方图
    plt.figure(figsize=(10, 6))
    plt.hist(noise_rms_map.flatten(), bins=50, alpha=0.7, color='steelblue', edgecolor='black')
    plt.axvline(global_noise_rms, color='red', linestyle='--', linewidth=2, 
                label=f'Mean: {global_noise_rms:.4f} K')
    plt.axvline(np.median(noise_rms_map), color='green', linestyle='--', linewidth=2,
                label=f'Median: {np.median(noise_rms_map):.4f} K')
    plt.xlabel('Noise RMS [K]', fontsize=12)
    plt.ylabel('Frequency', fontsize=12)
    plt.title('Distribution of Per-Pixel Noise RMS', fontsize=14)
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    
    # 保存直方图
    hist_file = os.path.join(output_dir, 'noise_distribution.png')
    plt.savefig(hist_file, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Noise distribution histogram saved to: {hist_file}")
    
    return noise_rms_map, global_noise_rms

# ==================== 4. 对每个成分单独进行振幅筛选 ====================
def filter_each_component_with_per_pixel_noise(rohsa_results, noise_rms_map, snr_threshold=2.0):
    """
    对每个高斯成分单独进行振幅筛选
    amplitude < snr_threshold * noise_at_this_pixel 的像素设为0
    
    注意：筛选基于物理单位的振幅，但返回的mask应用于像素单位的数据
    """
    print("\n" + "="*60)
    print("FILTERING EACH COMPONENT WITH PER-PIXEL NOISE")
    print("="*60)
    print(f"SNR threshold: {snr_threshold}σ")
    
    if noise_rms_map is None:
        print("No noise map available, skipping filtering")
        return rohsa_results, None, None
    
    # 使用物理单位的振幅进行SNR计算
    amplitude_phys = rohsa_results['physical']['amplitude'].copy()  # [n_x, n_y, n_comp] 单位: K
    n_components = rohsa_results['n_components']
    n_x, n_y, _ = amplitude_phys.shape
    
    # 扩展噪声图以匹配振幅的维度
    noise_expanded = noise_rms_map.T[:, :, np.newaxis]  # [n_x, n_y, 1]
    
    # 计算每个像素、每个分量的SNR
    with np.errstate(divide='ignore', invalid='ignore'):
        snr_maps = amplitude_phys / (noise_expanded + 1e-10)
        snr_maps = np.nan_to_num(snr_maps)
    
    print(f"\nSNR statistics before filtering:")
    for comp in range(n_components):
        comp_snr = snr_maps[:, :, comp]
        valid_snr = comp_snr[amplitude_phys[:, :, comp] > 0]
        if len(valid_snr) > 0:
            print(f"  Component {comp+1}: SNR range [{np.min(valid_snr):.2f}, {np.max(valid_snr):.2f}], "
                  f"mean {np.mean(valid_snr):.2f}")
    
    # 生成每个成分的mask（基于物理单位的SNR）
    masks = snr_maps >= snr_threshold  # [n_x, n_y, n_comp]
    
    print(f"\nPer component statistics after filtering:")
    for comp in range(n_components):
        orig_pixels = np.sum(amplitude_phys[:, :, comp] > 0)
        keep_pixels = np.sum(masks[:, :, comp])
        
        if orig_pixels > 0:
            retention = 100 * keep_pixels / orig_pixels
        else:
            retention = 0
            
        percent_total = 100 * keep_pixels / (n_x * n_y)
        print(f"  Component {comp+1}:")
        print(f"    Original non-zero: {orig_pixels}")
        print(f"    Kept: {keep_pixels} ({percent_total:.2f}% of total, {retention:.2f}% of original)")
        
        # 计算保留像素的最小SNR
        kept_snr = snr_maps[:, :, comp][masks[:, :, comp]]
        if len(kept_snr) > 0:
            print(f"    Kept SNR range: [{np.min(kept_snr):.2f}, {np.max(kept_snr):.2f}]")
    
    # 创建筛选后的结果（包含像素单位和物理单位）
    filtered_results = {
        'pixel': {
            'amplitude': rohsa_results['pixel']['amplitude'].copy(),
            'position': rohsa_results['pixel']['position'].copy(),
            'dispersion': rohsa_results['pixel']['dispersion'].copy(),
        },
        'physical': {
            'amplitude': rohsa_results['physical']['amplitude'].copy(),
            'position': rohsa_results['physical']['position'].copy(),
            'dispersion': rohsa_results['physical']['dispersion'].copy(),
        },
        'n_components': n_components,
        'shape': (n_x, n_y)
    }
    
    # 应用mask到像素单位和物理单位的数据
    for comp in range(n_components):
        comp_mask = masks[:, :, comp]
        
        # 像素单位
        filtered_results['pixel']['amplitude'][~comp_mask, comp] = 0
        filtered_results['pixel']['position'][~comp_mask, comp] = 0
        filtered_results['pixel']['dispersion'][~comp_mask, comp] = 0
        
        # 物理单位
        filtered_results['physical']['amplitude'][~comp_mask, comp] = 0
        filtered_results['physical']['position'][~comp_mask, comp] = 0
        filtered_results['physical']['dispersion'][~comp_mask, comp] = 0
    
    return filtered_results, masks, snr_maps

# ==================== 5. 保存筛选后的ROHSA结果到FITS（物理单位） ====================
def save_filtered_rohsa_fits(filtered_results, output_file):
    """保存筛选后的ROHSA结果到FITS文件（使用物理单位）"""
    try:
        n_components = filtered_results['n_components']
        n_x, n_y = filtered_results['shape']
        
        # 重新组合成原始格式 [3*n_components, n_y, n_x]
        combined = np.zeros((3*n_components, n_y, n_x))
        
        for comp in range(n_components):
            # 使用物理单位保存到FITS
            combined[3*comp] = filtered_results['physical']['amplitude'][:, :, comp].T
            combined[3*comp + 1] = filtered_results['physical']['position'][:, :, comp].T
            combined[3*comp + 2] = filtered_results['physical']['dispersion'][:, :, comp].T
        
        # 保存为FITS
        hdu = fits.PrimaryHDU(combined)
        hdu.writeto(output_file, overwrite=True)
        
        print(f"\nFiltered ROHSA results saved to FITS (physical units): {output_file}")
        return True
    except Exception as e:
        print(f"Error saving filtered results to FITS: {e}")
        return False

# ==================== 6. 保存筛选后的ROHSA结果到DAT（像素单位，与原始格式一致） ====================
def save_filtered_rohsa_dat(filtered_results, original_dat_file, output_file):
    """
    保存筛选后的ROHSA结果到.dat文件，使用像素单位，与原始格式完全一致
    注意：在.dat文件中，顺序是 y x amplitude position dispersion
    """
    print("\n" + "="*60)
    print("SAVING FILTERED RESULTS TO .DAT FORMAT (PIXEL UNITS)")
    print("="*60)
    
    try:
        n_components = filtered_results['n_components']
        n_x, n_y = filtered_results['shape']  # n_x是x维度，n_y是y维度
        
        print(f"  Spatial grid: X({n_x}) × Y({n_y})")
        print(f"  Components: {n_components}")
        print(f"  DAT file format: Y X AMP POS DISP (Y first, X second)")
        print(f"  Units: Pixel units (matching original ROHSA format)")
        
        # 统计非零像素（基于像素单位）
        total_entries = n_x * n_y * n_components
        nonzero_entries = np.sum(filtered_results['pixel']['amplitude'] != 0)
        print(f"  Non-zero entries: {nonzero_entries} ({100 * nonzero_entries / total_entries:.2f}%)")
        
        # 读取原始文件的头部信息
        header_lines = []
        try:
            with open(original_dat_file, 'r') as f:
                for i in range(27):  # 读取前27行作为头部
                    line = f.readline()
                    header_lines.append(line)
            print(f"  Read {len(header_lines)} header lines from original file")
        except Exception as e:
            print(f"  Warning: Could not read original file header: {e}")
            print("  Using default header")
            header_lines = [
                f"# ROHSA filtered results (pixel units)\n",
                f"# Number of components: {n_components}\n",
                f"# Grid size: X={n_x}, Y={n_y}\n",
                f"# SNR threshold applied: Yes\n",
                f"# Non-zero entries: {nonzero_entries} / {total_entries}\n",
                f"#\n",
                f"# Column 1: Y (pixel)\n",
                f"# Column 2: X (pixel)\n",
                f"# Column 3: Amplitude (pixel units)\n",
                f"# Column 4: Position (pixel units)\n",
                f"# Column 5: Dispersion (pixel units)\n",
                f"#\n"
            ]
        
        # 写入新的.dat文件（使用像素单位的数据）
        with open(output_file, 'w') as f:
            # 写入头部
            f.writelines(header_lines)
            
            # 写入数据 - 注意顺序：先Y后X
            line_count = 0
            for j in range(n_y):  # Y坐标（外层循环）
                for i in range(n_x):  # X坐标（内层循环）
                    for comp in range(n_components):
                        # 使用像素单位的数据
                        amp = filtered_results['pixel']['amplitude'][i, j, comp]
                        pos = filtered_results['pixel']['position'][i, j, comp]
                        dis = filtered_results['pixel']['dispersion'][i, j, comp]
                        
                        # 使用与原始文件完全相同的格式
                        line = f"    {j:4d}    {i:4d}    {amp:20.16f}    {pos:20.16f}    {dis:20.16f}\n"
                        f.write(line)
                        line_count += 1
            
            print(f"  Total lines written: {line_count}")
            print(f"  Data format: Y(4d) X(4d) AMP(20.16f) POS(20.16f) DISP(20.16f)")
        
        print(f"  DAT file saved to: {output_file}")
        
        # 验证前几行
        print(f"\n  First few lines of saved file:")
        with open(output_file, 'r') as f:
            for i, line in enumerate(f):
                if i >= 5:  # 显示前5行数据
                    break
                if i >= len(header_lines):  # 跳过头部
                    print(f"    {line.strip()}")
        
        return True
        
    except Exception as e:
        print(f"Error saving filtered results to DAT: {e}")
        import traceback
        traceback.print_exc()
        return False

# ==================== 7. 保存筛选后的ROHSA结果（同时保存FITS和DAT） ====================
def save_filtered_rohsa_all(filtered_results, original_dat_file, output_base):
    """
    保存筛选后的ROHSA结果
    - FITS: 物理单位
    - DAT: 像素单位（与原始格式一致）
    """
    print("\n" + "="*60)
    print("SAVING FILTERED ROHSA RESULTS")
    print("="*60)
    
    # 保存FITS（物理单位）
    fits_file = output_base + '.fits'
    save_filtered_rohsa_fits(filtered_results, fits_file)
    
    # 保存DAT（像素单位）
    dat_file = output_base + '.dat'
    save_filtered_rohsa_dat(filtered_results, original_dat_file, dat_file)
    
    return True

# ==================== 8. 绘制每个成分的对比图（物理单位） ====================
def plot_component_comparison(original_results, filtered_results, noise_map, comp_index, output_dir, snr_threshold):
    """
    为单个成分绘制对比图
    所有数值都是物理单位用于显示
    """
    # 获取该成分的数据（物理单位用于显示）
    orig_amp = original_results['physical']['amplitude'][:, :, comp_index].T  # [n_y, n_x]
    orig_pos = original_results['physical']['position'][:, :, comp_index].T
    orig_dis = original_results['physical']['dispersion'][:, :, comp_index].T
    
    filt_amp = filtered_results['physical']['amplitude'][:, :, comp_index].T
    filt_pos = filtered_results['physical']['position'][:, :, comp_index].T
    filt_dis = filtered_results['physical']['dispersion'][:, :, comp_index].T
    
    print(f"\nComponent {comp_index+1} plot ranges (physical units):")
    
    # 确定每个参数的统一范围（只考虑非零值）
    amp_mask = orig_amp > 0
    if np.any(amp_mask):
        amp_vmin = np.min(orig_amp[amp_mask])
        amp_vmax = np.max(orig_amp)
        print(f"  Amplitude range: [{amp_vmin:.4f}, {amp_vmax:.4f}] K")
    else:
        amp_vmin, amp_vmax = 0, 1
        print("  No valid amplitude values")
    
    pos_mask = (orig_pos != 0) & (~np.isnan(orig_pos))
    if np.any(pos_mask):
        pos_vmin = np.min(orig_pos[pos_mask])
        pos_vmax = np.max(orig_pos)
        print(f"  Position range: [{pos_vmin:.4f}, {pos_vmax:.4f}] km/s")
    else:
        pos_vmin, pos_vmax = -10, 10
        print("  No valid position values")
    
    dis_mask = orig_dis > 0
    if np.any(dis_mask):
        dis_vmin = np.min(orig_dis[dis_mask])
        dis_vmax = np.max(orig_dis)
        print(f"  Dispersion range: [{dis_vmin:.4f}, {dis_vmax:.4f}] km/s")
    else:
        dis_vmin, dis_vmax = 0, 10
        print("  No valid dispersion values")
    
    # 创建2×3子图
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    
    # ===== 第一行：原始参数 =====
    # 原始振幅
    im1 = axes[0, 0].imshow(orig_amp, origin='lower', cmap='inferno', vmin=amp_vmin, vmax=amp_vmax)
    axes[0, 0].set_title(f'Component {comp_index+1} - Original Amplitude', fontsize=12)
    cbar1 = plt.colorbar(im1, ax=axes[0, 0], shrink=0.8, pad=0.05, location='bottom')
    cbar1.set_label('Brightness Temp. [K]', fontsize=10)
    
    # 原始位置
    im2 = axes[0, 1].imshow(orig_pos, origin='lower', cmap='coolwarm', vmin=pos_vmin, vmax=pos_vmax)
    axes[0, 1].set_title(f'Component {comp_index+1} - Original Position', fontsize=12)
    cbar2 = plt.colorbar(im2, ax=axes[0, 1], shrink=0.8, pad=0.05, location='bottom')
    cbar2.set_label('Velocity [km s$^{-1}$]', fontsize=10)
    
    # 原始宽度
    im3 = axes[0, 2].imshow(orig_dis, origin='lower', cmap='cubehelix', vmin=dis_vmin, vmax=dis_vmax)
    axes[0, 2].set_title(f'Component {comp_index+1} - Original Dispersion', fontsize=12)
    cbar3 = plt.colorbar(im3, ax=axes[0, 2], shrink=0.8, pad=0.05, location='bottom')
    cbar3.set_label('Dispersion [km s$^{-1}$]', fontsize=10)
    
    # ===== 第二行：筛选后参数（使用相同范围）=====
    # 筛选后振幅
    im4 = axes[1, 0].imshow(filt_amp, origin='lower', cmap='inferno', vmin=amp_vmin, vmax=amp_vmax)
    axes[1, 0].set_title(f'Component {comp_index+1} - Filtered Amplitude', fontsize=12)
    cbar4 = plt.colorbar(im4, ax=axes[1, 0], shrink=0.8, pad=0.05, location='bottom')
    cbar4.set_label('Brightness Temp. [K]', fontsize=10)
    
    # 筛选后位置
    im5 = axes[1, 1].imshow(filt_pos, origin='lower', cmap='coolwarm', vmin=pos_vmin, vmax=pos_vmax)
    axes[1, 1].set_title(f'Component {comp_index+1} - Filtered Position', fontsize=12)
    cbar5 = plt.colorbar(im5, ax=axes[1, 1], shrink=0.8, pad=0.05, location='bottom')
    cbar5.set_label('Velocity [km s$^{-1}$]', fontsize=10)
    
    # 筛选后宽度
    im6 = axes[1, 2].imshow(filt_dis, origin='lower', cmap='cubehelix', vmin=dis_vmin, vmax=dis_vmax)
    axes[1, 2].set_title(f'Component {comp_index+1} - Filtered Dispersion', fontsize=12)
    cbar6 = plt.colorbar(im6, ax=axes[1, 2], shrink=0.8, pad=0.05, location='bottom')
    cbar6.set_label('Dispersion [km s$^{-1}$]', fontsize=10)
    
    # 添加总标题
    plt.suptitle(f'Component {comp_index+1}: Original vs Filtered (Per-pixel {snr_threshold}σ threshold)', 
                fontsize=14, y=1.02)
    
    plt.tight_layout()
    
    # 保存图片
    comp_file = os.path.join(output_dir, f'component{comp_index+1}_comparison.png')
    plt.savefig(comp_file, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Component {comp_index+1} comparison plot saved to: {comp_file}")


# ==================== 9. 绘制所有mask和SNR图 ====================
def plot_masks_and_snr(masks, snr_maps, noise_map, output_dir, snr_threshold):
    """绘制所有mask和SNR图"""
    if masks is None or snr_maps is None:
        print("No masks or SNR maps to plot")
        return
    
    n_components = masks.shape[2]
    n_x, n_y, _ = masks.shape
    
    # 绘制所有mask
    fig, axes = plt.subplots(1, n_components, figsize=(5*n_components, 4))
    if n_components == 1:
        axes = [axes]
    
    for comp in range(n_components):
        im = axes[comp].imshow(masks[:, :, comp].T, origin='lower', cmap='gray', aspect='auto')
        axes[comp].set_title(f'Component {comp+1} Mask', fontsize=12)
        axes[comp].set_xlabel('X pixel')
        axes[comp].set_ylabel('Y pixel')
        divider = make_axes_locatable(axes[comp])
        cax = divider.append_axes("right", size="5%", pad=0.05)
        cbar = plt.colorbar(im, cax=cax)
        cbar.set_label('Keep (1) / Remove (0)')
        
        # 添加统计信息
        keep_pixels = np.sum(masks[:, :, comp])
        total_pixels = n_x * n_y
        axes[comp].text(0.02, 0.98, f'Keep: {keep_pixels} ({100*keep_pixels/total_pixels:.1f}%)', 
                       transform=axes[comp].transAxes, fontsize=10,
                       verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    plt.suptitle(f'Component Masks (Per-pixel {snr_threshold}σ threshold)', fontsize=14)
    plt.tight_layout()
    
    mask_file = os.path.join(output_dir, 'all_masks.png')
    plt.savefig(mask_file, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"All masks plot saved to: {mask_file}")
    
    # 绘制SNR图
    fig, axes = plt.subplots(1, n_components, figsize=(5*n_components, 4))
    if n_components == 1:
        axes = [axes]
    
    # 确定SNR的统一范围
    valid_snr = snr_maps[snr_maps > 0]
    if len(valid_snr) > 0:
        snr_max = np.percentile(valid_snr, 95)
    else:
        snr_max = 10
    snr_min = 0
    
    for comp in range(n_components):
        im = axes[comp].imshow(snr_maps[:, :, comp].T, origin='lower', cmap='hot', 
                              vmin=snr_min, vmax=snr_max, aspect='auto')
        axes[comp].set_title(f'Component {comp+1} SNR', fontsize=12)
        axes[comp].set_xlabel('X pixel')
        axes[comp].set_ylabel('Y pixel')
        divider = make_axes_locatable(axes[comp])
        cax = divider.append_axes("right", size="5%", pad=0.05)
        cbar = plt.colorbar(im, cax=cax)
        cbar.set_label('SNR')
        
        # 标记阈值线
        if masks is not None:
            axes[comp].contour(masks[:, :, comp].T, levels=[0.5], colors='cyan', linewidths=0.5, alpha=0.5)
    
    plt.suptitle(f'Component SNR Maps (Per-pixel noise, threshold={snr_threshold}σ)', fontsize=14)
    plt.tight_layout()
    
    snr_file = os.path.join(output_dir, 'all_snr_maps.png')
    plt.savefig(snr_file, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"All SNR maps saved to: {snr_file}")

# ==================== 10. 绘制噪声图 ====================
def plot_noise_map(noise_map, output_dir):
    """绘制噪声图"""
    if noise_map is None:
        print("No noise map to plot")
        return
    
    plt.figure(figsize=(10, 8))
    im = plt.imshow(noise_map, origin='lower', cmap='hot', aspect='auto')
    plt.colorbar(im, label='Noise RMS [K]', shrink=0.8)
    plt.title('Per-Pixel Noise RMS Map', fontsize=14)
    plt.xlabel('X pixel', fontsize=12)
    plt.ylabel('Y pixel', fontsize=12)
    plt.tight_layout()
    
    noise_file = os.path.join(output_dir, 'noise_map.png')
    plt.savefig(noise_file, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Noise map saved to: {noise_file}")

# ==================== 11. 保存统计信息到文本文件 ====================
def save_statistics(original_results, filtered_results, masks, noise_map, snr_threshold, output_dir):
    """保存统计信息到文本文件"""
    n_components = original_results['n_components']
    n_x, n_y = original_results['shape']
    total_pixels = n_x * n_y
    
    stats_file = os.path.join(output_dir, 'statistics.txt')
    
    with open(stats_file, 'w') as f:
        f.write("="*60 + "\n")
        f.write("FILTERING STATISTICS\n")
        f.write("="*60 + "\n\n")
        
        f.write(f"Total pixels: {total_pixels}\n")
        f.write(f"SNR threshold: {snr_threshold}σ (per-pixel)\n")
        if noise_map is not None:
            f.write(f"Noise RMS range: [{np.min(noise_map):.4f}, {np.max(noise_map):.4f}] K\n")
            f.write(f"Noise RMS mean: {np.mean(noise_map):.4f} K\n")
            f.write(f"Noise RMS median: {np.median(noise_map):.4f} K\n\n")
        
        f.write("-"*40 + "\n")
        f.write("PER COMPONENT STATISTICS\n")
        f.write("-"*40 + "\n\n")
        
        for comp in range(n_components):
            orig_amp = original_results['physical']['amplitude'][:, :, comp]
            filt_amp = filtered_results['physical']['amplitude'][:, :, comp]
            
            orig_nonzero = np.sum(orig_amp > 0)
            filt_nonzero = np.sum(filt_amp > 0)
            
            f.write(f"Component {comp+1}:\n")
            f.write(f"  Original non-zero pixels: {orig_nonzero} ({100*orig_nonzero/total_pixels:.2f}% of total)\n")
            f.write(f"  Filtered non-zero pixels: {filt_nonzero} ({100*filt_nonzero/total_pixels:.2f}% of total)\n")
            
            if orig_nonzero > 0:
                f.write(f"  Retention rate: {100*filt_nonzero/orig_nonzero:.2f}%\n")
            
            if filt_nonzero > 0:
                f.write(f"\n  Amplitude (K):\n")
                f.write(f"    Original range: [{np.min(orig_amp[orig_amp>0]):.4f}, {np.max(orig_amp):.4f}]\n")
                f.write(f"    Filtered range: [{np.min(filt_amp[filt_amp>0]):.4f}, {np.max(filt_amp):.4f}]\n")
                f.write(f"    Original mean: {np.mean(orig_amp[orig_amp>0]):.4f}\n")
                f.write(f"    Filtered mean: {np.mean(filt_amp[filt_amp>0]):.4f}\n")
                
                # 位置信息
                orig_pos = original_results['physical']['position'][:, :, comp]
                filt_pos = filtered_results['physical']['position'][:, :, comp]
                f.write(f"\n  Position (km/s):\n")
                f.write(f"    Original range: [{np.min(orig_pos[orig_pos!=0]):.4f}, {np.max(orig_pos):.4f}]\n")
                f.write(f"    Filtered range: [{np.min(filt_pos[filt_pos!=0]):.4f}, {np.max(filt_pos):.4f}]\n")
                
                # 弥散信息
                orig_dis = original_results['physical']['dispersion'][:, :, comp]
                filt_dis = filtered_results['physical']['dispersion'][:, :, comp]
                f.write(f"\n  Dispersion (km/s):\n")
                f.write(f"    Original range: [{np.min(orig_dis[orig_dis>0]):.4f}, {np.max(orig_dis):.4f}]\n")
                f.write(f"    Filtered range: [{np.min(filt_dis[filt_dis>0]):.4f}, {np.max(filt_dis):.4f}]\n")
            
            # 计算该成分保留像素的噪声统计
            if masks is not None and noise_map is not None:
                kept_mask = masks[:, :, comp]
                kept_noise = noise_map.T[kept_mask]
                if len(kept_noise) > 0:
                    f.write(f"\n  Noise in kept pixels (K):\n")
                    f.write(f"    Mean: {np.mean(kept_noise):.4f}\n")
                    f.write(f"    Std: {np.std(kept_noise):.4f}\n")
                    f.write(f"    Range: [{np.min(kept_noise):.4f}, {np.max(kept_noise):.4f}]\n")
            
            f.write("\n" + "-"*30 + "\n\n")
    
    print(f"Statistics saved to: {stats_file}")

# ==================== 12. 主函数 ====================
def main(gaussian_file, fits_file, output_dir, noise_channels=None, snr_threshold=2.0):
    """
    主函数 - 对ROHSA结果进行SNR筛选
    - FITS输出: 物理单位
    - DAT输出: 像素单位（与原始格式一致）
    """
    
    print("\n" + "="*60)
    print("PER-PIXEL NOISE COMPONENT-WISE SNR FILTERING")
    print("="*60)
    print(f"ROHSA file: {gaussian_file}")
    print(f"FITS file: {fits_file}")
    print(f"Output directory: {output_dir}")
    print(f"SNR threshold: {snr_threshold}σ")
    
    # 创建输出目录
    os.makedirs(output_dir, exist_ok=True)
    print(f"Created output directory: {output_dir}")
    
    # 1. 读取ROHSA结果（同时获取像素单位和物理单位）
    original_results = read_rohsa_results(gaussian_file)
    n_components = original_results['n_components']
    
    # 2. 读取原始FITS用于计算噪声
    original_cube, header, velocities = read_original_fits(fits_file)
    
    # 3. 从数据计算每个像素的噪声
    noise_map, global_noise = calculate_per_pixel_noise(original_cube, output_dir, noise_channels, velocities)
    
    # 4. 对每个成分单独进行筛选
    filtered_results, masks, snr_maps = filter_each_component_with_per_pixel_noise(
        original_results, 
        noise_map, 
        snr_threshold
    )
    
    # 5. 保存筛选后的ROHSA结果
    output_base = os.path.join(output_dir, 'filtered_rohsa')
    save_filtered_rohsa_all(filtered_results, gaussian_file, output_base)
    
    # 6. 为每个成分绘制对比图
    for comp in range(n_components):
        plot_component_comparison(original_results, filtered_results, noise_map, comp, output_dir, snr_threshold)
    
    # 7. 绘制所有mask和SNR图
    plot_masks_and_snr(masks, snr_maps, noise_map, output_dir, snr_threshold)
    
    # 8. 绘制噪声图
    plot_noise_map(noise_map, output_dir)
    
    # 9. 保存统计信息
    save_statistics(original_results, filtered_results, masks, noise_map, snr_threshold, output_dir)
    
    print("\n" + "="*60)
    print("PROCESSING COMPLETED")
    print("="*60)
    print(f"All output files saved to: {output_dir}")
    print(f"  - Filtered FITS (physical units): filtered_rohsa.fits")
    print(f"  - Filtered DAT (pixel units): filtered_rohsa.dat")
    for comp in range(n_components):
        print(f"  - Component {comp+1} plot: component{comp+1}_comparison.png")
    print(f"  - All masks: all_masks.png")
    print(f"  - All SNR maps: all_snr_maps.png")
    print(f"  - Noise map: noise_map.png")
    print(f"  - Noise distribution: noise_distribution.png")
    print(f"  - Statistics: statistics.txt")
    
    return filtered_results, masks

# ==================== 13. 脚本执行 ====================
if __name__ == "__main__":
    # 请修改这些路径
    GAUSSIAN_FILE = "MS_ROHSA_3ngauss_1_3D_1.dat"  # ROHSA输出的.dat文件
    FITS_FILE = "CRAFTS_-4.7_-350_-150_Original.fits"  # 原始FITS数据文件
    OUTPUT_DIR = "./test/SNR=2_minpixels=5/threshold=1"  # 输出文件夹路径
    
    # 噪声通道参数（可选）
    NOISE_CHANNELS = [0, 190, 880, 990]  # 根据您的数据调整
    
    # SNR阈值
    SNR_THRESHOLD = 2
    
    # 运行
    filtered_results, masks = main(
        gaussian_file=GAUSSIAN_FILE,
        fits_file=FITS_FILE,
        output_dir=OUTPUT_DIR,
        noise_channels=NOISE_CHANNELS,
        snr_threshold=SNR_THRESHOLD
    )


PER-PIXEL NOISE COMPONENT-WISE SNR FILTERING
ROHSA file: MS_ROHSA_3ngauss_1_3D_1.dat
FITS file: CRAFTS_-4.7_-350_-150_Original.fits
Output directory: ./test/SNR=2_minpixels=5/threshold=1
SNR threshold: 2σ
Created output directory: ./test/SNR=2_minpixels=5/threshold=1

READING ROHSA RESULTS
Reading Gaussian parameters (pixel units)...
Opening data file
Gaussian pixel array shape: (9, 89, 153)
Data type: float64
Converting to physical units...
Spatial dimensions: 153 × 89
Number of Gaussian components: 3

Pixel units statistics:
  Component 1 amplitude (pixel): range [0.0000, 1.4986]
  Component 2 amplitude (pixel): range [0.0000, 1.8633]
  Component 3 amplitude (pixel): range [0.0000, 0.6251]

Physical units statistics:
  Component 1 amplitude (K): range [0.0000, 1.4986]
  Component 2 amplitude (K): range [0.0000, 1.8633]
  Component 3 amplitude (K): range [0.0000, 0.6251]

READING ORIGINAL FITS
Original cube shape: (994, 89, 153)
Data range: [-2.7293, 2.3161]
Velocity range: [599954.3

### 2. 筛选源，高斯分量分别进行

In [42]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from scipy import ndimage
from skimage import measure
import os


# ==================== 1. 读取筛选后的DAT文件（同时保留像素单位和物理单位） ====================
def read_filtered_dat(dat_file):
    """
    读取筛选后的ROHSA DAT文件
    返回像素单位和物理单位的结果
    """
    print("\n" + "="*60)
    print("READING FILTERED ROHSA DAT FILE")
    print("="*60)
    
    # 使用core模块读取像素单位的gaussian
    print("Reading Gaussian parameters (pixel units)...")
    gaussian_pixel = core.read_gaussian(dat_file)
    
    print(f"Gaussian pixel array shape: {gaussian_pixel.shape}")
    print(f"Data type: {gaussian_pixel.dtype}")
    
    # 转换为物理单位
    print("Converting to physical units...")
    gaussian_physical = core.physical_gaussian(gaussian_pixel)
    
    # 解析维度: (n_parameters * n_components, n_y, n_x)
    n_params_times_comp, n_y, n_x = gaussian_pixel.shape
    n_components = n_params_times_comp // 3
    
    print(f"Spatial dimensions: {n_x} × {n_y}")
    print(f"Number of Gaussian components: {n_components}")
    
    # 提取每个成分的参数 - 像素单位
    results_pixel = {}
    # 提取每个成分的参数 - 物理单位
    results_physical = {}
    
    for comp in range(n_components):
        # 像素单位
        amp_pixel = gaussian_pixel[3*comp]
        pos_pixel = gaussian_pixel[3*comp + 1]
        dis_pixel = gaussian_pixel[3*comp + 2]
        
        results_pixel[f'comp{comp+1}'] = {
            'amplitude': amp_pixel,
            'position': pos_pixel,
            'dispersion': dis_pixel
        }
        
        # 物理单位
        amp_phys = gaussian_physical[3*comp]
        pos_phys = gaussian_physical[3*comp + 1]
        dis_phys = gaussian_physical[3*comp + 2]
        
        results_physical[f'comp{comp+1}'] = {
            'amplitude': amp_phys,
            'position': pos_phys,
            'dispersion': dis_phys
        }
        
        print(f"\nComponent {comp+1}:")
        print(f"  Pixel units:")
        print(f"    Amplitude range: [{np.min(amp_pixel):.4f}, {np.max(amp_pixel):.4f}]")
        print(f"    Position range: [{np.min(pos_pixel):.4f}, {np.max(pos_pixel):.4f}]")
        print(f"    Dispersion range: [{np.min(dis_pixel):.4f}, {np.max(dis_pixel):.4f}]")
        print(f"  Physical units:")
        print(f"    Amplitude range: [{np.min(amp_phys):.4f}, {np.max(amp_phys):.4f}] K")
        print(f"    Position range: [{np.min(pos_phys):.4f}, {np.max(pos_phys):.4f}] km/s")
        print(f"    Dispersion range: [{np.min(dis_phys):.4f}, {np.max(dis_phys):.4f}] km/s")
        print(f"    Non-zero pixels: {np.sum(amp_phys > 0)}")
    
    results = {
        'pixel': results_pixel,
        'physical': results_physical,
        'n_components': n_components,
        'shape': (n_x, n_y),
        'original_data_pixel': gaussian_pixel,
        'original_data_physical': gaussian_physical
    }
    
    return results

# ==================== 2. 对每个成分进行源识别 ====================
def identify_sources_in_component(amplitude_map, min_pixels=10, max_gap=2):
    """
    在振幅图中识别独立的源
    
    参数:
        amplitude_map: 2D振幅图 (K)
        min_pixels: 源的最小像素数
        max_gap: 源内允许的最大间隔像素数
    
    返回:
        labeled_mask: 标记了不同源的mask
        n_sources: 识别出的源数量
        source_properties: 每个源的属性
    """
    print(f"\nIdentifying sources with min_pixels={min_pixels}, max_gap={max_gap}")
    
    # 创建二值mask（振幅>0的像素）
    binary_mask = amplitude_map > 0
    
    # 使用结构元素定义连通性（考虑最大间隔）
    structure = ndimage.generate_binary_structure(2, 1)
    if max_gap > 1:
        structure = ndimage.iterate_structure(structure, max_gap - 1)
    
    # 标记连通区域
    labeled_mask, num_features = ndimage.label(binary_mask, structure=structure)
    
    print(f"Initial connected components: {num_features}")
    
    # 筛选出大于最小像素数的源
    source_properties = []
    final_mask = np.zeros_like(labeled_mask)
    source_count = 0
    
    for label in range(1, num_features + 1):
        mask = labeled_mask == label
        pixel_count = np.sum(mask)
        
        if pixel_count >= min_pixels:
            source_count += 1
            final_mask[mask] = source_count
            
            # 计算源属性
            y_indices, x_indices = np.where(mask)
            amp_values = amplitude_map[mask]
            
            props = {
                'source_id': source_count,
                'pixel_count': pixel_count,
                'centroid': (np.mean(x_indices), np.mean(y_indices)),
                'peak_amplitude': np.max(amp_values),  # K
                'mean_amplitude': np.mean(amp_values), # K
                'bbox': (np.min(x_indices), np.max(x_indices), 
                        np.min(y_indices), np.max(y_indices)),
                'pixels': list(zip(x_indices, y_indices))
            }
            source_properties.append(props)
            
            print(f"  Source {source_count}: {pixel_count} pixels, "
                  f"peak={props['peak_amplitude']:.2f} K, "
                  f"centroid=({props['centroid'][0]:.1f}, {props['centroid'][1]:.1f})")
    
    print(f"Final sources after filtering: {source_count}")
    
    return final_mask, source_count, source_properties

# ==================== 3. 创建每个成分的源mask图（物理单位，colorbar放下面） ====================
def plot_component_sources(component_data_phys, component_idx, output_dir, min_pixels=10, max_gap=2):
    """
    为单个成分绘制源识别结果，使用物理单位，colorbar放在图下面
    """
    amplitude = component_data_phys['amplitude']      # K
    position = component_data_phys['position']        # km/s
    dispersion = component_data_phys['dispersion']    # km/s
    
    print(f"\nPlotting component {component_idx} with physical units:")
    print(f"  Amplitude range: [{np.min(amplitude[amplitude>0]):.4f}, {np.max(amplitude):.4f}] K")
    print(f"  Position range: [{np.min(position[position!=0]):.4f}, {np.max(position):.4f}] km/s")
    print(f"  Dispersion range: [{np.min(dispersion[dispersion>0]):.4f}, {np.max(dispersion):.4f}] km/s")
    
    # 识别源
    source_mask, n_sources, source_props = identify_sources_in_component(
        amplitude, min_pixels=min_pixels, max_gap=max_gap
    )
    
    # 确定每个参数的颜色范围
    amp_vmin = np.min(amplitude[amplitude>0]) if np.any(amplitude>0) else 0
    amp_vmax = np.max(amplitude)
    pos_vmin = np.min(position[position!=0]) if np.any(position!=0) else -10
    pos_vmax = np.max(position)
    dis_vmin = np.min(dispersion[dispersion>0]) if np.any(dispersion>0) else 0
    dis_vmax = np.max(dispersion)
    
    # 创建2×3子图
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    
    # ===== 第一行：原始参数 =====
    # 原始振幅
    im1 = axes[0, 0].imshow(amplitude, origin='lower', cmap='inferno', 
                            vmin=amp_vmin, vmax=amp_vmax)
    axes[0, 0].set_title(f'Component {component_idx} - Original Amplitude', fontsize=12)
    cbar1 = plt.colorbar(im1, ax=axes[0, 0], shrink=0.8, pad=0.05, location='bottom')
    cbar1.set_label('Brightness Temp. [K]', fontsize=10)
    
    # 源mask图
    from matplotlib.colors import ListedColormap
    ax2 = axes[0, 1]
    if n_sources > 0:
        colors = plt.cm.tab20(np.linspace(0, 1, n_sources))
        colors = np.vstack([[0,0,0,1], colors])
        cmap = ListedColormap(colors)
        
        im2 = ax2.imshow(source_mask, origin='lower', cmap=cmap, 
                         vmin=0, vmax=n_sources, interpolation='nearest')
        ax2.set_title(f'Component {component_idx} - Source Mask ({n_sources} sources)', fontsize=12)
        cbar2 = plt.colorbar(im2, ax=ax2, shrink=0.8, pad=0.05, location='bottom', ticks=range(n_sources+1))
        cbar2.set_label('Source ID', fontsize=10)
        
        # 在图上标注源的中心
        for props in source_props:
            x, y = props['centroid']
            ax2.plot(x, y, 'r+', markersize=10, markeredgewidth=2)
            ax2.text(x+2, y+2, f'{props["source_id"]}', 
                    color='white', fontsize=10, weight='bold',
                    bbox=dict(boxstyle='round', facecolor='black', alpha=0.5))
    else:
        ax2.imshow(source_mask, origin='lower', cmap='gray')
        ax2.set_title(f'Component {component_idx} - No Sources Found', fontsize=12)
    
    # 原始位置图
    im3 = axes[0, 2].imshow(position, origin='lower', cmap='coolwarm',
                            vmin=pos_vmin, vmax=pos_vmax)
    axes[0, 2].set_title(f'Component {component_idx} - Original Position', fontsize=12)
    cbar3 = plt.colorbar(im3, ax=axes[0, 2], shrink=0.8, pad=0.05, location='bottom')
    cbar3.set_label('Velocity [km s$^{-1}$]', fontsize=10)
    
    # ===== 第二行：源筛选后的参数 =====
    # 源筛选后的振幅图
    filtered_amplitude = np.zeros_like(amplitude)
    filtered_amplitude[source_mask > 0] = amplitude[source_mask > 0]
    
    im4 = axes[1, 0].imshow(filtered_amplitude, origin='lower', cmap='inferno',
                            vmin=amp_vmin, vmax=amp_vmax)
    axes[1, 0].set_title(f'Component {component_idx} - Source-Only Amplitude', fontsize=12)
    cbar4 = plt.colorbar(im4, ax=axes[1, 0], shrink=0.8, pad=0.05, location='bottom')
    cbar4.set_label('Brightness Temp. [K]', fontsize=10)
    
    # 源筛选后的位置图
    filtered_position = np.zeros_like(position)
    filtered_position[source_mask > 0] = position[source_mask > 0]
    
    im5 = axes[1, 1].imshow(filtered_position, origin='lower', cmap='coolwarm',
                            vmin=pos_vmin, vmax=pos_vmax)
    axes[1, 1].set_title(f'Component {component_idx} - Source-Only Position', fontsize=12)
    cbar5 = plt.colorbar(im5, ax=axes[1, 1], shrink=0.8, pad=0.05, location='bottom')
    cbar5.set_label('Velocity [km s$^{-1}$]', fontsize=10)
    
    # 源筛选后的宽度图
    filtered_dispersion = np.zeros_like(dispersion)
    filtered_dispersion[source_mask > 0] = dispersion[source_mask > 0]
    
    im6 = axes[1, 2].imshow(filtered_dispersion, origin='lower', cmap='cubehelix',
                            vmin=dis_vmin, vmax=dis_vmax)
    axes[1, 2].set_title(f'Component {component_idx} - Source-Only Dispersion', fontsize=12)
    cbar6 = plt.colorbar(im6, ax=axes[1, 2], shrink=0.8, pad=0.05, location='bottom')
    cbar6.set_label('Dispersion [km s$^{-1}$]', fontsize=10)
    
    # 添加总标题
    plt.suptitle(f'Component {component_idx} Source Identification\n'
                f'Min pixels: {min_pixels}, Max gap: {max_gap}\n'
                f'Amplitude: K | Position: km/s | Dispersion: km/s', 
                fontsize=14, y=1.02)
    
    plt.tight_layout()
    
    # 保存图片
    comp_file = os.path.join(output_dir, f'component{component_idx}_sources.png')
    plt.savefig(comp_file, dpi=150, bbox_inches='tight')
    plt.close()
    
    print(f"\nComponent {component_idx} source plot saved to: {comp_file}")
    
    return source_mask, n_sources, source_props

# ==================== 4. 保存源筛选后的FITS文件（物理单位，用于可视化） ====================
def save_source_filtered_fits_physical(results, all_source_masks, output_file):
    """
    保存源筛选后的FITS文件（物理单位，用于可视化）
    """
    try:
        n_components = results['n_components']
        n_y, n_x = results['shape'][1], results['shape'][0]
        
        # 创建物理单位的数据数组
        filtered_data_phys = np.zeros((3*n_components, n_y, n_x))
        
        for comp in range(n_components):
            source_mask = all_source_masks[comp]
            comp_data = results['physical'][f'comp{comp+1}']
            
            filtered_data_phys[3*comp] = comp_data['amplitude'] * (source_mask > 0)
            filtered_data_phys[3*comp + 1] = comp_data['position'] * (source_mask > 0)
            filtered_data_phys[3*comp + 2] = comp_data['dispersion'] * (source_mask > 0)
        
        hdu = fits.PrimaryHDU(filtered_data_phys)
        hdu.writeto(output_file, overwrite=True)
        print(f"\nSource-filtered FITS file (physical units) saved to: {output_file}")
        return True
    except Exception as e:
        print(f"Error saving source-filtered FITS: {e}")
        return False

# ==================== 5. 保存源筛选后的.dat文件（像素单位，与原始格式一致） ====================
def save_source_filtered_dat_pixel(results, all_source_masks, original_dat_file, output_file):
    """
    保存源筛选后的.dat文件（像素单位，与原始格式相同）
    注意：在.dat文件中，顺序是 y x amplitude position dispersion
    """
    print("\n" + "="*60)
    print("SAVING SOURCE-FILTERED RESULTS TO .DAT FORMAT (PIXEL UNITS)")
    print("="*60)
    
    try:
        n_components = results['n_components']
        n_y, n_x = results['shape'][1], results['shape'][0]
        
        print(f"Reorganizing data for .dat format...")
        print(f"  Spatial grid: X({n_x}) × Y({n_y})")
        print(f"  Components: {n_components}")
        print(f"  DAT file format: Y X AMP POS DISP (Y first, X second)")
        print(f"  Units: Pixel units (matching original ROHSA format)")
        
        # 读取原始文件的头信息
        with open(original_dat_file, 'r') as f:
            header_lines = []
            for i in range(27):
                line = f.readline()
                header_lines.append(line)
        
        # 创建像素单位的筛选后数据
        filtered_data_pixel = np.zeros((3*n_components, n_y, n_x))
        
        for comp in range(n_components):
            source_mask = all_source_masks[comp]
            comp_data_pixel = results['pixel'][f'comp{comp+1}']
            
            filtered_data_pixel[3*comp] = comp_data_pixel['amplitude'] * (source_mask > 0)
            filtered_data_pixel[3*comp + 1] = comp_data_pixel['position'] * (source_mask > 0)
            filtered_data_pixel[3*comp + 2] = comp_data_pixel['dispersion'] * (source_mask > 0)
        
        # 统计非零像素
        total_entries = n_x * n_y * n_components
        nonzero_entries = np.sum(filtered_data_pixel != 0)
        print(f"  Non-zero entries: {nonzero_entries} ({100 * nonzero_entries / total_entries:.2f}%)")
        
        # 准备数据行 - 注意顺序：先Y后X
        data_lines = []
        for j in range(n_y):  # Y坐标（外层循环）
            for i in range(n_x):  # X坐标（内层循环）
                for comp in range(n_components):
                    amp = filtered_data_pixel[3*comp, j, i]
                    mean = filtered_data_pixel[3*comp + 1, j, i]
                    sigma = filtered_data_pixel[3*comp + 2, j, i]
                    
                    line = f"    {j:4d}    {i:4d}    {amp:20.16f}    {mean:20.16f}    {sigma:20.16f}\n"
                    data_lines.append(line)
        
        # 写入新文件
        with open(output_file, 'w') as f:
            f.writelines(header_lines)
            f.writelines(data_lines)
        
        print(f"\n.dat file statistics:")
        print(f"  Total entries: {total_entries}")
        print(f"  Non-zero entries: {nonzero_entries}")
        print(f"  Data density: {100 * nonzero_entries / total_entries:.2f}%")
        print(f"\nSource-filtered .dat file (pixel units) saved to: {output_file}")
        
        # 验证前几行
        print(f"\n  First few lines of saved file:")
        with open(output_file, 'r') as f:
            for i, line in enumerate(f):
                if i >= 5:  # 显示前5行数据
                    break
                if i >= len(header_lines):  # 跳过头部
                    print(f"    {line.strip()}")
        
        return True
        
    except Exception as e:
        print(f"Error saving source-filtered .dat: {e}")
        import traceback
        traceback.print_exc()
        return False

# ==================== 6. 绘制汇总图（colorbar放下面） ====================
def plot_summary(all_source_masks, all_source_counts, output_dir):
    """绘制所有成分的源汇总图，colorbar放在下面"""
    n_components = len(all_source_masks)
    
    fig, axes = plt.subplots(1, n_components, figsize=(5*n_components, 5))
    if n_components == 1:
        axes = [axes]
    
    from matplotlib.colors import ListedColormap
    
    for comp in range(n_components):
        source_mask = all_source_masks[comp]
        n_sources = all_source_counts[comp]
        
        if n_sources > 0:
            colors = plt.cm.tab20(np.linspace(0, 1, n_sources))
            colors = np.vstack([[0,0,0,1], colors])
            cmap = ListedColormap(colors)
            
            im = axes[comp].imshow(source_mask, origin='lower', cmap=cmap,
                                   vmin=0, vmax=n_sources, aspect='auto', interpolation='nearest')
            axes[comp].set_title(f'Component {comp+1}\n{n_sources} sources', fontsize=12)
            cbar = plt.colorbar(im, ax=axes[comp], shrink=0.8, pad=0.15, location='bottom')
            cbar.set_label('Source ID', fontsize=10)
            cbar.set_ticks(range(n_sources+1))
        else:
            im = axes[comp].imshow(source_mask, origin='lower', cmap='gray', aspect='auto')
            axes[comp].set_title(f'Component {comp+1}\nNo sources', fontsize=12)
        
        axes[comp].set_xlabel('X pixel')
        axes[comp].set_ylabel('Y pixel')
    
    plt.suptitle('Source Identification Results - All Components', fontsize=14, y=1.02)
    plt.tight_layout()
    
    summary_file = os.path.join(output_dir, 'all_components_sources_summary.png')
    plt.savefig(summary_file, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"\nSummary plot saved to: {summary_file}")

# ==================== 7. 保存源信息到文本文件 ====================
def save_source_info(all_source_props, output_dir):
    """保存每个源的详细信息到文本文件"""
    info_file = os.path.join(output_dir, 'source_information.txt')
    
    with open(info_file, 'w') as f:
        f.write("="*60 + "\n")
        f.write("SOURCE IDENTIFICATION RESULTS\n")
        f.write("="*60 + "\n\n")
        f.write("All values are in physical units:\n")
        f.write("  - Amplitude: K (Kelvin)\n")
        f.write("  - Position: km/s\n")
        f.write("  - Dispersion: km/s\n\n")
        
        for comp_idx, source_props in enumerate(all_source_props):
            f.write(f"\n{'='*40}\n")
            f.write(f"COMPONENT {comp_idx+1}\n")
            f.write(f"{'='*40}\n\n")
            
            if len(source_props) == 0:
                f.write("No sources found in this component.\n")
                continue
            
            for props in source_props:
                f.write(f"Source {props['source_id']}:\n")
                f.write(f"  Number of pixels: {props['pixel_count']}\n")
                f.write(f"  Centroid (X, Y): ({props['centroid'][0]:.2f}, {props['centroid'][1]:.2f})\n")
                f.write(f"  Peak amplitude: {props['peak_amplitude']:.4f} K\n")
                f.write(f"  Mean amplitude: {props['mean_amplitude']:.4f} K\n")
                f.write(f"  Bounding box: X[{props['bbox'][0]}, {props['bbox'][1]}], "
                       f"Y[{props['bbox'][2]}, {props['bbox'][3]}]\n\n")
    
    print(f"Source information saved to: {info_file}")

# ==================== 8. 保存源mask到FITS文件 ====================
def save_source_masks(all_source_masks, output_file):
    """保存所有分量的源mask到FITS文件"""
    masks_array = np.array(all_source_masks)
    hdu = fits.PrimaryHDU(masks_array)
    hdu.writeto(output_file, overwrite=True)
    print(f"Source masks saved to: {output_file}")

# ==================== 9. 主函数 ====================
def main(filtered_dat_file, original_dat_file, output_dir, 
         min_pixels=10, max_gap=2):
    """
    主函数 - 对筛选后的DAT文件进行源识别
    
    参数:
        filtered_dat_file: 上一步筛选后的DAT文件
        original_dat_file: 原始的.dat文件（用于头信息）
        output_dir: 输出目录
        min_pixels: 源的最小像素数
        max_gap: 源内允许的最大间隔像素数
    """
    
    print("\n" + "="*60)
    print("SOURCE IDENTIFICATION FROM FILTERED ROHSA RESULTS")
    print("="*60)
    print(f"Filtered DAT file: {filtered_dat_file}")
    print(f"Output directory: {output_dir}")
    print(f"Min pixels per source: {min_pixels}")
    print(f"Max gap between pixels: {max_gap}")
    
    # 创建输出目录
    os.makedirs(output_dir, exist_ok=True)
    
    # 1. 读取筛选后的DAT文件（同时获取像素单位和物理单位）
    results = read_filtered_dat(filtered_dat_file)
    n_components = results['n_components']
    
    # 存储每个成分的源信息
    all_source_masks = []
    all_source_counts = []
    all_source_props = []
    
    # 2. 对每个成分进行源识别（使用物理单位的数据进行识别和绘图）
    for comp in range(n_components):
        print(f"\n{'='*40}")
        print(f"PROCESSING COMPONENT {comp+1}")
        print(f"{'='*40}")
        
        component_data_phys = results['physical'][f'comp{comp+1}']
        
        # 识别源并绘图
        source_mask, n_sources, source_props = plot_component_sources(
            component_data_phys, comp+1, output_dir, 
            min_pixels=min_pixels, max_gap=max_gap
        )
        
        all_source_masks.append(source_mask)
        all_source_counts.append(n_sources)
        all_source_props.append(source_props)
    
    # 3. 保存源筛选后的FITS文件（物理单位，用于可视化）
    source_filtered_fits_phys = os.path.join(output_dir, 'source_filtered_rohsa_physical.fits')
    save_source_filtered_fits_physical(results, all_source_masks, source_filtered_fits_phys)
    
    # 4. 保存源mask到FITS
    save_source_masks(all_source_masks, os.path.join(output_dir, 'source_masks.fits'))
    
    # 5. 保存源筛选后的.dat文件（像素单位，与原始格式一致）
    source_filtered_dat_pixel = os.path.join(output_dir, 'source_filtered_rohsa_pixel.dat')
    save_source_filtered_dat_pixel(results, all_source_masks, original_dat_file, 
                                  source_filtered_dat_pixel)
    
    # 6. 绘制汇总图
    plot_summary(all_source_masks, all_source_counts, output_dir)
    
    # 7. 保存源信息
    save_source_info(all_source_props, output_dir)
    
    # 8. 打印总结
    print("\n" + "="*60)
    print("SOURCE IDENTIFICATION COMPLETED")
    print("="*60)
    print(f"\nSummary:")
    for comp in range(n_components):
        print(f"  Component {comp+1}: {all_source_counts[comp]} sources identified")
    
    print(f"\nOutput files saved to: {output_dir}")
    print(f"  - Source-filtered FITS (physical): source_filtered_rohsa_physical.fits")
    print(f"  - Source-filtered DAT (pixel): source_filtered_rohsa_pixel.dat")
    print(f"  - Source masks: source_masks.fits")
    print(f"  - Individual component plots: component*_sources.png")
    print(f"  - Summary plot: all_components_sources_summary.png")
    print(f"  - Source information: source_information.txt")
    
    return all_source_masks, all_source_props

# ==================== 10. 脚本执行 ====================
if __name__ == "__main__":
    # 请修改这些路径
    FILTERED_DAT_FILE = "./baseline/SNR=1.5/output_file_2sigma/filtered_rohsa.dat"  # 上一步筛选后的DAT文件
    ORIGINAL_DAT_FILE = "./baseline/MS_ROHSA_3ngauss_1_3D.dat"  # 原始的.dat文件（用于头信息）
    OUTPUT_DIR = "./baseline/SNR=1.5//output_individual_source"  # 输出文件夹路径
    
    # 源识别参数
    MIN_PIXELS = 10   # 源的最小像素数
    MAX_GAP = 2      # 源内允许的最大间隔像素数
    
    # 运行
    all_source_masks, all_source_props = main(
        filtered_dat_file=FILTERED_DAT_FILE,
        original_dat_file=ORIGINAL_DAT_FILE,
        output_dir=OUTPUT_DIR,
        min_pixels=MIN_PIXELS,
        max_gap=MAX_GAP
    )


SOURCE IDENTIFICATION FROM FILTERED ROHSA RESULTS
Filtered DAT file: ./test/SNR=2.5_minpixels=5/threshold=1/filtered_rohsa.dat
Output directory: ./test/SNR=2.5_minpixels=15/output_individual_source
Min pixels per source: 15
Max gap between pixels: 2

READING FILTERED ROHSA DAT FILE
Reading Gaussian parameters (pixel units)...
Opening data file
Gaussian pixel array shape: (9, 89, 153)
Data type: float64
Converting to physical units...
Spatial dimensions: 153 × 89
Number of Gaussian components: 3

Component 1:
  Pixel units:
    Amplitude range: [0.0000, 1.4986]
    Position range: [-1.0000, 526.8174]
    Dispersion range: [0.0000, 100.0000]
  Physical units:
    Amplitude range: [0.0000, 1.4986] K
    Position range: [-256.0680, -149.8271] km/s
    Dispersion range: [0.0000, 20.1284] km/s
    Non-zero pixels: 4193

Component 2:
  Pixel units:
    Amplitude range: [0.0000, 1.8633]
    Position range: [-1.0000, 660.4800]
    Dispersion range: [0.0000, 73.4503]
  Physical units:
    Ampli

#### 每个源都提出来

In [18]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from scipy import ndimage
import os


# ==================== 1. 读取DAT文件 ====================
def read_filtered_dat(dat_file):
    print("\n" + "="*60)
    print("READING FILTERED ROHSA DAT FILE")
    print("="*60)
    
    gaussian_pixel = core.read_gaussian(dat_file)
    gaussian_physical = core.physical_gaussian(gaussian_pixel)
    
    n_params_times_comp, n_y, n_x = gaussian_pixel.shape
    n_components = n_params_times_comp // 3
    
    results = {}
    
    for comp in range(1, n_components + 1):
        results[f'comp{comp}'] = {
            'amplitude_pixel': gaussian_pixel[3*(comp-1)],
            'position_pixel': gaussian_pixel[3*(comp-1) + 1],
            'dispersion_pixel': gaussian_pixel[3*(comp-1) + 2],
            'amplitude_phys': gaussian_physical[3*(comp-1)],
            'position_phys': gaussian_physical[3*(comp-1) + 1],
            'dispersion_phys': gaussian_physical[3*(comp-1) + 2],
        }
    
    results['n_components'] = n_components
    results['shape'] = (n_x, n_y)
    
    return results


# ==================== 2. 提取源 ====================
def extract_sources_from_component(results, comp, min_pixels=10, max_gap=2):
    amplitude = results[f'comp{comp}']['amplitude_phys']
    position = results[f'comp{comp}']['position_phys']
    dispersion = results[f'comp{comp}']['dispersion_phys']
    
    binary_mask = amplitude > 0
    if not np.any(binary_mask):
        return []
    
    structure = ndimage.generate_binary_structure(2, 1)
    if max_gap > 1:
        structure = ndimage.iterate_structure(structure, max_gap - 1)
    
    labeled_mask, num_features = ndimage.label(binary_mask, structure=structure)
    
    sources = []
    
    for label in range(1, num_features + 1):
        mask = labeled_mask == label
        if np.sum(mask) < min_pixels:
            continue
        
        y_idx, x_idx = np.where(mask)
        
        amps = amplitude[y_idx, x_idx]
        poss = position[y_idx, x_idx]
        disp = dispersion[y_idx, x_idx]
        
        total_amp = np.sum(amps)
        if total_amp > 0:
            v = np.sum(poss * amps) / total_amp
            s = np.sum(disp * amps) / total_amp
        else:
            v = np.mean(poss)
            s = np.mean(disp)
        
        source = {
            'source_id': label,
            'component': comp,
            'pixel_count': len(amps),
            'pixels': list(zip(x_idx, y_idx)),
            'centroid': (np.mean(x_idx), np.mean(y_idx)),
            'weighted_average': {
                'velocity': v,
                'dispersion': s,
                'fwhm': 2.355 * s,
                'amplitude': total_amp / len(amps)
            }
        }
        sources.append(source)
    
    return sources


# ==================== 3. 保存单个源 ====================
def save_individual_source_dat(results, source, output_dir, header_lines):
    n_x, n_y = results['shape']
    n_components = results['n_components']
    
    # ✅ 新命名
    source_dir = os.path.join(output_dir, f'source_{source["global_id"]:03d}')
    os.makedirs(source_dir, exist_ok=True)
    
    data_lines = []
    
    pixel_set = set(source['pixels'])
    
    for y in range(n_y):
        for x in range(n_x):
            for comp in range(1, n_components + 1):
                
                if (x, y) in pixel_set and comp == source['component']:
                    amp = results[f'comp{comp}']['amplitude_pixel'][y, x]
                    pos = results[f'comp{comp}']['position_pixel'][y, x]
                    dis = results[f'comp{comp}']['dispersion_pixel'][y, x]
                else:
                    amp = pos = dis = 0
                
                data_lines.append(
                    f"{y:4d} {x:4d} {amp:20.16f} {pos:20.16f} {dis:20.16f}\n"
                )
    
    # ✅ 文件名统一
    dat_path = os.path.join(source_dir, f'source_{source["global_id"]:03d}.dat')
    
    with open(dat_path, 'w') as f:
        f.writelines(header_lines)
        f.writelines(data_lines)
    
    # info 文件
    info_path = os.path.join(source_dir, f'source_{source["global_id"]:03d}_info.txt')
    
    with open(info_path, 'w') as f:
        f.write(f"Source {source['global_id']}\n")
        f.write(f"Component: {source['component']}\n")
        f.write(f"Pixels: {source['pixel_count']}\n")
        f.write(f"Velocity: {source['weighted_average']['velocity']:.3f}\n")
    
    return source_dir


# ==================== 4. 保存全部 ====================
def save_all_individual_sources(results, all_sources, original_dat, output_dir):
    
    try:
        with open(original_dat) as f:
            header = [next(f) for _ in range(27)]
    except:
        header = ["\n"] * 27
    
    out_dir = os.path.join(output_dir, "individual_sources")
    os.makedirs(out_dir, exist_ok=True)
    
    for s in all_sources:
        save_individual_source_dat(results, s, out_dir, header)
    
    print(f"\nSaved {len(all_sources)} sources → {out_dir}")
    
    return out_dir


# ==================== 5. 主函数 ====================
def main(dat_file, original_dat_file, output_dir):
    
    results = read_filtered_dat(dat_file)
    
    n_components = results['n_components']
    
    all_sources = []
    global_id = 1   # ✅ 核心
    
    for comp in range(1, n_components + 1):
        sources = extract_sources_from_component(results, comp)
        
        for s in sources:
            s['global_id'] = global_id   # ✅ 唯一ID
            global_id += 1
            all_sources.append(s)
    
    print(f"Total sources: {len(all_sources)}")
    
    save_all_individual_sources(
        results,
        all_sources,
        original_dat_file,
        output_dir
    )
    
    print("\nDone.")


# ==================== 6. 运行 ====================
if __name__ == "__main__":
    
    DAT_FILE = "./n_gauss=3/output_individual_source/source_filtered_rohsa_pixel.dat"
    ORIGINAL_DAT_FILE = "MS_ROHSA_3ngauss_1_3D_1.dat"
    OUTPUT_DIR = "./n_gauss=3/output_individual_source_table"
    
    main(DAT_FILE, ORIGINAL_DAT_FILE, OUTPUT_DIR)


READING FILTERED ROHSA DAT FILE
Opening data file
Total sources: 9

Saved 9 sources → ./n_gauss=3/output_individual_source_table/individual_sources

Done.


In [14]:
import numpy as np
from astropy.io import fits
from scipy import ndimage
import os


# ==================== 1. 读取DAT文件 ====================
def read_filtered_dat(dat_file):
    gaussian_pixel = core.read_gaussian(dat_file)
    gaussian_physical = core.physical_gaussian(gaussian_pixel)
    
    n_params_times_comp, n_y, n_x = gaussian_pixel.shape
    n_components = n_params_times_comp // 3
    
    results = {}
    
    for comp in range(1, n_components + 1):
        results[f'comp{comp}'] = {
            'amplitude_pixel': gaussian_pixel[3*(comp-1)],
            'position_pixel': gaussian_pixel[3*(comp-1) + 1],
            'dispersion_pixel': gaussian_pixel[3*(comp-1) + 2],
            'amplitude_phys': gaussian_physical[3*(comp-1)],
            'position_phys': gaussian_physical[3*(comp-1) + 1],
            'dispersion_phys': gaussian_physical[3*(comp-1) + 2],
        }
    
    results['n_components'] = n_components
    results['shape'] = (n_x, n_y)
    
    return results


# ==================== 2. 提取源 ====================
def extract_sources_from_component(results, comp, min_pixels=5, max_gap=2):
    amplitude = results[f'comp{comp}']['amplitude_phys']
    position = results[f'comp{comp}']['position_phys']
    dispersion = results[f'comp{comp}']['dispersion_phys']
    
    binary_mask = amplitude > 0
    if not np.any(binary_mask):
        return []
    
    structure = ndimage.generate_binary_structure(2, 1)
    if max_gap > 1:
        structure = ndimage.iterate_structure(structure, max_gap - 1)
    
    labeled_mask, num_features = ndimage.label(binary_mask, structure=structure)
    
    sources = []
    
    for label in range(1, num_features + 1):
        mask = labeled_mask == label
        if np.sum(mask) < min_pixels:
            continue
        
        y_idx, x_idx = np.where(mask)
        
        amps = amplitude[y_idx, x_idx]
        poss = position[y_idx, x_idx]
        disp = dispersion[y_idx, x_idx]
        
        total_amp = np.sum(amps)
        if total_amp > 0:
            v = np.sum(poss * amps) / total_amp
            s = np.sum(disp * amps) / total_amp
        else:
            v = np.mean(poss)
            s = np.mean(disp)
        
        source = {
            'source_id': label,
            'component': comp,
            'pixel_count': len(amps),
            'pixels': list(zip(x_idx, y_idx)),
            'centroid': (np.mean(x_idx), np.mean(y_idx)),
            'weighted_average': {
                'velocity': v,
                'dispersion': s,
                'fwhm': 2.355 * s,
                'amplitude': total_amp / len(amps)
            }
        }
        sources.append(source)
    
    return sources


# ==================== ⭐ 3. 计算 RMS ====================
def compute_source_rms(source, data_cube):
    pixels = source['pixels']
    
    rms_values = []
    
    for (x, y) in pixels:
        spectrum = data_cube[:, y, x]
        
        # 简单方法：直接 std（包含信号）
        rms = np.std(spectrum)
        
        rms_values.append(rms)
    
    return np.mean(rms_values)


# ==================== 4. 保存单个源 ====================
def save_individual_source_dat(results, source, output_dir, header_lines, data_cube):
    n_x, n_y = results['shape']
    n_components = results['n_components']
    
    source_dir = os.path.join(output_dir, f'source_{source["global_id"]:03d}')
    os.makedirs(source_dir, exist_ok=True)
    
    data_lines = []
    pixel_set = set(source['pixels'])
    
    for y in range(n_y):
        for x in range(n_x):
            for comp in range(1, n_components + 1):
                
                if (x, y) in pixel_set and comp == source['component']:
                    amp = results[f'comp{comp}']['amplitude_pixel'][y, x]
                    pos = results[f'comp{comp}']['position_pixel'][y, x]
                    dis = results[f'comp{comp}']['dispersion_pixel'][y, x]
                else:
                    amp = pos = dis = 0
                
                data_lines.append(
                    f"{y:4d} {x:4d} {amp:20.16f} {pos:20.16f} {dis:20.16f}\n"
                )
    
    dat_path = os.path.join(source_dir, f'source_{source["global_id"]:03d}.dat')
    
    with open(dat_path, 'w') as f:
        f.writelines(header_lines)
        f.writelines(data_lines)
    
    # ⭐ 计算 RMS
    source_rms = compute_source_rms(source, data_cube)
    
    # info 文件
    info_path = os.path.join(source_dir, f'source_{source["global_id"]:03d}_info.txt')
    
    with open(info_path, 'w') as f:
        f.write(f"Source {source['global_id']}\n")
        f.write(f"Component: {source['component']}\n")
        f.write(f"Pixels: {source['pixel_count']}\n")
        f.write(f"Velocity: {source['weighted_average']['velocity']:.3f}\n")
        f.write(f"RMS: {source_rms:.4f}\n")
    
    return source_rms


# ==================== 5. 保存全部 ====================
def save_all_individual_sources(results, all_sources, original_dat, output_dir, data_cube):
    
    try:
        with open(original_dat) as f:
            header = [next(f) for _ in range(27)]
    except:
        header = ["\n"] * 27
    
    out_dir = os.path.join(output_dir, "individual_sources")
    os.makedirs(out_dir, exist_ok=True)
    
    rms_txt = os.path.join(out_dir, "rms_summary.txt")
    
    with open(rms_txt, 'w') as f:
        f.write("ID   RMS\n")
        
        for s in all_sources:
            rms = save_individual_source_dat(results, s, out_dir, header, data_cube)
            f.write(f"{s['global_id']:03d}  {rms:.4f}\n")
    
    print(f"\nSaved {len(all_sources)} sources → {out_dir}")
    print(f"RMS summary → {rms_txt}")
    
    return out_dir


# ==================== 6. 主函数 ====================
def main(dat_file, original_dat_file, cube_fits, output_dir):
    
    # ⭐ 读取原始数据 cube
    data_cube = fits.getdata(cube_fits)
    
    results = read_filtered_dat(dat_file)
    n_components = results['n_components']
    
    all_sources = []
    global_id = 1
    
    for comp in range(1, n_components + 1):
        sources = extract_sources_from_component(results, comp)
        
        for s in sources:
            s['global_id'] = global_id
            global_id += 1
            all_sources.append(s)
    
    print(f"Total sources: {len(all_sources)}")
    
    save_all_individual_sources(
        results,
        all_sources,
        original_dat_file,
        output_dir,
        data_cube
    )
    
    print("\nDone.")


# ==================== 7. 运行 ====================
if __name__ == "__main__":
    
    DAT_FILE = "./test/SNR=1_minpixels=10/output_individual_source/source_filtered_rohsa_pixel.dat"
    ORIGINAL_DAT_FILE = "MS_ROHSA_3ngauss_1_3D_1.dat"
    CUBE_FITS = "CRAFTS_-4.7_-350_-150_Original.fits"   #  替换成你的原始 cube
    OUTPUT_DIR = "./test/SNR=1_minpixels=15/output_individual_source_table"
    
    main(DAT_FILE, ORIGINAL_DAT_FILE, CUBE_FITS, OUTPUT_DIR)

Opening data file
Total sources: 18

Saved 18 sources → ./test/SNR=1_minpixels=15/output_individual_source_table/individual_sources
RMS summary → ./test/SNR=1_minpixels=15/output_individual_source_table/individual_sources/rms_summary.txt

Done.


### 3.合并源

In [12]:
from scipy import ndimage
from collections import defaultdict

def identify_individual_sources_in_component(amplitude_map, position_map, dispersion_map, 
                                              min_pixels=10, max_gap=2):
    """
    在单个成分内识别独立的源（基于空间连通性）
    
    参数:
        amplitude_map: 振幅图 (K)
        position_map: 速度图 (km/s)
        dispersion_map: 弥散图 (km/s)
        min_pixels: 源的最小像素数
        max_gap: 源内允许的最大间隔像素数
    
    返回:
        sources: 每个源的详细信息列表
    """
    # 创建二值mask（振幅>0的像素）
    binary_mask = amplitude_map > 0
    
    if not np.any(binary_mask):
        return []
    
    # 使用结构元素定义连通性
    structure = ndimage.generate_binary_structure(2, 1)
    if max_gap > 1:
        structure = ndimage.iterate_structure(structure, max_gap - 1)
    
    # 标记连通区域
    labeled_mask, num_features = ndimage.label(binary_mask, structure=structure)
    
    sources = []
    
    for label in range(1, num_features + 1):
        mask = labeled_mask == label
        pixel_count = np.sum(mask)
        
        # 筛选出大于最小像素数的源
        if pixel_count >= min_pixels:
            # 获取源内的像素坐标和值
            y_indices, x_indices = np.where(mask)
            
            # 收集该源的所有像素数据
            amplitudes = []
            positions = []
            dispersions = []
            pixels = []
            
            for y, x in zip(y_indices, x_indices):
                amp = amplitude_map[y, x]
                pos = position_map[y, x]
                dis = dispersion_map[y, x]
                
                amplitudes.append(amp)
                positions.append(pos)
                dispersions.append(dis)
                pixels.append((x, y))
            
            amplitudes = np.array(amplitudes)
            positions = np.array(positions)
            dispersions = np.array(dispersions)
            
            # 计算加权平均（以振幅为权重）
            total_amp = np.sum(amplitudes)
            if total_amp > 0:
                weighted_position = np.sum(positions * amplitudes) / total_amp
                weighted_dispersion = np.sum(dispersions * amplitudes) / total_amp
                weighted_fwhm = 2.355 * weighted_dispersion
                weighted_amplitude = total_amp / pixel_count  # 平均振幅
            else:
                weighted_position = np.mean(positions)
                weighted_dispersion = np.mean(dispersions)
                weighted_fwhm = 2.355 * weighted_dispersion
                weighted_amplitude = 0
            
            # 计算简单平均
            mean_amplitude = np.mean(amplitudes)
            mean_position = np.mean(positions)
            mean_dispersion = np.mean(dispersions)
            mean_fwhm = 2.355 * mean_dispersion
            
            # 找到峰值点（振幅最大的像素）
            peak_idx = np.argmax(amplitudes)
            peak_x, peak_y = pixels[peak_idx]
            peak_amplitude = amplitudes[peak_idx]
            peak_position = positions[peak_idx]
            peak_dispersion = dispersions[peak_idx]
            peak_fwhm = 2.355 * peak_dispersion
            
            # 计算质心（几何中心）
            centroid_x = np.mean(x_indices)
            centroid_y = np.mean(y_indices)
            
            # 计算边界框
            x_min, x_max = np.min(x_indices), np.max(x_indices)
            y_min, y_max = np.min(y_indices), np.max(y_indices)
            
            source_info = {
                'source_id': label,
                'pixel_count': pixel_count,
                'pixels': pixels,
                'centroid': (centroid_x, centroid_y),
                'bbox': (x_min, x_max, y_min, y_max),
                
                # 峰值属性
                'peak': {
                    'position': (peak_x, peak_y),
                    'amplitude': peak_amplitude,
                    'velocity': peak_position,
                    'dispersion': peak_dispersion,
                    'fwhm': peak_fwhm
                },
                
                # 加权平均（按振幅）
                'weighted_average': {
                    'amplitude': weighted_amplitude,
                    'velocity': weighted_position,
                    'dispersion': weighted_dispersion,
                    'fwhm': weighted_fwhm
                },
                
                # 简单平均
                'simple_average': {
                    'amplitude': mean_amplitude,
                    'velocity': mean_position,
                    'dispersion': mean_dispersion,
                    'fwhm': mean_fwhm
                },
                
                # 统计信息
                'statistics': {
                    'amplitude_min': np.min(amplitudes),
                    'amplitude_max': np.max(amplitudes),
                    'amplitude_std': np.std(amplitudes),
                    'velocity_min': np.min(positions),
                    'velocity_max': np.max(positions),
                    'velocity_std': np.std(positions),
                    'dispersion_min': np.min(dispersions),
                    'dispersion_max': np.max(dispersions),
                    'dispersion_std': np.std(dispersions)
                }
            }
            
            sources.append(source_info)
    
    return sources


def read_and_analyze_individual_sources(dat_file, min_pixels=10, max_gap=2):
    """
    读取.dat文件，为每个成分的每个独立源计算平均速度、平均线宽、平均振幅
    
    参数:
        dat_file: .dat文件路径
        min_pixels: 源的最小像素数
        max_gap: 源内允许的最大间隔像素数
    
    返回:
        all_sources: 包含所有源信息的字典，按成分组织
    """
    print("\n" + "="*80)
    print("READING DAT FILE AND ANALYZING INDIVIDUAL SOURCES")
    print("="*80)
    print(f"File: {dat_file}")
    print(f"Parameters: min_pixels={min_pixels}, max_gap={max_gap}")
    
    # 1. 读取.dat文件（像素单位）
    gaussian_pixel = core.read_gaussian(dat_file)
    
    # 2. 转换为物理单位
    gaussian_physical = core.physical_gaussian(gaussian_pixel)
    
    # 3. 解析维度
    n_params_times_comp, n_y, n_x = gaussian_pixel.shape
    n_components = n_params_times_comp // 3
    
    print(f"\nSpatial dimensions: {n_x} × {n_y}")
    print(f"Number of Gaussian components: {n_components}")
    
    # 4. 存储每个成分的源
    all_sources = {}
    
    for comp in range(1, n_components + 1):
        print(f"\n{'='*70}")
        print(f"PROCESSING COMPONENT {comp}")
        print(f"{'='*70}")
        
        # 提取物理单位的数据
        amplitude = gaussian_physical[3*(comp-1)]
        position = gaussian_physical[3*(comp-1) + 1]
        dispersion = gaussian_physical[3*(comp-1) + 2]
        
        # 检查是否有非零像素
        non_zero_count = np.sum(amplitude > 0)
        print(f"Total non-zero pixels: {non_zero_count}")
        
        if non_zero_count == 0:
            print("  No pixels found in this component")
            all_sources[comp] = []
            continue
        
        # 识别独立的源
        sources = identify_individual_sources_in_component(
            amplitude, position, dispersion, min_pixels, max_gap
        )
        
        print(f"Number of individual sources identified: {len(sources)}")
        
        # 为每个源添加组件ID
        for i, source in enumerate(sources):
            source['component'] = comp
            source['global_source_id'] = f"Comp{comp}_Src{i+1}"
        
        all_sources[comp] = sources
        
        # 打印每个源的摘要
        if sources:
            print(f"\n  Source Summary:")
            print(f"  {'Src':<6} {'Pixels':<8} {'Peak Pos':<12} {'Peak Amp':<12} {'Wtd Pos':<12} {'Wtd FWHM':<12}")
            print(f"  {'-'*70}")
            
            for src in sources:
                src_id = src['source_id']
                pixels = src['pixel_count']
                peak_pos = f"{src['peak']['velocity']:.2f}"
                peak_amp = f"{src['peak']['amplitude']:.2f}"
                wtd_pos = f"{src['weighted_average']['velocity']:.2f}"
                wtd_fwhm = f"{src['weighted_average']['fwhm']:.2f}"
                
                print(f"  {src_id:<6} {pixels:<8} {peak_pos:<12} {peak_amp:<12} {wtd_pos:<12} {wtd_fwhm:<12}")
    
    return all_sources


def print_detailed_source_info(all_sources):
    """
    打印每个源的详细信息
    """
    print("\n" + "="*80)
    print("DETAILED SOURCE INFORMATION")
    print("="*80)
    
    for comp in sorted(all_sources.keys()):
        sources = all_sources[comp]
        if not sources:
            continue
        
        print(f"\n{'='*70}")
        print(f"COMPONENT {comp} - {len(sources)} SOURCES")
        print(f"{'='*70}")
        
        for src in sources:
            print(f"\n  {'-'*60}")
            print(f"  Source {src['source_id']} (Global ID: {src['global_source_id']})")
            print(f"  {'-'*60}")
            
            print(f"\n    Basic Information:")
            print(f"      Number of pixels: {src['pixel_count']}")
            print(f"      Centroid (X, Y): ({src['centroid'][0]:.2f}, {src['centroid'][1]:.2f})")
            print(f"      Bounding Box: X[{src['bbox'][0]}, {src['bbox'][1]}], Y[{src['bbox'][2]}, {src['bbox'][3]}]")
            
            print(f"\n    Peak Pixel (Highest Amplitude):")
            print(f"      Position (X, Y): ({src['peak']['position'][0]}, {src['peak']['position'][1]})")
            print(f"      Amplitude: {src['peak']['amplitude']:.4f} K")
            print(f"      Velocity: {src['peak']['velocity']:.2f} km/s")
            print(f"      Dispersion: {src['peak']['dispersion']:.4f} km/s")
            print(f"      FWHM: {src['peak']['fwhm']:.4f} km/s")
            
            print(f"\n    Weighted Average (by amplitude):")
            print(f"      Mean Amplitude: {src['weighted_average']['amplitude']:.4f} K")
            print(f"      Mean Velocity: {src['weighted_average']['velocity']:.2f} km/s")
            print(f"      Mean Dispersion: {src['weighted_average']['dispersion']:.4f} km/s")
            print(f"      Mean FWHM: {src['weighted_average']['fwhm']:.4f} km/s")
            
            print(f"\n    Simple Average:")
            print(f"      Mean Amplitude: {src['simple_average']['amplitude']:.4f} K")
            print(f"      Mean Velocity: {src['simple_average']['velocity']:.2f} km/s")
            print(f"      Mean Dispersion: {src['simple_average']['dispersion']:.4f} km/s")
            print(f"      Mean FWHM: {src['simple_average']['fwhm']:.4f} km/s")
            
            print(f"\n    Statistics:")
            print(f"      Amplitude - Min: {src['statistics']['amplitude_min']:.4f} K, "
                  f"Max: {src['statistics']['amplitude_max']:.4f} K, "
                  f"Std: {src['statistics']['amplitude_std']:.4f} K")
            print(f"      Velocity - Min: {src['statistics']['velocity_min']:.2f} km/s, "
                  f"Max: {src['statistics']['velocity_max']:.2f} km/s, "
                  f"Std: {src['statistics']['velocity_std']:.2f} km/s")
            print(f"      Dispersion - Min: {src['statistics']['dispersion_min']:.4f} km/s, "
                  f"Max: {src['statistics']['dispersion_max']:.4f} km/s, "
                  f"Std: {src['statistics']['dispersion_std']:.4f} km/s")


def save_sources_to_file(all_sources, output_file):
    """
    将每个成分每个源的详细信息保存到文本文件
    """
    with open(output_file, 'w') as f:
        f.write("="*80 + "\n")
        f.write("INDIVIDUAL SOURCE ANALYSIS RESULTS\n")
        f.write("="*80 + "\n\n")
        
        for comp in sorted(all_sources.keys()):
            sources = all_sources[comp]
            if not sources:
                continue
            
            f.write(f"\n{'='*70}\n")
            f.write(f"COMPONENT {comp} - {len(sources)} SOURCES\n")
            f.write(f"{'='*70}\n\n")
            
            for src in sources:
                f.write(f"\n  {'-'*60}\n")
                f.write(f"  Source {src['source_id']} (Global ID: {src['global_source_id']})\n")
                f.write(f"  {'-'*60}\n\n")
                
                f.write(f"    Basic Information:\n")
                f.write(f"      Number of pixels: {src['pixel_count']}\n")
                f.write(f"      Centroid (X, Y): ({src['centroid'][0]:.2f}, {src['centroid'][1]:.2f})\n")
                f.write(f"      Bounding Box: X[{src['bbox'][0]}, {src['bbox'][1]}], Y[{src['bbox'][2]}, {src['bbox'][3]}]\n\n")
                
                f.write(f"    Peak Pixel (Highest Amplitude):\n")
                f.write(f"      Position (X, Y): ({src['peak']['position'][0]}, {src['peak']['position'][1]})\n")
                f.write(f"      Amplitude: {src['peak']['amplitude']:.6f} K\n")
                f.write(f"      Velocity: {src['peak']['velocity']:.6f} km/s\n")
                f.write(f"      Dispersion: {src['peak']['dispersion']:.6f} km/s\n")
                f.write(f"      FWHM: {src['peak']['fwhm']:.6f} km/s\n\n")
                
                f.write(f"    Weighted Average (by amplitude):\n")
                f.write(f"      Mean Amplitude: {src['weighted_average']['amplitude']:.6f} K\n")
                f.write(f"      Mean Velocity: {src['weighted_average']['velocity']:.6f} km/s\n")
                f.write(f"      Mean Dispersion: {src['weighted_average']['dispersion']:.6f} km/s\n")
                f.write(f"      Mean FWHM: {src['weighted_average']['fwhm']:.6f} km/s\n\n")
                
                f.write(f"    Simple Average:\n")
                f.write(f"      Mean Amplitude: {src['simple_average']['amplitude']:.6f} K\n")
                f.write(f"      Mean Velocity: {src['simple_average']['velocity']:.6f} km/s\n")
                f.write(f"      Mean Dispersion: {src['simple_average']['dispersion']:.6f} km/s\n")
                f.write(f"      Mean FWHM: {src['simple_average']['fwhm']:.6f} km/s\n\n")
                
                f.write(f"    Statistics:\n")
                f.write(f"      Amplitude - Min: {src['statistics']['amplitude_min']:.6f} K, "
                       f"Max: {src['statistics']['amplitude_max']:.6f} K, "
                       f"Std: {src['statistics']['amplitude_std']:.6f} K\n")
                f.write(f"      Velocity - Min: {src['statistics']['velocity_min']:.6f} km/s, "
                       f"Max: {src['statistics']['velocity_max']:.6f} km/s, "
                       f"Std: {src['statistics']['velocity_std']:.6f} km/s\n")
                f.write(f"      Dispersion - Min: {src['statistics']['dispersion_min']:.6f} km/s, "
                       f"Max: {src['statistics']['dispersion_max']:.6f} km/s, "
                       f"Std: {src['statistics']['dispersion_std']:.6f} km/s\n")
    
    print(f"\nDetailed source information saved to: {output_file}")


def create_summary_table(all_sources):
    """
    创建汇总表格，显示每个成分每个源的关键参数
    """
    print("\n" + "="*100)
    print("SUMMARY TABLE - ALL SOURCES BY COMPONENT")
    print("="*100)
    
    print(f"\n{'Comp':<6} {'Source':<10} {'Pixels':<8} {'Peak Pos':<12} {'Peak Amp':<12} {'Wtd Pos':<12} {'Wtd FWHM':<12} {'Wtd Amp':<12}")
    print("-"*100)
    
    for comp in sorted(all_sources.keys()):
        sources = all_sources[comp]
        if not sources:
            continue
        
        for src in sources:
            comp_str = f"Comp{comp}"
            src_id = src['source_id']
            pixels = src['pixel_count']
            peak_pos = f"{src['peak']['velocity']:.2f}"
            peak_amp = f"{src['peak']['amplitude']:.2f}"
            wtd_pos = f"{src['weighted_average']['velocity']:.2f}"
            wtd_fwhm = f"{src['weighted_average']['fwhm']:.2f}"
            wtd_amp = f"{src['weighted_average']['amplitude']:.2f}"
            
            print(f"{comp_str:<6} {src_id:<10} {pixels:<8} {peak_pos:<12} {peak_amp:<12} {wtd_pos:<12} {wtd_fwhm:<12} {wtd_amp:<12}")
    
    print("\n" + "="*100)
    print("Notes:")
    print("  - Peak: Pixel with highest amplitude")
    print("  - Wtd: Weighted average by amplitude")
    print("  - FWHM: Full Width at Half Maximum (2.355 × dispersion)")
    print("="*100)


def main():
    """
    主函数
    """
    # 请修改这些路径
    DAT_FILE ="./n_gauss=3/output_individual_source/source_filtered_rohsa_pixel.dat"   # 要分析的.dat文件
    OUTPUT_DIR = "./n_gauss=3/output_individual_source"  # 输出目录，"."表示当前目录
    
    # 源识别参数
    MIN_PIXELS = 10  # 源的最小像素数
    MAX_GAP = 2      # 源内允许的最大间隔像素数
    
    # 创建输出目录
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    # 分析每个成分的每个独立源
    all_sources = read_and_analyze_individual_sources(DAT_FILE, MIN_PIXELS, MAX_GAP)
    
    if all_sources:
        # 打印详细源信息
        print_detailed_source_info(all_sources)
        
        # 保存到文件
        output_file = os.path.join(OUTPUT_DIR, 'individual_source_analysis.txt')
        save_sources_to_file(all_sources, output_file)
        
        # 打印汇总表格
        create_summary_table(all_sources)
        
        # 统计总源数
        total_sources = sum(len(sources) for sources in all_sources.values())
        print(f"\n✓ Analysis completed!")
        print(f"  Total components: {len(all_sources)}")
        print(f"  Total individual sources: {total_sources}")
        print(f"  Results saved to: {output_file}")
    else:
        print("\n✗ No sources found in the file!")


if __name__ == "__main__":
    main()


READING DAT FILE AND ANALYZING INDIVIDUAL SOURCES
File: ./n_gauss=3/output_individual_source/source_filtered_rohsa_pixel.dat
Parameters: min_pixels=10, max_gap=2
Opening data file

Spatial dimensions: 153 × 89
Number of Gaussian components: 3

PROCESSING COMPONENT 1
Total non-zero pixels: 4872
Number of individual sources identified: 5

  Source Summary:
  Src    Pixels   Peak Pos     Peak Amp     Wtd Pos      Wtd FWHM    
  ----------------------------------------------------------------------
  1      4660     -236.77      1.50         -238.28      29.05       
  2      106      -239.70      0.40         -236.26      43.67       
  3      13       -245.55      0.29         -246.26      28.29       
  4      75       -241.03      0.31         -243.72      29.27       
  5      18       -232.40      0.25         -232.94      25.98       

PROCESSING COMPONENT 2
Total non-zero pixels: 291
Number of individual sources identified: 1

  Source Summary:
  Src    Pixels   Peak Pos     Peak 

In [5]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from scipy import ndimage
import os
from matplotlib.colors import ListedColormap
from collections import defaultdict


# ==================== 1. 读取DAT文件（同时获取像素单位和物理单位） ====================
def read_filtered_dat(dat_file):
    """
    读取筛选后的ROHSA DAT文件
    返回像素单位和物理单位的结果
    """
    print("\n" + "="*60)
    print("READING FILTERED ROHSA DAT FILE")
    print("="*60)
    
    # 使用core模块读取像素单位的gaussian
    print("Reading Gaussian parameters (pixel units)...")
    gaussian_pixel = core.read_gaussian(dat_file)
    
    print(f"Gaussian pixel array shape: {gaussian_pixel.shape}")
    print(f"Data type: {gaussian_pixel.dtype}")
    
    # 转换为物理单位
    print("Converting to physical units...")
    gaussian_physical = core.physical_gaussian(gaussian_pixel)
    
    # 检查返回值类型
    if isinstance(gaussian_physical, (int, float)):
        print("Warning: physical_gaussian returned a scalar. Using pixel units as fallback.")
        gaussian_physical = gaussian_pixel
    
    # 解析维度: (n_parameters * n_components, n_y, n_x)
    n_params_times_comp, n_y, n_x = gaussian_pixel.shape
    n_components = n_params_times_comp // 3
    
    print(f"Spatial dimensions: X={n_x}, Y={n_y}")
    print(f"Number of Gaussian components: {n_components}")
    
    # 提取每个成分的参数
    results = {}
    
    for comp in range(1, n_components + 1):
        # 像素单位
        amp_pixel = gaussian_pixel[3*(comp-1)]
        pos_pixel = gaussian_pixel[3*(comp-1) + 1]
        dis_pixel = gaussian_pixel[3*(comp-1) + 2]
        
        # 物理单位
        if isinstance(gaussian_physical, np.ndarray):
            amp_phys = gaussian_physical[3*(comp-1)]
            pos_phys = gaussian_physical[3*(comp-1) + 1]
            dis_phys = gaussian_physical[3*(comp-1) + 2]
        else:
            amp_phys = amp_pixel
            pos_phys = pos_pixel
            dis_phys = dis_pixel
        
        results[f'comp{comp}'] = {
            'amplitude_pixel': amp_pixel,
            'position_pixel': pos_pixel,
            'dispersion_pixel': dis_pixel,
            'amplitude_phys': amp_phys,
            'position_phys': pos_phys,
            'dispersion_phys': dis_phys,
            'component_id': comp
        }
        
        print(f"\nComponent {comp}:")
        print(f"  Non-zero pixels (phys): {np.sum(amp_phys > 0)}")
        if np.any(amp_phys > 0):
            print(f"  Position range (phys): [{np.min(pos_phys[pos_phys!=0]):.2f}, {np.max(pos_phys):.2f}] km/s")
    
    results['n_components'] = n_components
    results['shape'] = (n_x, n_y)
    results['original_data_pixel'] = gaussian_pixel
    results['original_data_physical'] = gaussian_physical
    
    return results

# ==================== 2. 提取每个成分的源 ====================
def extract_sources_from_component(results, comp, min_pixels=10, max_gap=2):
    """
    从单个成分提取源
    """
    amplitude = results[f'comp{comp}']['amplitude_phys']
    position = results[f'comp{comp}']['position_phys']
    dispersion = results[f'comp{comp}']['dispersion_phys']
    
    # 创建二值mask
    binary_mask = amplitude > 0
    
    if not np.any(binary_mask):
        return []
    
    # 连通区域分析
    structure = ndimage.generate_binary_structure(2, 1)
    if max_gap > 1:
        structure = ndimage.iterate_structure(structure, max_gap - 1)
    
    labeled_mask, num_features = ndimage.label(binary_mask, structure=structure)
    
    sources = []
    
    for label in range(1, num_features + 1):
        mask = labeled_mask == label
        pixel_count = np.sum(mask)
        
        if pixel_count >= min_pixels:
            y_indices, x_indices = np.where(mask)
            
            amplitudes = []
            positions = []
            dispersions = []
            pixels = []
            
            for y, x in zip(y_indices, x_indices):
                amp = amplitude[y, x]
                pos = position[y, x]
                dis = dispersion[y, x]
                
                amplitudes.append(amp)
                positions.append(pos)
                dispersions.append(dis)
                pixels.append((x, y))
            
            amplitudes = np.array(amplitudes)
            positions = np.array(positions)
            dispersions = np.array(dispersions)
            
            # 计算加权平均（以振幅为权重）
            total_amp = np.sum(amplitudes)
            if total_amp > 0:
                weighted_position = np.sum(positions * amplitudes) / total_amp
                weighted_dispersion = np.sum(dispersions * amplitudes) / total_amp
                weighted_fwhm = 2.355 * weighted_dispersion
                weighted_amplitude = total_amp / pixel_count
            else:
                weighted_position = np.mean(positions)
                weighted_dispersion = np.mean(dispersions)
                weighted_fwhm = 2.355 * weighted_dispersion
                weighted_amplitude = 0
            
            # 计算质心
            centroid_x = np.mean(x_indices)
            centroid_y = np.mean(y_indices)
            
            source_info = {
                'source_id': label,
                'component': comp,
                'pixel_count': pixel_count,
                'pixels': pixels,
                'centroid': (centroid_x, centroid_y),
                'weighted_average': {
                    'amplitude': weighted_amplitude,
                    'velocity': weighted_position,
                    'dispersion': weighted_dispersion,
                    'fwhm': weighted_fwhm
                },
                'bbox': (np.min(x_indices), np.max(x_indices), 
                        np.min(y_indices), np.max(y_indices))
            }
            sources.append(source_info)
    
    return sources

# ==================== 3. 判断合并条件 ====================
def check_merge_condition(source1, source2, velocity_threshold_factor=0.7):
    """
    判断两个源是否可以合并
    条件：速度差 < velocity_threshold_factor * 二者之中较宽的FWHM
    """
    v1 = source1['weighted_average']['velocity']
    v2 = source2['weighted_average']['velocity']
    fwhm1 = source1['weighted_average']['fwhm']
    fwhm2 = source2['weighted_average']['fwhm']
    
    dv = abs(v1 - v2)
    wider_fwhm = max(fwhm1, fwhm2)
    threshold = velocity_threshold_factor * wider_fwhm
    
    can_merge = dv < threshold
    
    return can_merge, dv, threshold

# ==================== 4. 合并多个源 ====================
def merge_sources(sources_to_merge, results):
    """
    合并多个源
    """
    if len(sources_to_merge) == 0:
        return None
    
    # 收集所有像素
    all_pixels = []
    all_components = []
    
    for src in sources_to_merge:
        all_pixels.extend(src['pixels'])
        all_components.append(src['component'])
    
    all_pixels = list(set(all_pixels))
    
    # 计算合并后的属性（振幅加权）
    total_amp = sum([src['weighted_average']['amplitude'] * src['pixel_count'] 
                     for src in sources_to_merge])
    
    if total_amp > 0:
        weighted_velocities = []
        weighted_dispersions = []
        
        for src in sources_to_merge:
            weight = src['weighted_average']['amplitude'] * src['pixel_count'] / total_amp
            weighted_velocities.append(src['weighted_average']['velocity'] * weight)
            weighted_dispersions.append(src['weighted_average']['dispersion'] * weight)
        
        merged_velocity = sum(weighted_velocities)
        merged_dispersion = sum(weighted_dispersions)
    else:
        merged_velocity = np.mean([src['weighted_average']['velocity'] for src in sources_to_merge])
        merged_dispersion = np.mean([src['weighted_average']['dispersion'] for src in sources_to_merge])
    
    # 计算质心
    x_coords = [p[0] for p in all_pixels]
    y_coords = [p[1] for p in all_pixels]
    
    merged_source = {
        'merged_id': None,
        'pixel_count': len(all_pixels),
        'pixels': all_pixels,
        'centroid': (np.mean(x_coords), np.mean(y_coords)),
        'component_sources': sources_to_merge,
        'components': list(set(all_components)),
        'weighted_average': {
            'velocity': merged_velocity,
            'dispersion': merged_dispersion,
            'fwhm': 2.355 * merged_dispersion,
            'amplitude': total_amp / len(all_pixels) if total_amp > 0 else 0
        },
        'is_merged': True
    }
    
    return merged_source

# ==================== 5. 创建最终参数图 ====================
def create_final_parameter_maps(all_sources, results, shape):
    """
    创建最终的参数图
    """
    n_x, n_y = shape
    n_components = results['n_components']
    
    merged_amplitude = np.zeros((n_y, n_x))
    merged_position = np.zeros((n_y, n_x))
    merged_dispersion = np.zeros((n_y, n_x))
    merged_source_id = np.zeros((n_y, n_x), dtype=int)
    
    for source in all_sources:
        source_id = source['merged_id']
        
        for x, y in source['pixels']:
            if 0 <= y < n_y and 0 <= x < n_x:
                pixel_amplitudes = []
                pixel_positions = []
                pixel_dispersions = []
                
                for comp_src in source['component_sources']:
                    comp = comp_src['component']
                    if (x, y) in comp_src['pixels']:
                        amp = results[f'comp{comp}']['amplitude_phys'][y, x]
                        pos = results[f'comp{comp}']['position_phys'][y, x]
                        dis = results[f'comp{comp}']['dispersion_phys'][y, x]
                        
                        if amp > 0:
                            pixel_amplitudes.append(amp)
                            pixel_positions.append(pos)
                            pixel_dispersions.append(dis)
                
                if pixel_amplitudes:
                    weights = np.array(pixel_amplitudes) / np.sum(pixel_amplitudes)
                    merged_amplitude[y, x] = np.sum(pixel_amplitudes)
                    merged_position[y, x] = np.sum(np.array(pixel_positions) * weights)
                    merged_dispersion[y, x] = np.sum(np.array(pixel_dispersions) * weights)
                    merged_source_id[y, x] = source_id
    
    return merged_amplitude, merged_position, merged_dispersion, merged_source_id

# ==================== 6. 绘制所有源 ====================
def plot_all_sources(all_sources, merged_amplitude, merged_position, 
                     merged_dispersion, merged_source_id, output_dir):
    """
    绘制所有源
    """
    print("\n" + "="*60)
    print("PLOTTING ALL SOURCES")
    print("="*60)
    
    n_sources = len(all_sources)
    
    # 图1：所有源的mask
    fig, axes = plt.subplots(2, 2, figsize=(16, 14))
    
    # 源mask
    if n_sources > 0:
        colors = plt.cm.tab20(np.linspace(0, 1, n_sources))
        colors = np.vstack([[0,0,0,1], colors])
        cmap = ListedColormap(colors)
        
        im1 = axes[0, 0].imshow(merged_source_id, origin='lower', cmap=cmap,
                                vmin=0, vmax=n_sources)
        axes[0, 0].set_title(f'All Sources ({n_sources} sources)', fontsize=14)
        cbar = plt.colorbar(im1, ax=axes[0, 0], ticks=range(n_sources+1), location='bottom')
        cbar.set_label('Source ID')
        
        # 标注源的中心
        for source in all_sources:
            x, y = source['centroid']
            axes[0, 0].plot(x, y, 'r+', markersize=10, markeredgewidth=2)
            axes[0, 0].text(x+2, y+2, f'{source["merged_id"]}', 
                           color='white', fontsize=10, weight='bold',
                           bbox=dict(boxstyle='round', facecolor='black', alpha=0.5))
    else:
        axes[0, 0].imshow(merged_source_id, origin='lower', cmap='gray')
        axes[0, 0].set_title('All Sources - No Sources', fontsize=14)
    
    axes[0, 0].set_xlabel('X pixel')
    axes[0, 0].set_ylabel('Y pixel')
    
    # 振幅图
    im2 = axes[0, 1].imshow(merged_amplitude, origin='lower', cmap='inferno')
    axes[0, 1].set_title('All Sources - Total Amplitude', fontsize=14)
    plt.colorbar(im2, ax=axes[0, 1], label='Brightness Temp. [K]', location='bottom')
    axes[0, 1].set_xlabel('X pixel')
    axes[0, 1].set_ylabel('Y pixel')
    
    # 位置图
    im3 = axes[1, 0].imshow(merged_position, origin='lower', cmap='coolwarm', vmin=-300, vmax=-200)
    axes[1, 0].set_title('All Sources - Weighted Position', fontsize=14)
    plt.colorbar(im3, ax=axes[1, 0], label='Velocity [km s$^{-1}$]', location='bottom')
    axes[1, 0].set_xlabel('X pixel')
    axes[1, 0].set_ylabel('Y pixel')
    
    # 宽度图
    im4 = axes[1, 1].imshow(merged_dispersion, origin='lower', cmap='cubehelix', vmin=0, vmax=20)
    axes[1, 1].set_title('All Sources - Weighted Dispersion', fontsize=14)
    plt.colorbar(im4, ax=axes[1, 1], label='Dispersion [km s$^{-1}$]', location='bottom')
    axes[1, 1].set_xlabel('X pixel')
    axes[1, 1].set_ylabel('Y pixel')
    
    plt.suptitle('All Sources Overview', fontsize=16, y=1.02)
    plt.tight_layout()
    
    all_file = os.path.join(output_dir, 'all_sources_overview.png')
    plt.savefig(all_file, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"All sources overview saved to: {all_file}")
    
    # 图2：每个源的组成图
    if n_sources > 0:
        n_cols = min(3, n_sources)
        n_rows = (n_sources + n_cols - 1) // n_cols
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 4*n_rows))
        if n_sources == 1:
            axes = np.array([axes])
        axes = axes.flatten()
        
        for i, source in enumerate(all_sources):
            ax = axes[i]
            source_mask = merged_source_id == source['merged_id']
            ax.imshow(source_mask, origin='lower', cmap='Blues', alpha=0.3)
            
            # 标注组成它的原始源
            colors = ['red', 'green', 'blue', 'purple', 'orange', 'brown']
            for j, comp_src in enumerate(source['component_sources']):
                comp_mask = np.zeros_like(source_mask)
                for x, y in comp_src['pixels']:
                    if 0 <= y < comp_mask.shape[0] and 0 <= x < comp_mask.shape[1]:
                        comp_mask[y, x] = True
                
                edges = ndimage.binary_dilation(comp_mask) & ~comp_mask
                y_edges, x_edges = np.where(edges)
                ax.scatter(x_edges, y_edges, c=colors[j % len(colors)], 
                          s=1, label=f'Comp{comp_src["component"]}-Src{comp_src["source_id"]}', alpha=0.8)
            
            # 显示源的类型
            if source.get('is_merged', False):
                source_type = f'Merged (Comp{source["components"]})'
            else:
                source_type = f'Comp{source["components"][0]}'
            
            ax.set_title(f'Source {source["merged_id"]}: {source_type}\n'
                        f'{source["pixel_count"]} pixels, '
                        f'v={source["weighted_average"]["velocity"]:.1f} km/s', fontsize=9)
            ax.set_xlabel('X pixel')
            ax.set_ylabel('Y pixel')
            if len(source['component_sources']) <= 3:
                ax.legend(loc='upper right', fontsize=6)
        
        for i in range(n_sources, len(axes)):
            axes[i].axis('off')
        
        plt.suptitle('Individual Sources Composition', fontsize=14)
        plt.tight_layout()
        
        comp_file = os.path.join(output_dir, 'all_sources_composition.png')
        plt.savefig(comp_file, dpi=150, bbox_inches='tight')
        plt.close()
        print(f"All sources composition saved to: {comp_file}")

# ==================== 7. 保存DAT文件 ====================
def save_final_dat(results, all_sources, original_dat_file, output_file):
    """
    保存最终的DAT文件（像素单位）
    """
    print("\n" + "="*60)
    print("SAVING FINAL RESULTS TO .DAT FORMAT (PIXEL UNITS)")
    print("="*60)
    
    try:
        n_x, n_y = results['shape']
        n_components = results['n_components']
        
        with open(original_dat_file, 'r') as f:
            header_lines = []
            for i in range(27):
                line = f.readline()
                header_lines.append(line)
        
        source_id_map = {}
        for source in all_sources:
            for x, y in source['pixels']:
                source_id_map[(x, y)] = source['merged_id']
        
        data_lines = []
        
        for y in range(n_y):
            for x in range(n_x):
                for comp in range(1, n_components + 1):
                    if (x, y) in source_id_map:
                        for source in all_sources:
                            if source['merged_id'] == source_id_map[(x, y)]:
                                pixel_amp = 0
                                pixel_pos = 0
                                pixel_dis = 0
                                for comp_src in source['component_sources']:
                                    if comp_src['component'] == comp and (x, y) in comp_src['pixels']:
                                        amp = results[f'comp{comp}']['amplitude_pixel'][y, x]
                                        pos = results[f'comp{comp}']['position_pixel'][y, x]
                                        dis = results[f'comp{comp}']['dispersion_pixel'][y, x]
                                        if amp > 0:
                                            pixel_amp = amp
                                            pixel_pos = pos
                                            pixel_dis = dis
                                            break
                                line = f"    {y:4d}    {x:4d}    {pixel_amp:20.16f}    {pixel_pos:20.16f}    {pixel_dis:20.16f}\n"
                                data_lines.append(line)
                                break
                    else:
                        line = f"    {y:4d}    {x:4d}    {0:20.16f}    {0:20.16f}    {0:20.16f}\n"
                        data_lines.append(line)
        
        with open(output_file, 'w') as f:
            f.writelines(header_lines)
            f.writelines(data_lines)
        
        print(f"\nFinal .dat file saved to: {output_file}")
        
    except Exception as e:
        print(f"Error saving final .dat: {e}")

# ==================== 8. 保存源信息 ====================
def save_source_info(all_sources, output_dir):
    """
    保存源信息到文本文件
    """
    info_file = os.path.join(output_dir, 'final_source_information.txt')
    
    with open(info_file, 'w') as f:
        f.write("="*60 + "\n")
        f.write("FINAL SOURCE INFORMATION\n")
        f.write("="*60 + "\n\n")
        
        f.write(f"Total sources: {len(all_sources)}\n\n")
        
        for source in all_sources:
            f.write(f"\n{'='*50}\n")
            f.write(f"SOURCE {source['merged_id']}\n")
            f.write(f"{'='*50}\n\n")
            
            if source.get('is_merged', False):
                f.write(f"Type: MERGED\n")
                f.write(f"Components involved: {source['components']}\n")
            else:
                f.write(f"Type: SINGLE COMPONENT\n")
                f.write(f"Component: {source['components'][0]}\n")
            
            f.write(f"Number of pixels: {source['pixel_count']}\n")
            f.write(f"Centroid (X, Y): ({source['centroid'][0]:.2f}, {source['centroid'][1]:.2f})\n")
            f.write(f"Mean Velocity: {source['weighted_average']['velocity']:.6f} km/s\n")
            f.write(f"Mean FWHM: {source['weighted_average']['fwhm']:.6f} km/s\n")
            f.write(f"Mean Amplitude: {source['weighted_average']['amplitude']:.6f} K\n\n")
            
            f.write("Original Sources:\n")
            for comp_src in source['component_sources']:
                f.write(f"  Component {comp_src['component']}, Source {comp_src['source_id']}:\n")
                f.write(f"    Velocity: {comp_src['weighted_average']['velocity']:.6f} km/s\n")
                f.write(f"    FWHM: {comp_src['weighted_average']['fwhm']:.6f} km/s\n")
                f.write(f"    Pixels: {comp_src['pixel_count']}\n")
    
    print(f"Source information saved to: {info_file}")

# ==================== 9. 主函数 ====================
def main(dat_file, original_dat_file, output_dir, 
         min_pixels=10, max_gap=2, velocity_threshold_factor=0.7):
    """
    主函数 - 分别比较Component 2的第一个源和Component 3的第1、2个源
    """
    print("\n" + "="*60)
    print("SOURCE MERGING (Component 2 vs Component 3)")
    print("="*60)
    print(f"Input DAT file: {dat_file}")
    print(f"Output directory: {output_dir}")
    print(f"Min pixels per source: {min_pixels}")
    print(f"Max spatial gap: {max_gap}")
    print(f"Velocity threshold factor: {velocity_threshold_factor}")
    
    os.makedirs(output_dir, exist_ok=True)
    
    # 1. 读取DAT文件
    results = read_filtered_dat(dat_file)
    shape = results['shape']
    n_components = results['n_components']
    
    print(f"\n检测到 {n_components} 个高斯分量")
    
    # 2. 提取每个成分的源
    print("\n" + "="*60)
    print("EXTRACTING SOURCES FROM EACH COMPONENT")
    print("="*60)
    
    comp1_sources = extract_sources_from_component(results, 1, min_pixels, max_gap)
    comp2_sources = extract_sources_from_component(results, 2, min_pixels, max_gap)
    comp3_sources = extract_sources_from_component(results, 3, min_pixels, max_gap)
    
    print(f"\nComponent 1: {len(comp1_sources)} sources")
    for src in comp1_sources:
        print(f"  Source {src['source_id']}: {src['pixel_count']} pixels, v={src['weighted_average']['velocity']:.2f} km/s, FWHM={src['weighted_average']['fwhm']:.2f}")
    
    print(f"\nComponent 2: {len(comp2_sources)} sources")
    for src in comp2_sources:
        print(f"  Source {src['source_id']}: {src['pixel_count']} pixels, v={src['weighted_average']['velocity']:.2f} km/s, FWHM={src['weighted_average']['fwhm']:.2f}")
    
    print(f"\nComponent 3: {len(comp3_sources)} sources")
    for src in comp3_sources:
        print(f"  Source {src['source_id']}: {src['pixel_count']} pixels, v={src['weighted_average']['velocity']:.2f} km/s, FWHM={src['weighted_average']['fwhm']:.2f}")
    
    # 3. 分别比较Component 2的第一个源和Component 3的第1、2个源
    all_final_sources = []
    check_results = []
    used_sources = set()
    sources_to_merge = []
    
    if comp2_sources:
        comp2_source = comp2_sources[0]  # Component 2的第一个源
        print(f"\n{'='*60}")
        print(f"检查 Component 2 源 1 (v={comp2_source['weighted_average']['velocity']:.2f} km/s, FWHM={comp2_source['weighted_average']['fwhm']:.2f})")
        print(f"{'='*60}")
        
        # 先添加Component 2源到待合并列表
        sources_to_merge.append(comp2_source)
        
        # 检查Component 3的第1、2个源
        comp3_sources_to_check = []
        if len(comp3_sources) >= 1:
            comp3_sources_to_check.append(comp3_sources[0])
        if len(comp3_sources) >= 2:
            comp3_sources_to_check.append(comp3_sources[1])
        
        for comp3_src in comp3_sources_to_check:
            can_merge, dv, threshold = check_merge_condition(
                comp2_source, comp3_src, velocity_threshold_factor
            )
            
            check_results.append({
                'type': f'Comp2-Src1 vs Comp3-Src{comp3_src["source_id"]}',
                'v1': comp2_source['weighted_average']['velocity'],
                'v2': comp3_src['weighted_average']['velocity'],
                'fwhm1': comp2_source['weighted_average']['fwhm'],
                'fwhm2': comp3_src['weighted_average']['fwhm'],
                'dv': dv,
                'threshold': threshold,
                'can_merge': can_merge
            })
            
            print(f"\n  Comp2-Src1 vs Comp3-Src{comp3_src['source_id']}:")
            print(f"    v1={comp2_source['weighted_average']['velocity']:.2f}, v2={comp3_src['weighted_average']['velocity']:.2f}, dv={dv:.2f}")
            print(f"    fwhm1={comp2_source['weighted_average']['fwhm']:.2f}, fwhm2={comp3_src['weighted_average']['fwhm']:.2f}")
            print(f"    Wider FWHM: {max(comp2_source['weighted_average']['fwhm'], comp3_src['weighted_average']['fwhm']):.2f}")
            print(f"    Threshold (0.7 × wider FWHM): {threshold:.2f}")
            print(f"    Can merge: {can_merge}")
            
            if can_merge:
                sources_to_merge.append(comp3_src)
                used_sources.add(('comp3', comp3_src['source_id']))
                print(f"    ✓ 添加到合并列表")
            else:
                print(f"    ✗ 不满足合并条件")
        
        # 如果有可合并的源（除了Comp2源本身）
        if len(sources_to_merge) > 1:
            merged = merge_sources(sources_to_merge, results)
            if merged:
                all_final_sources.append(merged)
                used_sources.add(('comp2', comp2_source['source_id']))
                print(f"\n✓ 创建合并源，包含 {len(sources_to_merge)} 个源")
                print(f"  合并的源: Component 2-1 + Component 3 源 {[src['source_id'] for src in sources_to_merge[1:]]}")
        else:
            # 不能合并，保留为独立源
            all_final_sources.append({
                'merged_id': None,
                'pixel_count': comp2_source['pixel_count'],
                'pixels': comp2_source['pixels'],
                'centroid': comp2_source['centroid'],
                'component_sources': [comp2_source],
                'components': [2],
                'weighted_average': comp2_source['weighted_average'],
                'is_merged': False
            })
            used_sources.add(('comp2', comp2_source['source_id']))
            print(f"\n✗ 没有可合并的源，保留为独立源")
    else:
        print("\nComponent 2 没有源")
    
    # 4. 添加剩余的源（未使用的源）
    temp_sources = []
    
    # 添加Component 1的所有源
    for src in comp1_sources:
        temp_sources.append({
            'merged_id': None,
            'pixel_count': src['pixel_count'],
            'pixels': src['pixels'],
            'centroid': src['centroid'],
            'component_sources': [src],
            'components': [1],
            'weighted_average': src['weighted_average'],
            'is_merged': False
        })
    
    # 添加已经创建的合并源
    for src in all_final_sources:
        temp_sources.append(src)
    
    # 添加Component 2中未使用的源（除了第一个）
    for src in comp2_sources:
        if ('comp2', src['source_id']) not in used_sources:
            temp_sources.append({
                'merged_id': None,
                'pixel_count': src['pixel_count'],
                'pixels': src['pixels'],
                'centroid': src['centroid'],
                'component_sources': [src],
                'components': [2],
                'weighted_average': src['weighted_average'],
                'is_merged': False
            })
    
    # 添加Component 3中未使用的源（除了被合并的）
    for src in comp3_sources:
        if ('comp3', src['source_id']) not in used_sources:
            temp_sources.append({
                'merged_id': None,
                'pixel_count': src['pixel_count'],
                'pixels': src['pixels'],
                'centroid': src['centroid'],
                'component_sources': [src],
                'components': [3],
                'weighted_average': src['weighted_average'],
                'is_merged': False
            })
    
    # 重新分配ID
    for i, source in enumerate(temp_sources):
        source['merged_id'] = i + 1
    
    all_final_sources = temp_sources
    
    print(f"\n{'='*60}")
    print("最终源列表")
    print("="*60)
    print(f"总源数: {len(all_final_sources)}")
    for src in all_final_sources:
        if src.get('is_merged', False):
            print(f"  源 {src['merged_id']}: 合并源 (分量 {src['components']}), "
                  f"{src['pixel_count']} 像素, v={src['weighted_average']['velocity']:.2f} km/s")
        else:
            print(f"  源 {src['merged_id']}: 分量 {src['components'][0]}, "
                  f"{src['pixel_count']} 像素, v={src['weighted_average']['velocity']:.2f} km/s")
    
    # 5. 创建参数图
    merged_amplitude, merged_position, merged_dispersion, merged_source_id = \
        create_final_parameter_maps(all_final_sources, results, shape)
    
    # 6. 绘制所有源
    plot_all_sources(all_final_sources, merged_amplitude, merged_position,
                     merged_dispersion, merged_source_id, output_dir)
    
    # 7. 保存源信息
    save_source_info(all_final_sources, output_dir)
    
    # 8. 保存DAT文件
    final_dat = os.path.join(output_dir, 'final_sources.dat')
    save_final_dat(results, all_final_sources, original_dat_file, final_dat)
    
    # 9. 保存合并检查结果
    check_file = os.path.join(output_dir, 'merge_check_results.txt')
    with open(check_file, 'w') as f:
        f.write("="*60 + "\n")
        f.write("合并条件检查结果\n")
        f.write("="*60 + "\n\n")
        
        for check in check_results:
            f.write(f"\n{check['type']}:\n")
            f.write(f"  Component 2 速度: {check['v1']:.4f} km/s\n")
            f.write(f"  Component 2 FWHM: {check['fwhm1']:.4f} km/s\n")
            f.write(f"  Component 3 速度: {check['v2']:.4f} km/s\n")
            f.write(f"  Component 3 FWHM: {check['fwhm2']:.4f} km/s\n")
            f.write(f"  较宽的FWHM: {max(check['fwhm1'], check['fwhm2']):.4f} km/s\n")
            f.write(f"  速度差: {check['dv']:.4f} km/s\n")
            f.write(f"  阈值 (0.7 × 较宽FWHM): {check['threshold']:.4f} km/s\n")
            f.write(f"  是否合并: {check['can_merge']}\n")
    
    print(f"\n合并检查结果保存至: {check_file}")
    
    print("\n" + "="*60)
    print("处理完成")
    print("="*60)
    print(f"\n输出文件保存至: {output_dir}")
    print(f"  - all_sources_overview.png")
    print(f"  - all_sources_composition.png")
    print(f"  - final_source_information.txt")
    print(f"  - final_sources.dat")
    print(f"  - merge_check_results.txt")


# ==================== 10. 脚本执行 ====================
if __name__ == "__main__":
    # 请修改这些路径
    DAT_FILE = "./n_gauss=3/output_individual_source/source_filtered_rohsa_pixel.dat"
    ORIGINAL_DAT_FILE = "MS_ROHSA_3ngauss_1_3D_1.dat"
    OUTPUT_DIR = "./n_gauss=3/output_merged_sources"
    
    # 参数设置
    MIN_PIXELS = 10
    MAX_GAP = 2
    VELOCITY_THRESHOLD_FACTOR = 0.7
    
    # 运行
    main(
        dat_file=DAT_FILE,
        original_dat_file=ORIGINAL_DAT_FILE,
        output_dir=OUTPUT_DIR,
        min_pixels=MIN_PIXELS,
        max_gap=MAX_GAP,
        velocity_threshold_factor=VELOCITY_THRESHOLD_FACTOR
    )


SOURCE MERGING (Component 2 vs Component 3)
Input DAT file: ./n_gauss=3/output_individual_source/source_filtered_rohsa_pixel.dat
Output directory: ./n_gauss=3/output_merged_sources
Min pixels per source: 10
Max spatial gap: 2
Velocity threshold factor: 0.7

READING FILTERED ROHSA DAT FILE
Reading Gaussian parameters (pixel units)...
Opening data file
Gaussian pixel array shape: (9, 89, 153)
Data type: float64
Converting to physical units...
Spatial dimensions: X=153, Y=89
Number of Gaussian components: 3

Component 1:
  Non-zero pixels (phys): 4872
  Position range (phys): [-255.87, -149.83] km/s

Component 2:
  Non-zero pixels (phys): 291
  Position range (phys): [-282.77, -149.83] km/s

Component 3:
  Non-zero pixels (phys): 56
  Position range (phys): [-297.10, -149.83] km/s

检测到 3 个高斯分量

EXTRACTING SOURCES FROM EACH COMPONENT

Component 1: 5 sources
  Source 1: 4660 pixels, v=-238.28 km/s, FWHM=29.05
  Source 2: 106 pixels, v=-236.26 km/s, FWHM=43.67
  Source 3: 13 pixels, v=-246.

### 更好的绘图

In [8]:
# ==================== 完整的导入语句 ====================
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from scipy import ndimage
import os
from matplotlib.colors import ListedColormap
from collections import defaultdict
from astropy.wcs import WCS
from astropy import units as u
from spectral_cube import SpectralCube
from pvextractor import extract_pv_slice, Path
from skimage import measure
from reproject import reproject_interp
import matplotlib.ticker as ticker
import matplotlib.font_manager as fm

# 设置字体
try:
    # 尝试设置Times New Roman
    plt.rcParams['font.family'] = 'Times New Roman'
    # 测试字体是否可用
    fm.findfont('Times New Roman', fallback_to_default=False)
except:
    print("Times New Roman not available, using default font")
    plt.rcParams['font.family'] = 'serif'
    plt.rcParams['font.serif'] = ['DejaVu Serif', 'Times New Roman', 'Times']

# 设置数学字体为STIX（类似Times）
plt.rcParams['mathtext.fontset'] = 'stix'


# ==================== 1. 读取DAT文件（同时获取像素单位和物理单位） ====================
def read_filtered_dat(dat_file):
    """
    读取筛选后的ROHSA DAT文件
    返回像素单位和物理单位的结果
    """
    print("\n" + "="*60)
    print("READING FILTERED ROHSA DAT FILE")
    print("="*60)
    
    # 使用core模块读取像素单位的gaussian
    print("Reading Gaussian parameters (pixel units)...")
    gaussian_pixel = core.read_gaussian(dat_file)
    
    print(f"Gaussian pixel array shape: {gaussian_pixel.shape}")
    print(f"Data type: {gaussian_pixel.dtype}")
    
    # 转换为物理单位
    print("Converting to physical units...")
    gaussian_physical = core.physical_gaussian(gaussian_pixel)
    
    # 检查返回值类型
    if isinstance(gaussian_physical, (int, float)):
        print("Warning: physical_gaussian returned a scalar. Using pixel units as fallback.")
        gaussian_physical = gaussian_pixel
    
    # 解析维度: (n_parameters * n_components, n_y, n_x)
    n_params_times_comp, n_y, n_x = gaussian_pixel.shape
    n_components = n_params_times_comp // 3
    
    print(f"Spatial dimensions: X={n_x}, Y={n_y}")
    print(f"Number of Gaussian components: {n_components}")
    
    # 提取每个成分的参数
    results = {}
    
    for comp in range(1, n_components + 1):
        # 像素单位
        amp_pixel = gaussian_pixel[3*(comp-1)]
        pos_pixel = gaussian_pixel[3*(comp-1) + 1]
        dis_pixel = gaussian_pixel[3*(comp-1) + 2]
        
        # 物理单位
        if isinstance(gaussian_physical, np.ndarray):
            amp_phys = gaussian_physical[3*(comp-1)]
            pos_phys = gaussian_physical[3*(comp-1) + 1]
            dis_phys = gaussian_physical[3*(comp-1) + 2]
        else:
            amp_phys = amp_pixel
            pos_phys = pos_pixel
            dis_phys = dis_pixel
        
        results[f'comp{comp}'] = {
            'amplitude_pixel': amp_pixel,
            'position_pixel': pos_pixel,
            'dispersion_pixel': dis_pixel,
            'amplitude_phys': amp_phys,
            'position_phys': pos_phys,
            'dispersion_phys': dis_phys,
            'component_id': comp
        }
        
        print(f"\nComponent {comp}:")
        print(f"  Non-zero pixels (phys): {np.sum(amp_phys > 0)}")
        if np.any(amp_phys > 0):
            print(f"  Position range (phys): [{np.min(pos_phys[pos_phys!=0]):.2f}, {np.max(pos_phys):.2f}] km/s")
    
    results['n_components'] = n_components
    results['shape'] = (n_x, n_y)
    results['original_data_pixel'] = gaussian_pixel
    results['original_data_physical'] = gaussian_physical
    
    return results


# ==================== 2. 提取每个成分的源 ====================
def extract_sources_from_component(results, comp, min_pixels=10, max_gap=2):
    """
    从单个成分提取源
    """
    amplitude = results[f'comp{comp}']['amplitude_phys']
    position = results[f'comp{comp}']['position_phys']
    dispersion = results[f'comp{comp}']['dispersion_phys']
    
    # 创建二值mask
    binary_mask = amplitude > 0
    
    if not np.any(binary_mask):
        return []
    
    # 连通区域分析
    structure = ndimage.generate_binary_structure(2, 1)
    if max_gap > 1:
        structure = ndimage.iterate_structure(structure, max_gap - 1)
    
    labeled_mask, num_features = ndimage.label(binary_mask, structure=structure)
    
    sources = []
    
    for label in range(1, num_features + 1):
        mask = labeled_mask == label
        pixel_count = np.sum(mask)
        
        if pixel_count >= min_pixels:
            y_indices, x_indices = np.where(mask)
            
            amplitudes = []
            positions = []
            dispersions = []
            pixels = []
            
            for y, x in zip(y_indices, x_indices):
                amp = amplitude[y, x]
                pos = position[y, x]
                dis = dispersion[y, x]
                
                amplitudes.append(amp)
                positions.append(pos)
                dispersions.append(dis)
                pixels.append((x, y))
            
            amplitudes = np.array(amplitudes)
            positions = np.array(positions)
            dispersions = np.array(dispersions)
            
            # 计算加权平均（以振幅为权重）
            total_amp = np.sum(amplitudes)
            if total_amp > 0:
                weighted_position = np.sum(positions * amplitudes) / total_amp
                weighted_dispersion = np.sum(dispersions * amplitudes) / total_amp
                weighted_fwhm = 2.355 * weighted_dispersion
                weighted_amplitude = total_amp / pixel_count
            else:
                weighted_position = np.mean(positions)
                weighted_dispersion = np.mean(dispersions)
                weighted_fwhm = 2.355 * weighted_dispersion
                weighted_amplitude = 0
            
            # 计算质心
            centroid_x = np.mean(x_indices)
            centroid_y = np.mean(y_indices)
            
            source_info = {
                'source_id': label,
                'component': comp,
                'pixel_count': pixel_count,
                'pixels': pixels,
                'centroid': (centroid_x, centroid_y),
                'weighted_average': {
                    'amplitude': weighted_amplitude,
                    'velocity': weighted_position,
                    'dispersion': weighted_dispersion,
                    'fwhm': weighted_fwhm
                },
                'bbox': (np.min(x_indices), np.max(x_indices), 
                        np.min(y_indices), np.max(y_indices))
            }
            sources.append(source_info)
    
    return sources

# ==================== 3. 判断合并条件 ====================
def check_merge_condition(source1, source2, velocity_threshold_factor=0.7):
    """
    判断两个源是否可以合并
    条件：速度差 < velocity_threshold_factor * 二者之中较宽的FWHM
    """
    v1 = source1['weighted_average']['velocity']
    v2 = source2['weighted_average']['velocity']
    fwhm1 = source1['weighted_average']['fwhm']
    fwhm2 = source2['weighted_average']['fwhm']
    
    dv = abs(v1 - v2)
    wider_fwhm = max(fwhm1, fwhm2)
    threshold = velocity_threshold_factor * wider_fwhm
    
    can_merge = dv < threshold
    
    return can_merge, dv, threshold

# ==================== 4. 合并多个源 ====================
def merge_sources(sources_to_merge, results):
    """
    合并多个源
    """
    if len(sources_to_merge) == 0:
        return None
    
    # 收集所有像素
    all_pixels = []
    all_components = []
    
    for src in sources_to_merge:
        all_pixels.extend(src['pixels'])
        all_components.append(src['component'])
    
    all_pixels = list(set(all_pixels))
    
    # 计算合并后的属性（振幅加权）
    total_amp = sum([src['weighted_average']['amplitude'] * src['pixel_count'] 
                     for src in sources_to_merge])
    
    if total_amp > 0:
        weighted_velocities = []
        weighted_dispersions = []
        
        for src in sources_to_merge:
            weight = src['weighted_average']['amplitude'] * src['pixel_count'] / total_amp
            weighted_velocities.append(src['weighted_average']['velocity'] * weight)
            weighted_dispersions.append(src['weighted_average']['dispersion'] * weight)
        
        merged_velocity = sum(weighted_velocities)
        merged_dispersion = sum(weighted_dispersions)
    else:
        merged_velocity = np.mean([src['weighted_average']['velocity'] for src in sources_to_merge])
        merged_dispersion = np.mean([src['weighted_average']['dispersion'] for src in sources_to_merge])
    
    # 计算质心
    x_coords = [p[0] for p in all_pixels]
    y_coords = [p[1] for p in all_pixels]
    
    merged_source = {
        'merged_id': None,
        'pixel_count': len(all_pixels),
        'pixels': all_pixels,
        'centroid': (np.mean(x_coords), np.mean(y_coords)),
        'component_sources': sources_to_merge,
        'components': list(set(all_components)),
        'weighted_average': {
            'velocity': merged_velocity,
            'dispersion': merged_dispersion,
            'fwhm': 2.355 * merged_dispersion,
            'amplitude': total_amp / len(all_pixels) if total_amp > 0 else 0
        },
        'is_merged': True
    }
    
    return merged_source

# ==================== 5. 创建最终参数图（包含Moment 0） ====================
def create_final_parameter_maps(all_sources, results, shape):
    """
    创建最终的参数图
    """
    n_x, n_y = shape
    n_components = results['n_components']
    
    merged_amplitude = np.zeros((n_y, n_x))
    merged_position = np.zeros((n_y, n_x))
    merged_dispersion = np.zeros((n_y, n_x))
    merged_fwhm = np.zeros((n_y, n_x))
    merged_moment0 = np.zeros((n_y, n_x))  # 添加Moment 0数组
    merged_source_id = np.zeros((n_y, n_x), dtype=int)
    
    for source in all_sources:
        source_id = source['merged_id']
        
        for x, y in source['pixels']:
            if 0 <= y < n_y and 0 <= x < n_x:
                pixel_amplitudes = []
                pixel_positions = []
                pixel_dispersions = []
                
                for comp_src in source['component_sources']:
                    comp = comp_src['component']
                    if (x, y) in comp_src['pixels']:
                        amp = results[f'comp{comp}']['amplitude_phys'][y, x]
                        pos = results[f'comp{comp}']['position_phys'][y, x]
                        dis = results[f'comp{comp}']['dispersion_phys'][y, x]
                        
                        if amp > 0:
                            pixel_amplitudes.append(amp)
                            pixel_positions.append(pos)
                            pixel_dispersions.append(dis)
                
                if pixel_amplitudes:
                    weights = np.array(pixel_amplitudes) / np.sum(pixel_amplitudes)
                    merged_amplitude[y, x] = np.sum(pixel_amplitudes)
                    merged_position[y, x] = np.sum(np.array(pixel_positions) * weights)
                    merged_dispersion[y, x] = np.sum(np.array(pixel_dispersions) * weights)
                    merged_fwhm[y, x] = 2.355 * merged_dispersion[y, x]
                    
                    # 计算Moment 0 (积分强度)
                    # 对于高斯函数：∫ A * exp(-(v-v0)^2/(2σ^2)) dv = A * σ * sqrt(2π)
                    # 注意：速度单位是km/s，所以Moment 0的单位是 K * km/s
                    merged_moment0[y, x] = merged_amplitude[y, x] * merged_dispersion[y, x] * np.sqrt(2 * np.pi)
                    
                    merged_source_id[y, x] = source_id
    
    return merged_amplitude, merged_position, merged_dispersion, merged_fwhm, merged_moment0, merged_source_id


# ==================== 7. 绘制所有源（使用WCS坐标） ====================
def plot_all_sources(all_sources, merged_amplitude, merged_position, 
                     merged_fwhm, merged_moment0, merged_source_id, output_dir, wcs_2d):
    """
    绘制所有源，使用WCS坐标系统
    """
    print("\n" + "="*60)
    print("PLOTTING ALL SOURCES WITH WCS")
    print("="*60)
    
    n_sources = len(all_sources)
    
    # 创建2x2的子图，去掉大标题，子图间距更小
    fig = plt.figure(figsize=(16, 14))
    
    # 调整子图间距，让子图更靠近
    plt.subplots_adjust(left=0.08, right=0.92, bottom=0.08, top=0.95, 
                        wspace=0.15, hspace=0.1)
    
    # 为每个子图创建WCS投影
    ax1 = plt.subplot(2, 2, 1, projection=wcs_2d)
    ax2 = plt.subplot(2, 2, 2, projection=wcs_2d)
    ax3 = plt.subplot(2, 2, 3, projection=wcs_2d)
    ax4 = plt.subplot(2, 2, 4, projection=wcs_2d)
    
    axes_list = [ax1, ax2, ax3, ax4]
    
    # 图1：所有源的mask
    if n_sources > 0:
        colors = plt.cm.tab20(np.linspace(0, 1, n_sources))
        colors = np.vstack([[0,0,0,1], colors])

        # 强制指定源6、源7颜色（红色 + 深蓝色，不与viridis撞色）
        if n_sources >= 6:
            colors[6] = [1.0, 0.2, 0.2, 1.0]   # Source 6 → 红色
        if n_sources >= 7:
            colors[7] = [0.0, 0.2, 0.6, 1.0]  # Source 7 → 深蓝色 darkblue

        cmap = ListedColormap(colors)
        
        im1 = ax1.imshow(merged_source_id, origin='lower', cmap=cmap,
                        vmin=0, vmax=n_sources, transform=ax1.get_transform('pixel'))
        
        # 标注源的中心
        for source in all_sources:
            x_center, y_center = source['centroid']
            # 转换到世界坐标
            world_coords = wcs_2d.pixel_to_world(x_center, y_center)
            ra_center = world_coords.ra.deg
            dec_center = world_coords.dec.deg
            
            # 计算偏移方向
            angle = source['merged_id'] * 0
            offset_ra = 0.15 * np.cos(np.radians(angle))
            offset_dec = 0.1 * np.sin(np.radians(angle))
            
            # 标注中心点
            ax1.plot(ra_center, dec_center, 'w+', markersize=8, markeredgewidth=2, 
                    transform=ax1.get_transform('world'))
            
            # 标注数字，位置偏移
            ax1.text(ra_center + offset_ra, dec_center + offset_dec, f'{source["merged_id"]}', 
                    color='white', fontsize=12, weight='bold',
                    transform=ax1.get_transform('world'),
                    bbox=dict(boxstyle='round', facecolor='black', alpha=0.3, pad=0.3))
        
        # 添加colorbar
        cbar1 = plt.colorbar(im1, ax=ax1, pad=0.15, shrink=1, aspect=30, location='bottom')
        cbar1.ax.tick_params(labelsize=16)
    else:
        im1 = ax1.imshow(merged_source_id, origin='lower', cmap='gray')
    
    # 图2：Moment 0图（积分强度）
    # 计算合适的colorbar范围（去掉异常值）
    moment0_data = merged_moment0[merged_moment0 > 0]
    if len(moment0_data) > 0:
        vmin = np.percentile(moment0_data, 2)  # 2%分位数
        vmax = np.percentile(moment0_data, 98)  # 98%分位数
    else:
        vmin, vmax = 0, 100
    
    im2 = ax2.imshow(merged_moment0, origin='lower', cmap='viridis',
                    vmin=0, vmax=40, transform=ax2.get_transform('pixel'))
    cbar2 = plt.colorbar(im2, ax=ax2, pad=0.15, shrink=1, aspect=30, location='bottom')
    cbar2.set_label('Column density [K km s$^{-1}$]', fontsize=18)
    cbar2.ax.tick_params(labelsize=16)
    
    # 图3：位置图（速度）
    im3 = ax3.imshow(merged_position, origin='lower', cmap='coolwarm',
                    vmin=-300, vmax=-200, transform=ax3.get_transform('pixel'))
    cbar3 = plt.colorbar(im3, ax=ax3, pad=0.15, shrink=1, aspect=30, location='bottom')
    cbar3.set_label('Velocity [km s$^{-1}$]', fontsize=18)
    cbar3.ax.tick_params(labelsize=16)
    
    # 图4：FWHM图
    im4 = ax4.imshow(merged_fwhm, origin='lower', cmap='cubehelix',
                    vmin=0, vmax=50, transform=ax4.get_transform('pixel'))
    cbar4 = plt.colorbar(im4, ax=ax4, pad=0.15, shrink=1, aspect=30, location='bottom')
    cbar4.set_label('FWHM [km s$^{-1}$]', fontsize=18)
    cbar4.ax.tick_params(labelsize=16)
    
    # 配置所有子图的坐标轴
    for i, ax in enumerate(axes_list):
        # 设置标签
        ax.set_xlabel("Right Ascension [hours]", fontsize=18)
        
        # 根据子图位置决定是否显示Dec标签
        if i == 0 or i == 2:  # 第一列（子图1和3）显示Dec标签
            ax.set_ylabel("Declination [°]", fontsize=18)
            # 确保Dec坐标轴可见
            ax.coords[1].set_ticklabel_visible(True)
            ax.coords[1].set_ticks_visible(True)
            ax.coords[1].set_axislabel('Declination [°]')
        else:  # 第二列（子图2和4）不显示Dec标签
            ax.set_ylabel("")
            # 隐藏Dec坐标轴的标签和刻度
            ax.coords[1].set_ticklabel_visible(False)
            ax.coords[1].set_ticks_visible(False)
            ax.coords[1].set_axislabel('')
        
        ax.tick_params(axis='both', which='both', direction='out', labelsize=14)
        
        # 配置WCS坐标轴
        # RA坐标轴设置
        ax.coords[0].set_ticks_position('b')
        ax.coords[0].set_ticklabel_position('b')
        ax.coords[0].set_axislabel('Right Ascension [hours]')
        ax.coords[0].set_format_unit(u.hourangle)
        
        # Dec坐标轴设置
        if i == 0 or i == 2:
            ax.coords[1].set_ticks_position('l')
            ax.coords[1].set_ticklabel_position('l')
        else:
            ax.coords[1].set_ticks_position('')
            ax.coords[1].set_ticklabel_position('')
        
        ax.coords[1].set_format_unit(u.deg)
        
        # 创建上侧的RA角度副坐标轴
        try:
            overlay = ax.get_coords_overlay('fk5')
            overlay[0].set_axislabel('Right Ascension [°]', fontsize=18)
            overlay[0].tick_params(labelsize=14, direction='in')
            overlay[0].set_ticks_position('t')
            overlay[0].set_ticklabel_position('t')
            overlay[0].set_format_unit(u.deg)
            
            # 隐藏不需要的坐标轴
            overlay[1].set_axislabel('')
            overlay[1].set_ticklabel_visible(False)
            overlay[1].set_ticks_visible(False)
            
            # 添加网格
            overlay.grid(color='grey', ls='dotted', lw=0.5)
        except:
            pass
    
    all_file = os.path.join(output_dir, 'all_sources_overview.pdf')
    plt.savefig(all_file, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"All sources overview saved to: {all_file}")
    
    # 图2：每个源的组成图
    if n_sources > 0:
        n_cols = min(3, n_sources)
        n_rows = (n_sources + n_cols - 1) // n_cols
        fig = plt.figure(figsize=(5*n_cols, 4*n_rows))
        
        plt.subplots_adjust(left=0.08, right=0.92, bottom=0.08, top=0.95, 
                           wspace=0.2, hspace=0.2)
        
        for i, source in enumerate(all_sources):
            ax = fig.add_subplot(n_rows, n_cols, i+1, projection=wcs_2d)
            source_mask = merged_source_id == source['merged_id']
            
            ax.imshow(source_mask, origin='lower', cmap='Blues', alpha=0.3,
                     transform=ax.get_transform('pixel'))
            
            # 标注组成它的原始源
            colors = ['red', 'green', 'blue', 'purple', 'orange', 'brown']
            for j, comp_src in enumerate(source['component_sources']):
                comp_mask = np.zeros_like(source_mask)
                for x, y in comp_src['pixels']:
                    if 0 <= y < comp_mask.shape[0] and 0 <= x < comp_mask.shape[1]:
                        comp_mask[y, x] = True
                
                edges = ndimage.binary_dilation(comp_mask) & ~comp_mask
                y_edges, x_edges = np.where(edges)
                
                world_coords = wcs_2d.pixel_to_world(x_edges, y_edges)
                ra_edges = world_coords.ra.deg
                dec_edges = world_coords.dec.deg
                
                ax.scatter(ra_edges, dec_edges, c=colors[j % len(colors)], 
                          s=2, label=f'Comp{comp_src["component"]}-Src{comp_src["source_id"]}', 
                          alpha=0.8, transform=ax.get_transform('world'))
            
            # 显示源的类型
            if source.get('is_merged', False):
                source_type = f'Merged (Comp{source["components"]})'
            else:
                source_type = f'Comp{source["components"][0]}'
            
            # 计算标注偏移
            offset_x = 0.02
            offset_y = 0.02
            if source['merged_id'] % 2 == 0:
                offset_x = -0.02
            if source['merged_id'] % 3 == 0:
                offset_y = -0.02
            
            # 获取源的中心坐标
            x_center, y_center = source['centroid']
            world_coords = wcs_2d.pixel_to_world(x_center, y_center)
            ra_center = world_coords.ra.deg
            dec_center = world_coords.dec.deg
            
            # 在源中心附近添加信息文本
            info_text = f'{source["pixel_count"]}p\n{source["weighted_average"]["velocity"]:.0f}kms\nFWHM:{source["weighted_average"]["fwhm"]:.0f}'
            ax.text(ra_center + offset_x, dec_center + offset_y, info_text,
                   fontsize=6, transform=ax.get_transform('world'),
                   bbox=dict(boxstyle='round', facecolor='white', alpha=0.7, pad=0.2))
            
            ax.set_title(f'Source {source["merged_id"]}: {source_type}', fontsize=9, pad=3)
            
            ax.set_xlabel("RA [hours]", fontsize=8)
            ax.set_ylabel("Dec [°]", fontsize=8)
            ax.tick_params(axis='both', which='both', direction='out', labelsize=7)
            
            ax.coords[0].set_format_unit(u.hourangle)
            ax.coords[1].set_format_unit(u.deg)
            
            if len(source['component_sources']) <= 3:
                ax.legend(loc='upper right', fontsize=6, framealpha=0.8)
        
        plt.tight_layout()
        
        comp_file = os.path.join(output_dir, 'all_sources_composition.pdf')
        plt.savefig(comp_file, dpi=150, bbox_inches='tight')
        plt.close()
        print(f"All sources composition saved to: {comp_file}")

# ==================== 7. 保存每个源为独立的DAT文件 ====================
def save_individual_source_dat(results, source, output_dir, original_header_lines):
    """
    保存单个源为独立的DAT文件
    """
    n_x, n_y = results['shape']
    n_components = results['n_components']
    
    # 创建源专属的输出目录
    source_dir = os.path.join(output_dir, f'source_{source["merged_id"]:03d}')
    os.makedirs(source_dir, exist_ok=True)
    
    # 创建该源的数据数组（只包含该源的像素）
    data_lines = []
    
    for y in range(n_y):
        for x in range(n_x):
            # 检查该像素是否属于当前源
            pixel_belongs_to_source = False
            for comp_src in source['component_sources']:
                if (x, y) in comp_src['pixels']:
                    pixel_belongs_to_source = True
                    break
            
            if pixel_belongs_to_source:
                # 该像素属于当前源，写入所有分量的数据
                for comp in range(1, n_components + 1):
                    amp = 0
                    pos = 0
                    dis = 0
                    
                    # 检查该像素是否属于当前分量
                    for comp_src in source['component_sources']:
                        if comp_src['component'] == comp and (x, y) in comp_src['pixels']:
                            amp = results[f'comp{comp}']['amplitude_pixel'][y, x]
                            pos = results[f'comp{comp}']['position_pixel'][y, x]
                            dis = results[f'comp{comp}']['dispersion_pixel'][y, x]
                            break
                    
                    line = f"    {y:4d}    {x:4d}    {amp:20.16f}    {pos:20.16f}    {dis:20.16f}\n"
                    data_lines.append(line)
            else:
                # 该像素不属于当前源，所有分量写0
                for comp in range(1, n_components + 1):
                    line = f"    {y:4d}    {x:4d}    {0:20.16f}    {0:20.16f}    {0:20.16f}\n"
                    data_lines.append(line)
    
    # 保存DAT文件
    dat_filename = f'source_{source["merged_id"]:03d}.dat'
    dat_filepath = os.path.join(source_dir, dat_filename)
    
    with open(dat_filepath, 'w') as f:
        f.writelines(original_header_lines)
        f.writelines(data_lines)
    
    print(f"  Source {source['merged_id']} saved to: {dat_filepath}")
    
    # 同时保存源的参数信息
    info_filepath = os.path.join(source_dir, f'source_{source["merged_id"]:03d}_info.txt')
    with open(info_filepath, 'w') as f:
        f.write("="*60 + "\n")
        f.write(f"SOURCE {source['merged_id']} INFORMATION\n")
        f.write("="*60 + "\n\n")
        
        if source.get('is_merged', False):
            f.write(f"Type: MERGED\n")
            f.write(f"Components involved: {source['components']}\n")
        else:
            f.write(f"Type: SINGLE COMPONENT\n")
            f.write(f"Component: {source['components'][0]}\n")
        
        f.write(f"Number of pixels: {source['pixel_count']}\n")
        f.write(f"Centroid (X, Y): ({source['centroid'][0]:.2f}, {source['centroid'][1]:.2f})\n")
        f.write(f"Mean Velocity: {source['weighted_average']['velocity']:.6f} km/s\n")
        f.write(f"Mean FWHM: {source['weighted_average']['fwhm']:.6f} km/s\n")
        f.write(f"Mean Amplitude: {source['weighted_average']['amplitude']:.6f} K\n\n")
        
        f.write("Original Sources:\n")
        for comp_src in source['component_sources']:
            f.write(f"  Component {comp_src['component']}, Source {comp_src['source_id']}:\n")
            f.write(f"    Velocity: {comp_src['weighted_average']['velocity']:.6f} km/s\n")
            f.write(f"    FWHM: {comp_src['weighted_average']['fwhm']:.6f} km/s\n")
            f.write(f"    Pixels: {comp_src['pixel_count']}\n")
        
        f.write("\n" + "="*60 + "\n")
        f.write("PARAMETER STATISTICS\n")
        f.write("="*60 + "\n\n")
        
        # 计算该源的参数统计
        amplitudes = []
        positions = []
        dispersions = []
        
        for comp_src in source['component_sources']:
            comp = comp_src['component']
            for x, y in comp_src['pixels']:
                amp = results[f'comp{comp}']['amplitude_phys'][y, x]
                pos = results[f'comp{comp}']['position_phys'][y, x]
                dis = results[f'comp{comp}']['dispersion_phys'][y, x]
                if amp > 0:
                    amplitudes.append(amp)
                    positions.append(pos)
                    dispersions.append(dis)
        
        if amplitudes:
            f.write(f"Amplitude (K):\n")
            f.write(f"  Mean: {np.mean(amplitudes):.6f}\n")
            f.write(f"  Std:  {np.std(amplitudes):.6f}\n")
            f.write(f"  Min:  {np.min(amplitudes):.6f}\n")
            f.write(f"  Max:  {np.max(amplitudes):.6f}\n\n")
            
            f.write(f"Velocity (km/s):\n")
            f.write(f"  Mean: {np.mean(positions):.6f}\n")
            f.write(f"  Std:  {np.std(positions):.6f}\n")
            f.write(f"  Min:  {np.min(positions):.6f}\n")
            f.write(f"  Max:  {np.max(positions):.6f}\n\n")
            
            f.write(f"Dispersion (km/s):\n")
            f.write(f"  Mean: {np.mean(dispersions):.6f}\n")
            f.write(f"  Std:  {np.std(dispersions):.6f}\n")
            f.write(f"  Min:  {np.min(dispersions):.6f}\n")
            f.write(f"  Max:  {np.max(dispersions):.6f}\n\n")
    
    return source_dir

def save_all_individual_sources(results, all_sources, original_dat_file, output_dir):
    """
    保存所有源为独立的DAT文件
    """
    print("\n" + "="*60)
    print("SAVING INDIVIDUAL SOURCES TO .DAT FILES")
    print("="*60)
    
    # 读取原始DAT文件的头部
    try:
        with open(original_dat_file, 'r') as f:
            header_lines = []
            for i in range(27):
                line = f.readline()
                header_lines.append(line)
    except:
        # 如果无法读取原始文件，创建默认头部
        header_lines = [f"Header line {i+1}\n" for i in range(27)]
        print("Warning: Could not read original DAT header, using default header.")
    
    # 为每个源创建单独的目录和文件
    source_dirs = []
    for source in all_sources:
        source_dir = save_individual_source_dat(results, source, output_dir, header_lines)
        source_dirs.append(source_dir)
    
    # 创建总索引文件
    index_file = os.path.join(output_dir, 'sources_index.txt')
    with open(index_file, 'w') as f:
        f.write("="*60 + "\n")
        f.write("INDIVIDUAL SOURCES INDEX\n")
        f.write("="*60 + "\n\n")
        
        f.write(f"Total sources: {len(all_sources)}\n\n")
        
        for source in all_sources:
            f.write(f"\n{'='*50}\n")
            f.write(f"Source {source['merged_id']:03d}\n")
            f.write(f"{'='*50}\n")
            f.write(f"Directory: source_{source['merged_id']:03d}/\n")
            f.write(f"Data file: source_{source['merged_id']:03d}.dat\n")
            f.write(f"Info file: source_{source['merged_id']:03d}_info.txt\n\n")
            
            if source.get('is_merged', False):
                f.write(f"Type: MERGED (from Components {source['components']})\n")
            else:
                f.write(f"Type: SINGLE COMPONENT (Component {source['components'][0]})\n")
            
            f.write(f"Number of pixels: {source['pixel_count']}\n")
            f.write(f"Centroid: ({source['centroid'][0]:.2f}, {source['centroid'][1]:.2f})\n")
            f.write(f"Mean velocity: {source['weighted_average']['velocity']:.2f} km/s\n")
            f.write(f"Mean FWHM: {source['weighted_average']['fwhm']:.2f} km/s\n")
            f.write(f"Mean amplitude: {source['weighted_average']['amplitude']:.2f} K\n")
    
    print(f"\nAll individual source files saved to: {output_dir}")
    print(f"Index file saved to: {index_file}")
    print(f"Total {len(all_sources)} sources saved")

# ==================== 8. 保存DAT文件 ====================
def save_final_dat(results, all_sources, original_dat_file, output_file):
    """
    保存最终的DAT文件（像素单位）
    """
    print("\n" + "="*60)
    print("SAVING FINAL RESULTS TO .DAT FORMAT (PIXEL UNITS)")
    print("="*60)
    
    try:
        n_x, n_y = results['shape']
        n_components = results['n_components']
        
        with open(original_dat_file, 'r') as f:
            header_lines = []
            for i in range(27):
                line = f.readline()
                header_lines.append(line)
        
        source_id_map = {}
        for source in all_sources:
            for x, y in source['pixels']:
                source_id_map[(x, y)] = source['merged_id']
        
        data_lines = []
        
        for y in range(n_y):
            for x in range(n_x):
                for comp in range(1, n_components + 1):
                    if (x, y) in source_id_map:
                        for source in all_sources:
                            if source['merged_id'] == source_id_map[(x, y)]:
                                pixel_amp = 0
                                pixel_pos = 0
                                pixel_dis = 0
                                for comp_src in source['component_sources']:
                                    if comp_src['component'] == comp and (x, y) in comp_src['pixels']:
                                        amp = results[f'comp{comp}']['amplitude_pixel'][y, x]
                                        pos = results[f'comp{comp}']['position_pixel'][y, x]
                                        dis = results[f'comp{comp}']['dispersion_pixel'][y, x]
                                        if amp > 0:
                                            pixel_amp = amp
                                            pixel_pos = pos
                                            pixel_dis = dis
                                            break
                                line = f"    {y:4d}    {x:4d}    {pixel_amp:20.16f}    {pixel_pos:20.16f}    {pixel_dis:20.16f}\n"
                                data_lines.append(line)
                                break
                    else:
                        line = f"    {y:4d}    {x:4d}    {0:20.16f}    {0:20.16f}    {0:20.16f}\n"
                        data_lines.append(line)
        
        with open(output_file, 'w') as f:
            f.writelines(header_lines)
            f.writelines(data_lines)
        
        print(f"\nFinal .dat file saved to: {output_file}")
        
    except Exception as e:
        print(f"Error saving final .dat: {e}")

# ==================== 9. 保存源信息 ====================
def save_source_info(all_sources, output_dir):
    """
    保存源信息到文本文件
    """
    info_file = os.path.join(output_dir, 'final_source_information.txt')
    
    with open(info_file, 'w') as f:
        f.write("="*60 + "\n")
        f.write("FINAL SOURCE INFORMATION\n")
        f.write("="*60 + "\n\n")
        
        f.write(f"Total sources: {len(all_sources)}\n\n")
        
        for source in all_sources:
            f.write(f"\n{'='*50}\n")
            f.write(f"SOURCE {source['merged_id']}\n")
            f.write(f"{'='*50}\n\n")
            
            if source.get('is_merged', False):
                f.write(f"Type: MERGED\n")
                f.write(f"Components involved: {source['components']}\n")
            else:
                f.write(f"Type: SINGLE COMPONENT\n")
                f.write(f"Component: {source['components'][0]}\n")
            
            f.write(f"Number of pixels: {source['pixel_count']}\n")
            f.write(f"Centroid (X, Y): ({source['centroid'][0]:.2f}, {source['centroid'][1]:.2f})\n")
            f.write(f"Mean Velocity: {source['weighted_average']['velocity']:.6f} km/s\n")
            f.write(f"Mean FWHM: {source['weighted_average']['fwhm']:.6f} km/s\n")
            f.write(f"Mean Amplitude: {source['weighted_average']['amplitude']:.6f} K\n\n")
            
            f.write("Original Sources:\n")
            for comp_src in source['component_sources']:
                f.write(f"  Component {comp_src['component']}, Source {comp_src['source_id']}:\n")
                f.write(f"    Velocity: {comp_src['weighted_average']['velocity']:.6f} km/s\n")
                f.write(f"    FWHM: {comp_src['weighted_average']['fwhm']:.6f} km/s\n")
                f.write(f"    Pixels: {comp_src['pixel_count']}\n")
    
    print(f"Source information saved to: {info_file}")

# ==================== 10. 主函数 ====================
def main(dat_file, original_dat_file, output_dir, fits_file=None,
         min_pixels=10, max_gap=2, velocity_threshold_factor=0.7):
    """
    主函数 - 分别比较Component 2的第一个源和Component 3的第1、2个源
    """
    print("\n" + "="*60)
    print("SOURCE MERGING (Component 2 vs Component 3)")
    print("="*60)
    print(f"Input DAT file: {dat_file}")
    print(f"Output directory: {output_dir}")
    print(f"Min pixels per source: {min_pixels}")
    print(f"Max spatial gap: {max_gap}")
    print(f"Velocity threshold factor: {velocity_threshold_factor}")
    
    os.makedirs(output_dir, exist_ok=True)
    
    # 获取WCS坐标信息
    wcs_2d = None
    if fits_file is not None and os.path.exists(fits_file):
        print(f"\nReading WCS info from FITS: {fits_file}")
        wcs_2d, n_x, n_y = get_coordinate_info(fits_file)
        if wcs_2d is not None:
            print("Successfully loaded WCS coordinates")
        else:
            print("Could not load WCS coordinates")
    else:
        print("No FITS file provided")
    
    # 1. 读取DAT文件
    results = read_filtered_dat(dat_file)
    shape = results['shape']
    n_components = results['n_components']
    
    print(f"\n检测到 {n_components} 个高斯分量")
    
    # 2. 提取每个成分的源
    print("\n" + "="*60)
    print("EXTRACTING SOURCES FROM EACH COMPONENT")
    print("="*60)
    
    comp1_sources = extract_sources_from_component(results, 1, min_pixels, max_gap)
    comp2_sources = extract_sources_from_component(results, 2, min_pixels, max_gap)
    comp3_sources = extract_sources_from_component(results, 3, min_pixels, max_gap)
    
    print(f"\nComponent 1: {len(comp1_sources)} sources")
    for src in comp1_sources:
        print(f"  Source {src['source_id']}: {src['pixel_count']} pixels, v={src['weighted_average']['velocity']:.2f} km/s, FWHM={src['weighted_average']['fwhm']:.2f}")
    
    print(f"\nComponent 2: {len(comp2_sources)} sources")
    for src in comp2_sources:
        print(f"  Source {src['source_id']}: {src['pixel_count']} pixels, v={src['weighted_average']['velocity']:.2f} km/s, FWHM={src['weighted_average']['fwhm']:.2f}")
    
    print(f"\nComponent 3: {len(comp3_sources)} sources")
    for src in comp3_sources:
        print(f"  Source {src['source_id']}: {src['pixel_count']} pixels, v={src['weighted_average']['velocity']:.2f} km/s, FWHM={src['weighted_average']['fwhm']:.2f}")
    
    # 3. 分别比较Component 2的第一个源和Component 3的第1、2个源
    all_final_sources = []
    check_results = []
    used_sources = set()
    sources_to_merge = []
    
    if comp2_sources:
        comp2_source = comp2_sources[0]  # Component 2的第一个源
        print(f"\n{'='*60}")
        print(f"检查 Component 2 源 1 (v={comp2_source['weighted_average']['velocity']:.2f} km/s, FWHM={comp2_source['weighted_average']['fwhm']:.2f})")
        print(f"{'='*60}")
        
        # 先添加Component 2源到待合并列表
        sources_to_merge.append(comp2_source)
        
        # 检查Component 3的第1、2个源
        comp3_sources_to_check = []
        if len(comp3_sources) >= 1:
            comp3_sources_to_check.append(comp3_sources[0])
        if len(comp3_sources) >= 2:
            comp3_sources_to_check.append(comp3_sources[1])
        
        for comp3_src in comp3_sources_to_check:
            can_merge, dv, threshold = check_merge_condition(
                comp2_source, comp3_src, velocity_threshold_factor
            )
            
            check_results.append({
                'type': f'Comp2-Src1 vs Comp3-Src{comp3_src["source_id"]}',
                'v1': comp2_source['weighted_average']['velocity'],
                'v2': comp3_src['weighted_average']['velocity'],
                'fwhm1': comp2_source['weighted_average']['fwhm'],
                'fwhm2': comp3_src['weighted_average']['fwhm'],
                'dv': dv,
                'threshold': threshold,
                'can_merge': can_merge
            })
            
            print(f"\n  Comp2-Src1 vs Comp3-Src{comp3_src['source_id']}:")
            print(f"    v1={comp2_source['weighted_average']['velocity']:.2f}, v2={comp3_src['weighted_average']['velocity']:.2f}, dv={dv:.2f}")
            print(f"    fwhm1={comp2_source['weighted_average']['fwhm']:.2f}, fwhm2={comp3_src['weighted_average']['fwhm']:.2f}")
            print(f"    Wider FWHM: {max(comp2_source['weighted_average']['fwhm'], comp3_src['weighted_average']['fwhm']):.2f}")
            print(f"    Threshold (0.7 × wider FWHM): {threshold:.2f}")
            print(f"    Can merge: {can_merge}")
            
            if can_merge:
                sources_to_merge.append(comp3_src)
                used_sources.add(('comp3', comp3_src['source_id']))
                print(f"    ✓ 添加到合并列表")
            else:
                print(f"    ✗ 不满足合并条件")
        
        # 如果有可合并的源（除了Comp2源本身）
        if len(sources_to_merge) > 1:
            merged = merge_sources(sources_to_merge, results)
            if merged:
                all_final_sources.append(merged)
                used_sources.add(('comp2', comp2_source['source_id']))
                print(f"\n✓ 创建合并源，包含 {len(sources_to_merge)} 个源")
                print(f"  合并的源: Component 2-1 + Component 3 源 {[src['source_id'] for src in sources_to_merge[1:]]}")
        else:
            # 不能合并，保留为独立源
            all_final_sources.append({
                'merged_id': None,
                'pixel_count': comp2_source['pixel_count'],
                'pixels': comp2_source['pixels'],
                'centroid': comp2_source['centroid'],
                'component_sources': [comp2_source],
                'components': [2],
                'weighted_average': comp2_source['weighted_average'],
                'is_merged': False
            })
            used_sources.add(('comp2', comp2_source['source_id']))
            print(f"\n✗ 没有可合并的源，保留为独立源")
    else:
        print("\nComponent 2 没有源")
    
    # 4. 添加剩余的源（未使用的源）
    temp_sources = []
    
    # 添加Component 1的所有源
    for src in comp1_sources:
        temp_sources.append({
            'merged_id': None,
            'pixel_count': src['pixel_count'],
            'pixels': src['pixels'],
            'centroid': src['centroid'],
            'component_sources': [src],
            'components': [1],
            'weighted_average': src['weighted_average'],
            'is_merged': False
        })
    
    # 添加已经创建的合并源
    for src in all_final_sources:
        temp_sources.append(src)
    
    # 添加Component 2中未使用的源（除了第一个）
    for src in comp2_sources:
        if ('comp2', src['source_id']) not in used_sources:
            temp_sources.append({
                'merged_id': None,
                'pixel_count': src['pixel_count'],
                'pixels': src['pixels'],
                'centroid': src['centroid'],
                'component_sources': [src],
                'components': [2],
                'weighted_average': src['weighted_average'],
                'is_merged': False
            })
    
    # 添加Component 3中未使用的源（除了被合并的）
    for src in comp3_sources:
        if ('comp3', src['source_id']) not in used_sources:
            temp_sources.append({
                'merged_id': None,
                'pixel_count': src['pixel_count'],
                'pixels': src['pixels'],
                'centroid': src['centroid'],
                'component_sources': [src],
                'components': [3],
                'weighted_average': src['weighted_average'],
                'is_merged': False
            })
    
    # 重新分配ID
    for i, source in enumerate(temp_sources):
        source['merged_id'] = i + 1
    
    all_final_sources = temp_sources
    
    print(f"\n{'='*60}")
    print("最终源列表")
    print("="*60)
    print(f"总源数: {len(all_final_sources)}")
    for src in all_final_sources:
        if src.get('is_merged', False):
            print(f"  源 {src['merged_id']}: 合并源 (分量 {src['components']}), "
                  f"{src['pixel_count']} 像素, v={src['weighted_average']['velocity']:.2f} km/s, "
                  f"FWHM={src['weighted_average']['fwhm']:.2f} km/s")
        else:
            print(f"  源 {src['merged_id']}: 分量 {src['components'][0]}, "
                  f"{src['pixel_count']} 像素, v={src['weighted_average']['velocity']:.2f} km/s, "
                  f"FWHM={src['weighted_average']['fwhm']:.2f} km/s")
    
    # 5. 创建参数图（现在返回6个值）
    merged_amplitude, merged_position, merged_dispersion, merged_fwhm, merged_moment0, merged_source_id = \
        create_final_parameter_maps(all_final_sources, results, shape)
    
    # 6. 绘制所有源（使用WCS坐标）- 传入merged_moment0
    if wcs_2d is not None:
        plot_all_sources(all_final_sources, merged_amplitude, merged_position,
                        merged_fwhm, merged_moment0, merged_source_id, output_dir, wcs_2d)
    else:
        print("Warning: No WCS information available, cannot plot with celestial coordinates")
        # 如果没有WCS，可以创建简单的像素坐标图
        print("Creating simple pixel coordinate plot...")
        # 这里可以添加一个简单的像素坐标绘图函数
    
    # 7. 保存源信息
    save_source_info(all_final_sources, output_dir)
    
    # 8. 保存DAT文件
    final_dat = os.path.join(output_dir, 'final_sources.dat')
    save_final_dat(results, all_final_sources, original_dat_file, final_dat)
    
    # 9. 保存合并检查结果
    check_file = os.path.join(output_dir, 'merge_check_results.txt')
    with open(check_file, 'w') as f:
        f.write("="*60 + "\n")
        f.write("合并条件检查结果\n")
        f.write("="*60 + "\n\n")
        
        for check in check_results:
            f.write(f"\n{check['type']}:\n")
            f.write(f"  Component 2 速度: {check['v1']:.4f} km/s\n")
            f.write(f"  Component 2 FWHM: {check['fwhm1']:.4f} km/s\n")
            f.write(f"  Component 3 速度: {check['v2']:.4f} km/s\n")
            f.write(f"  Component 3 FWHM: {check['fwhm2']:.4f} km/s\n")
            f.write(f"  较宽的FWHM: {max(check['fwhm1'], check['fwhm2']):.4f} km/s\n")
            f.write(f"  速度差: {check['dv']:.4f} km/s\n")
            f.write(f"  阈值 (0.7 × 较宽FWHM): {check['threshold']:.4f} km/s\n")
            f.write(f"  是否合并: {check['can_merge']}\n")
    
    print(f"\n合并检查结果保存至: {check_file}")
    
    print("\n" + "="*60)
    print("处理完成")
    print("="*60)
    print(f"\n输出文件保存至: {output_dir}")
    print(f"  - all_sources_overview.pdf")
    print(f"  - all_sources_composition.pdf")
    print(f"  - final_source_information.txt")
    print(f"  - final_sources.dat")
    print(f"  - merge_check_results.txt")

def get_coordinate_info(fits_file):
    """
    从FITS文件读取WCS信息，获取RA和DEC坐标
    """
    try:
        # 使用SpectralCube读取
        cube = SpectralCube.read(fits_file)
        print(f"SpectralCube shape: {cube.shape}")
        
        # 获取2D空间WCS
        wcs_2d = cube.wcs.celestial
        
        # 获取空间维度大小
        n_y, n_x = cube.shape[-2:]
        
        return wcs_2d, n_x, n_y
        
    except Exception as e:
        print(f"Warning: Could not read coordinate info from FITS: {e}")
        return None, None, None




# ==================== 11. 脚本执行 ====================
if __name__ == "__main__":
    # 请修改这些路径
    DAT_FILE = "./n_gauss=3/output_individual_source/source_filtered_rohsa_pixel.dat"
    ORIGINAL_DAT_FILE = "MS_ROHSA_3ngauss_1_3D_1.dat"
    FITS_FILE = "CRAFTS_-4.7_-350_-150_Original.fits"  # 你的FITS文件路径
    OUTPUT_DIR = "./n_gauss=3/output_merged_sources/picture"
    
    # 参数设置
    MIN_PIXELS = 10
    MAX_GAP = 2
    VELOCITY_THRESHOLD_FACTOR = 0.7
    
    # 运行
    main(
        dat_file=DAT_FILE,
        original_dat_file=ORIGINAL_DAT_FILE,
        output_dir=OUTPUT_DIR,
        fits_file=FITS_FILE,  # 传入FITS文件路径
        min_pixels=MIN_PIXELS,
        max_gap=MAX_GAP,
        velocity_threshold_factor=VELOCITY_THRESHOLD_FACTOR
    )

Times New Roman not available, using default font

SOURCE MERGING (Component 2 vs Component 3)
Input DAT file: ./n_gauss=3/output_individual_source/source_filtered_rohsa_pixel.dat
Output directory: ./n_gauss=3/output_merged_sources/picture
Min pixels per source: 10
Max spatial gap: 2
Velocity threshold factor: 0.7

Reading WCS info from FITS: CRAFTS_-4.7_-350_-150_Original.fits
SpectralCube shape: (994, 89, 153)
Successfully loaded WCS coordinates

READING FILTERED ROHSA DAT FILE
Reading Gaussian parameters (pixel units)...
Opening data file
Gaussian pixel array shape: (9, 89, 153)
Data type: float64
Converting to physical units...
Spatial dimensions: X=153, Y=89
Number of Gaussian components: 3

Component 1:
  Non-zero pixels (phys): 4872
  Position range (phys): [-255.87, -149.83] km/s

Component 2:
  Non-zero pixels (phys): 291
  Position range (phys): [-282.77, -149.83] km/s

Component 3:
  Non-zero pixels (phys): 56
  Position range (phys): [-297.10, -149.83] km/s

检测到 3 个高斯分量

EX

In [9]:
!cp /home/yx/n_gauss=3/output_merged_sources/picture/all_sources_overview.pdf /mnt/d/MS/cut/photo


### 保存mask

In [37]:
# ==================== 添加生成Mask的功能 ====================
def generate_source_masks(all_sources, merged_source_id, results, output_dir, wcs_2d):
    """
    生成各种格式的源mask文件
    """
    print("\n" + "="*60)
    print("GENERATING SOURCE MASKS")
    print("="*60)
    
    n_y, n_x = merged_source_id.shape
    n_sources = len(all_sources)
    
    # 1. 生成每个源的单独mask（2D）
    print("\n生成每个源的单独mask (2D)...")
    masks_dir = os.path.join(output_dir, 'masks')
    os.makedirs(masks_dir, exist_ok=True)
    
    for source in all_sources:
        source_id = source['merged_id']
        source_mask = (merged_source_id == source_id).astype(np.float32)
        
        # 保存为numpy数组
        np.save(os.path.join(masks_dir, f'source_{source_id}_mask.npy'), source_mask)
        
        # 如果有WCS信息，保存为FITS文件
        if wcs_2d is not None:
            from astropy.io import fits
            header = wcs_2d.to_header()
            fits.writeto(os.path.join(masks_dir, f'source_{source_id}_mask.fits'), 
                        source_mask, header, overwrite=True)
        
        print(f"  Source {source_id}: {np.sum(source_mask)} pixels")
    
    # 2. 生成所有源的合并mask（每个源有不同的ID值）
    print("\n生成所有源的合并mask (不同ID值)...")
    if wcs_2d is not None:
        from astropy.io import fits
        header = wcs_2d.to_header()
        fits.writeto(os.path.join(output_dir, 'all_sources_mask.fits'), 
                    merged_source_id.astype(np.int32), header, overwrite=True)
    np.save(os.path.join(output_dir, 'all_sources_mask.npy'), merged_source_id)
    
    # 3. 生成二值mask（0和1，所有源合并）
    print("\n生成二值mask (所有源合并为1)...")
    binary_mask = (merged_source_id > 0).astype(np.float32)
    if wcs_2d is not None:
        from astropy.io import fits
        header = wcs_2d.to_header()
        fits.writeto(os.path.join(output_dir, 'binary_mask.fits'), 
                    binary_mask, header, overwrite=True)
    np.save(os.path.join(output_dir, 'binary_mask.npy'), binary_mask)
    
    # 4. 生成每个源在不同速度通道的3D mask（用于position-velocity图）
    print("\n生成3D mask (速度通道)...")
    # 获取速度范围
    velocities = []
    for source in all_sources:
        velocities.append(source['weighted_average']['velocity'])
    
    v_min = min(velocities) - 50  # 扩展速度范围
    v_max = max(velocities) + 50
    
    # 创建速度轴（假设速度分辨率是1 km/s，可以根据实际情况调整）
    # 注意：这里需要根据你的实际数据调整速度范围和分辨率
    v_resolution = 1.0  # km/s，需要根据实际情况设置
    n_vel_channels = int((v_max - v_min) / v_resolution) + 1
    velocity_axis = np.linspace(v_min, v_max, n_vel_channels)
    
    # 创建3D mask (velocity, y, x)
    mask_3d = np.zeros((n_vel_channels, n_y, n_x), dtype=np.float32)
    
    for source in all_sources:
        source_id = source['merged_id']
        source_mask_2d = (merged_source_id == source_id).astype(np.float32)
        
        # 获取源的速度范围和FWHM
        v_center = source['weighted_average']['velocity']
        fwhm = source['weighted_average']['fwhm']
        sigma = fwhm / 2.355
        
        # 在速度轴上创建高斯轮廓
        for i, v in enumerate(velocity_axis):
            # 高斯权重
            weight = np.exp(-0.5 * ((v - v_center) / sigma) ** 2)
            mask_3d[i, :, :] += source_mask_2d * weight
    
    # 保存3D mask
    if wcs_2d is not None:
        # 创建3D WCS
        from astropy.wcs import WCS
        from astropy.io import fits
        
        # 创建3D WCS对象
        wcs_3d = WCS(naxis=3)
        wcs_3d.wcs.ctype = ['RA---CAR', 'DEC--CAR', 'VELO']
        wcs_3d.wcs.cunit = ['deg', 'deg', 'km/s']
        
        # 设置坐标轴
        wcs_3d.wcs.crpix = [1, 1, 1]
        wcs_3d.wcs.crval = [wcs_2d.wcs.crval[0], wcs_2d.wcs.crval[1], v_min]
        wcs_3d.wcs.cdelt = [wcs_2d.wcs.cdelt[0], wcs_2d.wcs.cdelt[1], v_resolution]
        
        # 保存3D mask
        header_3d = wcs_3d.to_header()
        fits.writeto(os.path.join(output_dir, 'sources_3d_mask.fits'), 
                    mask_3d.astype(np.float32), header_3d, overwrite=True)
    
    np.save(os.path.join(output_dir, 'sources_3d_mask.npy'), mask_3d)
    print(f"  3D mask shape: {mask_3d.shape} (velocity, y, x)")
    print(f"  Velocity range: {v_min:.1f} - {v_max:.1f} km/s")
    
    # 5. 生成每个源的区域文件（用于CASA或DS9）
    print("\n生成区域文件 (用于DS9/CASA)...")
    region_file = os.path.join(output_dir, 'sources.reg')
    with open(region_file, 'w') as f:
        f.write('# Region file format: DS9 version 4.1\n')
        f.write('global color=green dashlist=8 3 width=1 font="helvetica 10 normal" select=1 highlite=1 dash=0 fixed=0 edit=1 move=1 delete=1 include=1 source=1\n')
        f.write('fk5\n')
        
        for source in all_sources:
            source_id = source['merged_id']
            x_center, y_center = source['centroid']
            
            if wcs_2d is not None:
                # 转换到世界坐标
                world_coords = wcs_2d.pixel_to_world(x_center, y_center)
                ra = world_coords.ra.deg
                dec = world_coords.dec.deg
                
                # 计算源的半径（基于像素数估算）
                radius = np.sqrt(source['pixel_count'] / np.pi) * 0.5  # 简单估算
                f.write(f'circle({ra:.6f},{dec:.6f},{radius:.4f}")\n')
            else:
                # 使用像素坐标
                f.write(f'circle({x_center:.1f},{y_center:.1f},{np.sqrt(source["pixel_count"]/np.pi):.1f})\n')
    
    print(f"  区域文件保存至: {region_file}")
    
    # 6. 生成mask信息文本文件
    print("\n生成mask信息文件...")
    info_file = os.path.join(output_dir, 'mask_information.txt')
    with open(info_file, 'w') as f:
        f.write("="*60 + "\n")
        f.write("SOURCE MASK INFORMATION\n")
        f.write("="*60 + "\n\n")
        
        f.write(f"Total sources: {n_sources}\n")
        f.write(f"Image size: {n_x} x {n_y} pixels\n\n")
        
        f.write("Available mask files:\n")
        f.write("  - all_sources_mask.fits/npy: Each source has a unique ID value\n")
        f.write("  - binary_mask.fits/npy: Binary mask (1 for source, 0 for background)\n")
        f.write("  - sources_3d_mask.fits/npy: 3D mask with velocity dimension\n")
        f.write("  - masks/source_X_mask.fits/npy: Individual source masks\n")
        f.write("  - sources.reg: DS9 region file\n\n")
        
        f.write("Source details:\n")
        for source in all_sources:
            source_id = source['merged_id']
            f.write(f"\nSource {source_id}:\n")
            f.write(f"  Pixels: {source['pixel_count']}\n")
            f.write(f"  Centroid (pixel): ({source['centroid'][0]:.1f}, {source['centroid'][1]:.1f})\n")
            f.write(f"  Velocity: {source['weighted_average']['velocity']:.2f} km/s\n")
            f.write(f"  FWHM: {source['weighted_average']['fwhm']:.2f} km/s\n")
            
            if wcs_2d is not None:
                world_coords = wcs_2d.pixel_to_world(source['centroid'][0], source['centroid'][1])
                f.write(f"  RA: {world_coords.ra.to_string(unit=u.hour, sep=':')}\n")
                f.write(f"  Dec: {world_coords.dec.to_string(unit=u.deg, sep=':')}\n")
    
    print(f"  Mask信息保存至: {info_file}")
    
    return masks_dir


# ==================== 添加叠加mask到原始数据的函数 ====================
def overlay_masks_on_data(fits_file, mask_file, output_file, source_ids=None):
    """
    将mask叠加到原始FITS数据上
    source_ids: 要显示的源ID列表，如果为None则显示所有源
    """
    print("\n" + "="*60)
    print("OVERLAYING MASKS ON ORIGINAL DATA")
    print("="*60)
    
    try:
        from astropy.io import fits
        import matplotlib.pyplot as plt
        from astropy.wcs import WCS
        
        # 读取原始数据
        with fits.open(fits_file) as hdul:
            data = hdul[0].data
            header = hdul[0].header
        
        # 读取mask
        if isinstance(mask_file, str):
            mask_data = fits.getdata(mask_file)
        else:
            mask_data = mask_file
        
        # 创建叠加图
        if len(data.shape) == 3:
            # 对于3D数据，取速度通道的平均或中位数
            data_2d = np.mean(data, axis=0)
        else:
            data_2d = data
        
        # 创建图形
        fig = plt.figure(figsize=(12, 10))
        
        # 创建WCS投影
        if 'CDELT1' in header:
            wcs = WCS(header)
            if len(data.shape) == 3:
                wcs = wcs.dropaxis(0)  # 去掉速度轴
            ax = plt.subplot(projection=wcs)
        else:
            ax = plt.gca()
        
        # 显示原始数据
        im = ax.imshow(data_2d, origin='lower', cmap='gray', 
                      vmin=np.percentile(data_2d[data_2d > 0], 5),
                      vmax=np.percentile(data_2d[data_2d > 0], 95))
        
        # 叠加mask轮廓
        if source_ids is None:
            # 显示所有源
            unique_ids = np.unique(mask_data)
            unique_ids = unique_ids[unique_ids > 0]
            source_ids = unique_ids
        
        colors = plt.cm.tab10(np.linspace(0, 1, len(source_ids)))
        
        for source_id, color in zip(source_ids, colors):
            source_mask = (mask_data == source_id)
            if np.any(source_mask):
                # 提取轮廓
                from scipy import ndimage
                contours = ndimage.find_objects(source_mask)
                for contour in contours:
                    if contour is not None:
                        y_slice, x_slice = contour
                        # 创建轮廓线
                        from skimage import measure
                        contours_list = measure.find_contours(source_mask[y_slice, x_slice], 0.5)
                        for contour_line in contours_list:
                            contour_line[:, 0] += x_slice.start
                            contour_line[:, 1] += y_slice.start
                            ax.plot(contour_line[:, 0], contour_line[:, 1], 
                                   color=color, linewidth=2, 
                                   label=f'Source {source_id}')
        
        # 添加图例
        ax.legend(loc='upper right', fontsize=10)
        
        # 设置标签
        ax.set_xlabel('RA', fontsize=12)
        ax.set_ylabel('Dec', fontsize=12)
        ax.set_title('Original Data with Source Masks', fontsize=14)
        
        # 添加colorbar
        plt.colorbar(im, ax=ax, label='Intensity')
        
        plt.tight_layout()
        plt.savefig(output_file, dpi=150, bbox_inches='tight')
        plt.close()
        
        print(f"Mask overlay saved to: {output_file}")
        
    except Exception as e:
        print(f"Error overlaying masks: {e}")


# ==================== 修改主函数，添加mask生成 ====================
def main(dat_file, original_dat_file, output_dir, fits_file=None,
         min_pixels=10, max_gap=2, velocity_threshold_factor=0.7):
    """
    主函数 - 分别比较Component 2的第一个源和Component 3的第1、2个源
    """
    print("\n" + "="*60)
    print("SOURCE MERGING (Component 2 vs Component 3)")
    print("="*60)
    print(f"Input DAT file: {dat_file}")
    print(f"Output directory: {output_dir}")
    print(f"Min pixels per source: {min_pixels}")
    print(f"Max spatial gap: {max_gap}")
    print(f"Velocity threshold factor: {velocity_threshold_factor}")
    
    os.makedirs(output_dir, exist_ok=True)
    
    # 获取WCS坐标信息
    wcs_2d = None
    if fits_file is not None and os.path.exists(fits_file):
        print(f"\nReading WCS info from FITS: {fits_file}")
        wcs_2d, n_x, n_y = get_coordinate_info(fits_file)
        if wcs_2d is not None:
            print("Successfully loaded WCS coordinates")
        else:
            print("Could not load WCS coordinates")
    else:
        print("No FITS file provided")
    
    # 1. 读取DAT文件
    results = read_filtered_dat(dat_file)
    shape = results['shape']
    n_components = results['n_components']
    
    print(f"\n检测到 {n_components} 个高斯分量")
    
    # 2. 提取每个成分的源
    print("\n" + "="*60)
    print("EXTRACTING SOURCES FROM EACH COMPONENT")
    print("="*60)
    
    comp1_sources = extract_sources_from_component(results, 1, min_pixels, max_gap)
    comp2_sources = extract_sources_from_component(results, 2, min_pixels, max_gap)
    comp3_sources = extract_sources_from_component(results, 3, min_pixels, max_gap)
    
    print(f"\nComponent 1: {len(comp1_sources)} sources")
    for src in comp1_sources:
        print(f"  Source {src['source_id']}: {src['pixel_count']} pixels, v={src['weighted_average']['velocity']:.2f} km/s, FWHM={src['weighted_average']['fwhm']:.2f}")
    
    print(f"\nComponent 2: {len(comp2_sources)} sources")
    for src in comp2_sources:
        print(f"  Source {src['source_id']}: {src['pixel_count']} pixels, v={src['weighted_average']['velocity']:.2f} km/s, FWHM={src['weighted_average']['fwhm']:.2f}")
    
    print(f"\nComponent 3: {len(comp3_sources)} sources")
    for src in comp3_sources:
        print(f"  Source {src['source_id']}: {src['pixel_count']} pixels, v={src['weighted_average']['velocity']:.2f} km/s, FWHM={src['weighted_average']['fwhm']:.2f}")
    
    # 3. 分别比较Component 2的第一个源和Component 3的第1、2个源
    all_final_sources = []
    check_results = []
    used_sources = set()
    sources_to_merge = []
    
    if comp2_sources:
        comp2_source = comp2_sources[0]  # Component 2的第一个源
        print(f"\n{'='*60}")
        print(f"检查 Component 2 源 1 (v={comp2_source['weighted_average']['velocity']:.2f} km/s, FWHM={comp2_source['weighted_average']['fwhm']:.2f})")
        print(f"{'='*60}")
        
        # 先添加Component 2源到待合并列表
        sources_to_merge.append(comp2_source)
        
        # 检查Component 3的第1、2个源
        comp3_sources_to_check = []
        if len(comp3_sources) >= 1:
            comp3_sources_to_check.append(comp3_sources[0])
        if len(comp3_sources) >= 2:
            comp3_sources_to_check.append(comp3_sources[1])
        
        for comp3_src in comp3_sources_to_check:
            can_merge, dv, threshold = check_merge_condition(
                comp2_source, comp3_src, velocity_threshold_factor
            )
            
            check_results.append({
                'type': f'Comp2-Src1 vs Comp3-Src{comp3_src["source_id"]}',
                'v1': comp2_source['weighted_average']['velocity'],
                'v2': comp3_src['weighted_average']['velocity'],
                'fwhm1': comp2_source['weighted_average']['fwhm'],
                'fwhm2': comp3_src['weighted_average']['fwhm'],
                'dv': dv,
                'threshold': threshold,
                'can_merge': can_merge
            })
            
            print(f"\n  Comp2-Src1 vs Comp3-Src{comp3_src['source_id']}:")
            print(f"    v1={comp2_source['weighted_average']['velocity']:.2f}, v2={comp3_src['weighted_average']['velocity']:.2f}, dv={dv:.2f}")
            print(f"    fwhm1={comp2_source['weighted_average']['fwhm']:.2f}, fwhm2={comp3_src['weighted_average']['fwhm']:.2f}")
            print(f"    Wider FWHM: {max(comp2_source['weighted_average']['fwhm'], comp3_src['weighted_average']['fwhm']):.2f}")
            print(f"    Threshold (0.7 × wider FWHM): {threshold:.2f}")
            print(f"    Can merge: {can_merge}")
            
            if can_merge:
                sources_to_merge.append(comp3_src)
                used_sources.add(('comp3', comp3_src['source_id']))
                print(f"    ✓ 添加到合并列表")
            else:
                print(f"    ✗ 不满足合并条件")
        
        # 如果有可合并的源（除了Comp2源本身）
        if len(sources_to_merge) > 1:
            merged = merge_sources(sources_to_merge, results)
            if merged:
                all_final_sources.append(merged)
                used_sources.add(('comp2', comp2_source['source_id']))
                print(f"\n✓ 创建合并源，包含 {len(sources_to_merge)} 个源")
                print(f"  合并的源: Component 2-1 + Component 3 源 {[src['source_id'] for src in sources_to_merge[1:]]}")
        else:
            # 不能合并，保留为独立源
            all_final_sources.append({
                'merged_id': None,
                'pixel_count': comp2_source['pixel_count'],
                'pixels': comp2_source['pixels'],
                'centroid': comp2_source['centroid'],
                'component_sources': [comp2_source],
                'components': [2],
                'weighted_average': comp2_source['weighted_average'],
                'is_merged': False
            })
            used_sources.add(('comp2', comp2_source['source_id']))
            print(f"\n✗ 没有可合并的源，保留为独立源")
    else:
        print("\nComponent 2 没有源")
    
    # 4. 添加剩余的源（未使用的源）
    temp_sources = []
    
    # 添加Component 1的所有源
    for src in comp1_sources:
        temp_sources.append({
            'merged_id': None,
            'pixel_count': src['pixel_count'],
            'pixels': src['pixels'],
            'centroid': src['centroid'],
            'component_sources': [src],
            'components': [1],
            'weighted_average': src['weighted_average'],
            'is_merged': False
        })
    
    # 添加已经创建的合并源
    for src in all_final_sources:
        temp_sources.append(src)
    
    # 添加Component 2中未使用的源（除了第一个）
    for src in comp2_sources:
        if ('comp2', src['source_id']) not in used_sources:
            temp_sources.append({
                'merged_id': None,
                'pixel_count': src['pixel_count'],
                'pixels': src['pixels'],
                'centroid': src['centroid'],
                'component_sources': [src],
                'components': [2],
                'weighted_average': src['weighted_average'],
                'is_merged': False
            })
    
    # 添加Component 3中未使用的源（除了被合并的）
    for src in comp3_sources:
        if ('comp3', src['source_id']) not in used_sources:
            temp_sources.append({
                'merged_id': None,
                'pixel_count': src['pixel_count'],
                'pixels': src['pixels'],
                'centroid': src['centroid'],
                'component_sources': [src],
                'components': [3],
                'weighted_average': src['weighted_average'],
                'is_merged': False
            })
    
    # 重新分配ID
    for i, source in enumerate(temp_sources):
        source['merged_id'] = i + 1
    
    all_final_sources = temp_sources
    
    print(f"\n{'='*60}")
    print("最终源列表")
    print("="*60)
    print(f"总源数: {len(all_final_sources)}")
    for src in all_final_sources:
        if src.get('is_merged', False):
            print(f"  源 {src['merged_id']}: 合并源 (分量 {src['components']}), "
                  f"{src['pixel_count']} 像素, v={src['weighted_average']['velocity']:.2f} km/s, "
                  f"FWHM={src['weighted_average']['fwhm']:.2f} km/s")
        else:
            print(f"  源 {src['merged_id']}: 分量 {src['components'][0]}, "
                  f"{src['pixel_count']} 像素, v={src['weighted_average']['velocity']:.2f} km/s, "
                  f"FWHM={src['weighted_average']['fwhm']:.2f} km/s")
    
    # 5. 创建参数图
    merged_amplitude, merged_position, merged_dispersion, merged_fwhm, merged_moment0, merged_source_id = \
        create_final_parameter_maps(all_final_sources, results, shape)
    
    # 6. 绘制所有源（使用WCS坐标）
    if wcs_2d is not None:
        plot_all_sources(all_final_sources, merged_amplitude, merged_position,
                        merged_fwhm, merged_moment0, merged_source_id, output_dir, wcs_2d)
        
        # 7. 生成源masks（新增）
        generate_source_masks(all_final_sources, merged_source_id, results, output_dir, wcs_2d)
        
        # 8. 如果提供了原始FITS文件，创建mask叠加图
        if fits_file is not None and os.path.exists(fits_file):
            overlay_file = os.path.join(output_dir, 'masks_overlay_on_data.png')
            mask_file = os.path.join(output_dir, 'all_sources_mask.fits')
            overlay_masks_on_data(fits_file, mask_file, overlay_file)
    else:
        print("Warning: No WCS information available, cannot create celestial coordinate plots")
    
    # 9. 保存源信息
    save_source_info(all_final_sources, output_dir)
    
    # 10. 保存DAT文件
    final_dat = os.path.join(output_dir, 'final_sources.dat')
    save_final_dat(results, all_final_sources, original_dat_file, final_dat)
    
    # 11. 保存合并检查结果
    check_file = os.path.join(output_dir, 'merge_check_results.txt')
    with open(check_file, 'w') as f:
        f.write("="*60 + "\n")
        f.write("合并条件检查结果\n")
        f.write("="*60 + "\n\n")
        
        for check in check_results:
            f.write(f"\n{check['type']}:\n")
            f.write(f"  Component 2 速度: {check['v1']:.4f} km/s\n")
            f.write(f"  Component 2 FWHM: {check['fwhm1']:.4f} km/s\n")
            f.write(f"  Component 3 速度: {check['v2']:.4f} km/s\n")
            f.write(f"  Component 3 FWHM: {check['fwhm2']:.4f} km/s\n")
            f.write(f"  较宽的FWHM: {max(check['fwhm1'], check['fwhm2']):.4f} km/s\n")
            f.write(f"  速度差: {check['dv']:.4f} km/s\n")
            f.write(f"  阈值 (0.7 × 较宽FWHM): {check['threshold']:.4f} km/s\n")
            f.write(f"  是否合并: {check['can_merge']}\n")
    
    print(f"\n合并检查结果保存至: {check_file}")
    
    print("\n" + "="*60)
    print("处理完成")
    print("="*60)
    print(f"\n输出文件保存至: {output_dir}")
    print(f"  - all_sources_overview.pdf")
    print(f"  - all_sources_composition.pdf")
    print(f"  - all_sources_mask.fits/npy (所有源的mask，不同ID值)")
    print(f"  - binary_mask.fits/npy (二值mask)")
    print(f"  - sources_3d_mask.fits/npy (3D mask，用于PV图)")
    print(f"  - masks/source_X_mask.fits/npy (每个源的单独mask)")
    print(f"  - sources.reg (DS9区域文件)")
    print(f"  - masks_overlay_on_data.png (mask叠加在原始数据上)")
    print(f"  - final_source_information.txt")
    print(f"  - final_sources.dat")
    print(f"  - merge_check_results.txt")

# ==================== 11. 脚本执行 ====================
if __name__ == "__main__":
    # 请修改这些路径
    DAT_FILE = "./n_gauss=3/output_individual_source/source_filtered_rohsa_pixel.dat"
    ORIGINAL_DAT_FILE = "MS_ROHSA_3ngauss_1_3D_1.dat"
    FITS_FILE = "CRAFTS_-4.7_-350_-150_Original.fits"  # 你的FITS文件路径
    OUTPUT_DIR = "./n_gauss=3/output_merged_sources/picture/mask"
    
    # 参数设置
    MIN_PIXELS = 10
    MAX_GAP = 2
    VELOCITY_THRESHOLD_FACTOR = 0.7
    
    # 运行
    main(
        dat_file=DAT_FILE,
        original_dat_file=ORIGINAL_DAT_FILE,
        output_dir=OUTPUT_DIR,
        fits_file=FITS_FILE,
        min_pixels=MIN_PIXELS,
        max_gap=MAX_GAP,
        velocity_threshold_factor=VELOCITY_THRESHOLD_FACTOR
    )


SOURCE MERGING (Component 2 vs Component 3)
Input DAT file: ./n_gauss=3/output_individual_source/source_filtered_rohsa_pixel.dat
Output directory: ./n_gauss=3/output_merged_sources/picture/mask
Min pixels per source: 10
Max spatial gap: 2
Velocity threshold factor: 0.7

Reading WCS info from FITS: CRAFTS_-4.7_-350_-150_Original.fits
SpectralCube shape: (994, 89, 153)
Successfully loaded WCS coordinates

READING FILTERED ROHSA DAT FILE
Reading Gaussian parameters (pixel units)...
Opening data file
Gaussian pixel array shape: (9, 89, 153)
Data type: float64
Converting to physical units...
Spatial dimensions: X=153, Y=89
Number of Gaussian components: 3

Component 1:
  Non-zero pixels (phys): 4872
  Position range (phys): [-255.87, -149.83] km/s

Component 2:
  Non-zero pixels (phys): 291
  Position range (phys): [-282.77, -149.83] km/s

Component 3:
  Non-zero pixels (phys): 56
  Position range (phys): [-297.10, -149.83] km/s

检测到 3 个高斯分量

EXTRACTING SOURCES FROM EACH COMPONENT

Compone

<Figure size 1200x1000 with 0 Axes>

### 保存每个源

In [35]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from scipy import ndimage
import os
from matplotlib.colors import ListedColormap
from collections import defaultdict


# ==================== 1. 读取DAT文件（同时获取像素单位和物理单位） ====================
def read_filtered_dat(dat_file):
    """
    读取筛选后的ROHSA DAT文件
    返回像素单位和物理单位的结果
    """
    print("\n" + "="*60)
    print("READING FILTERED ROHSA DAT FILE")
    print("="*60)
    
    # 使用core模块读取像素单位的gaussian
    print("Reading Gaussian parameters (pixel units)...")
    gaussian_pixel = core.read_gaussian(dat_file)
    
    print(f"Gaussian pixel array shape: {gaussian_pixel.shape}")
    print(f"Data type: {gaussian_pixel.dtype}")
    
    # 转换为物理单位
    print("Converting to physical units...")
    gaussian_physical = core.physical_gaussian(gaussian_pixel)
    
    # 检查返回值类型
    if isinstance(gaussian_physical, (int, float)):
        print("Warning: physical_gaussian returned a scalar. Using pixel units as fallback.")
        gaussian_physical = gaussian_pixel
    
    # 解析维度: (n_parameters * n_components, n_y, n_x)
    n_params_times_comp, n_y, n_x = gaussian_pixel.shape
    n_components = n_params_times_comp // 3
    
    print(f"Spatial dimensions: X={n_x}, Y={n_y}")
    print(f"Number of Gaussian components: {n_components}")
    
    # 提取每个成分的参数
    results = {}
    
    for comp in range(1, n_components + 1):
        # 像素单位
        amp_pixel = gaussian_pixel[3*(comp-1)]
        pos_pixel = gaussian_pixel[3*(comp-1) + 1]
        dis_pixel = gaussian_pixel[3*(comp-1) + 2]
        
        # 物理单位
        if isinstance(gaussian_physical, np.ndarray):
            amp_phys = gaussian_physical[3*(comp-1)]
            pos_phys = gaussian_physical[3*(comp-1) + 1]
            dis_phys = gaussian_physical[3*(comp-1) + 2]
        else:
            amp_phys = amp_pixel
            pos_phys = pos_pixel
            dis_phys = dis_pixel
        
        results[f'comp{comp}'] = {
            'amplitude_pixel': amp_pixel,
            'position_pixel': pos_pixel,
            'dispersion_pixel': dis_pixel,
            'amplitude_phys': amp_phys,
            'position_phys': pos_phys,
            'dispersion_phys': dis_phys,
            'component_id': comp
        }
        
        print(f"\nComponent {comp}:")
        print(f"  Non-zero pixels (phys): {np.sum(amp_phys > 0)}")
        if np.any(amp_phys > 0):
            print(f"  Position range (phys): [{np.min(pos_phys[pos_phys!=0]):.2f}, {np.max(pos_phys):.2f}] km/s")
    
    results['n_components'] = n_components
    results['shape'] = (n_x, n_y)
    results['original_data_pixel'] = gaussian_pixel
    results['original_data_physical'] = gaussian_physical
    
    return results

# ==================== 2. 提取每个成分的源 ====================
def extract_sources_from_component(results, comp, min_pixels=10, max_gap=2):
    """
    从单个成分提取源
    """
    amplitude = results[f'comp{comp}']['amplitude_phys']
    position = results[f'comp{comp}']['position_phys']
    dispersion = results[f'comp{comp}']['dispersion_phys']
    
    # 创建二值mask
    binary_mask = amplitude > 0
    
    if not np.any(binary_mask):
        return []
    
    # 连通区域分析
    structure = ndimage.generate_binary_structure(2, 1)
    if max_gap > 1:
        structure = ndimage.iterate_structure(structure, max_gap - 1)
    
    labeled_mask, num_features = ndimage.label(binary_mask, structure=structure)
    
    sources = []
    
    for label in range(1, num_features + 1):
        mask = labeled_mask == label
        pixel_count = np.sum(mask)
        
        if pixel_count >= min_pixels:
            y_indices, x_indices = np.where(mask)
            
            amplitudes = []
            positions = []
            dispersions = []
            pixels = []
            
            for y, x in zip(y_indices, x_indices):
                amp = amplitude[y, x]
                pos = position[y, x]
                dis = dispersion[y, x]
                
                amplitudes.append(amp)
                positions.append(pos)
                dispersions.append(dis)
                pixels.append((x, y))
            
            amplitudes = np.array(amplitudes)
            positions = np.array(positions)
            dispersions = np.array(dispersions)
            
            # 计算加权平均（以振幅为权重）
            total_amp = np.sum(amplitudes)
            if total_amp > 0:
                weighted_position = np.sum(positions * amplitudes) / total_amp
                weighted_dispersion = np.sum(dispersions * amplitudes) / total_amp
                weighted_fwhm = 2.355 * weighted_dispersion
                weighted_amplitude = total_amp / pixel_count
            else:
                weighted_position = np.mean(positions)
                weighted_dispersion = np.mean(dispersions)
                weighted_fwhm = 2.355 * weighted_dispersion
                weighted_amplitude = 0
            
            # 计算质心
            centroid_x = np.mean(x_indices)
            centroid_y = np.mean(y_indices)
            
            source_info = {
                'source_id': label,
                'component': comp,
                'pixel_count': pixel_count,
                'pixels': pixels,
                'centroid': (centroid_x, centroid_y),
                'weighted_average': {
                    'amplitude': weighted_amplitude,
                    'velocity': weighted_position,
                    'dispersion': weighted_dispersion,
                    'fwhm': weighted_fwhm
                },
                'bbox': (np.min(x_indices), np.max(x_indices), 
                        np.min(y_indices), np.max(y_indices))
            }
            sources.append(source_info)
    
    return sources

# ==================== 3. 判断合并条件 ====================
def check_merge_condition(source1, source2, velocity_threshold_factor=0.7):
    """
    判断两个源是否可以合并
    条件：速度差 < velocity_threshold_factor * 二者之中较宽的FWHM
    """
    v1 = source1['weighted_average']['velocity']
    v2 = source2['weighted_average']['velocity']
    fwhm1 = source1['weighted_average']['fwhm']
    fwhm2 = source2['weighted_average']['fwhm']
    
    dv = abs(v1 - v2)
    wider_fwhm = max(fwhm1, fwhm2)
    threshold = velocity_threshold_factor * wider_fwhm
    
    can_merge = dv < threshold
    
    return can_merge, dv, threshold

# ==================== 4. 合并多个源 ====================
def merge_sources(sources_to_merge, results):
    """
    合并多个源
    """
    if len(sources_to_merge) == 0:
        return None
    
    # 收集所有像素
    all_pixels = []
    all_components = []
    
    for src in sources_to_merge:
        all_pixels.extend(src['pixels'])
        all_components.append(src['component'])
    
    all_pixels = list(set(all_pixels))
    
    # 计算合并后的属性（振幅加权）
    total_amp = sum([src['weighted_average']['amplitude'] * src['pixel_count'] 
                     for src in sources_to_merge])
    
    if total_amp > 0:
        weighted_velocities = []
        weighted_dispersions = []
        
        for src in sources_to_merge:
            weight = src['weighted_average']['amplitude'] * src['pixel_count'] / total_amp
            weighted_velocities.append(src['weighted_average']['velocity'] * weight)
            weighted_dispersions.append(src['weighted_average']['dispersion'] * weight)
        
        merged_velocity = sum(weighted_velocities)
        merged_dispersion = sum(weighted_dispersions)
    else:
        merged_velocity = np.mean([src['weighted_average']['velocity'] for src in sources_to_merge])
        merged_dispersion = np.mean([src['weighted_average']['dispersion'] for src in sources_to_merge])
    
    # 计算质心
    x_coords = [p[0] for p in all_pixels]
    y_coords = [p[1] for p in all_pixels]
    
    merged_source = {
        'merged_id': None,
        'pixel_count': len(all_pixels),
        'pixels': all_pixels,
        'centroid': (np.mean(x_coords), np.mean(y_coords)),
        'component_sources': sources_to_merge,
        'components': list(set(all_components)),
        'weighted_average': {
            'velocity': merged_velocity,
            'dispersion': merged_dispersion,
            'fwhm': 2.355 * merged_dispersion,
            'amplitude': total_amp / len(all_pixels) if total_amp > 0 else 0
        },
        'is_merged': True
    }
    
    return merged_source

# ==================== 5. 创建最终参数图（包含Moment 0） ====================
def create_final_parameter_maps(all_sources, results, shape):
    """
    创建最终的参数图
    """
    n_x, n_y = shape
    n_components = results['n_components']
    
    merged_amplitude = np.zeros((n_y, n_x))
    merged_position = np.zeros((n_y, n_x))
    merged_dispersion = np.zeros((n_y, n_x))
    merged_fwhm = np.zeros((n_y, n_x))
    merged_moment0 = np.zeros((n_y, n_x))  # 添加Moment 0数组
    merged_source_id = np.zeros((n_y, n_x), dtype=int)
    
    for source in all_sources:
        source_id = source['merged_id']
        
        for x, y in source['pixels']:
            if 0 <= y < n_y and 0 <= x < n_x:
                pixel_amplitudes = []
                pixel_positions = []
                pixel_dispersions = []
                
                for comp_src in source['component_sources']:
                    comp = comp_src['component']
                    if (x, y) in comp_src['pixels']:
                        amp = results[f'comp{comp}']['amplitude_phys'][y, x]
                        pos = results[f'comp{comp}']['position_phys'][y, x]
                        dis = results[f'comp{comp}']['dispersion_phys'][y, x]
                        
                        if amp > 0:
                            pixel_amplitudes.append(amp)
                            pixel_positions.append(pos)
                            pixel_dispersions.append(dis)
                
                if pixel_amplitudes:
                    weights = np.array(pixel_amplitudes) / np.sum(pixel_amplitudes)
                    merged_amplitude[y, x] = np.sum(pixel_amplitudes)
                    merged_position[y, x] = np.sum(np.array(pixel_positions) * weights)
                    merged_dispersion[y, x] = np.sum(np.array(pixel_dispersions) * weights)
                    merged_fwhm[y, x] = 2.355 * merged_dispersion[y, x]
                    
                    # 计算Moment 0 (积分强度)
                    # 对于高斯函数：∫ A * exp(-(v-v0)^2/(2σ^2)) dv = A * σ * sqrt(2π)
                    # 注意：速度单位是km/s，所以Moment 0的单位是 K * km/s
                    merged_moment0[y, x] = merged_amplitude[y, x] * merged_dispersion[y, x] * np.sqrt(2 * np.pi)
                    
                    merged_source_id[y, x] = source_id
    
    return merged_amplitude, merged_position, merged_dispersion, merged_fwhm, merged_moment0, merged_source_id


# ==================== 7. 绘制所有源（使用WCS坐标） ====================
def plot_all_sources(all_sources, merged_amplitude, merged_position, 
                     merged_fwhm, merged_moment0, merged_source_id, output_dir, wcs_2d):
    """
    绘制所有源，使用WCS坐标系统
    """
    print("\n" + "="*60)
    print("PLOTTING ALL SOURCES WITH WCS")
    print("="*60)
    
    n_sources = len(all_sources)
    
    # 创建2x2的子图，去掉大标题，子图间距更小
    fig = plt.figure(figsize=(16, 14))
    
    # 调整子图间距，让子图更靠近
    plt.subplots_adjust(left=0.08, right=0.92, bottom=0.08, top=0.95, 
                        wspace=0.15, hspace=0.1)
    
    # 为每个子图创建WCS投影
    ax1 = plt.subplot(2, 2, 1, projection=wcs_2d)
    ax2 = plt.subplot(2, 2, 2, projection=wcs_2d)
    ax3 = plt.subplot(2, 2, 3, projection=wcs_2d)
    ax4 = plt.subplot(2, 2, 4, projection=wcs_2d)
    
    axes_list = [ax1, ax2, ax3, ax4]
    
    # 图1：所有源的mask
    if n_sources > 0:
        colors = plt.cm.tab20(np.linspace(0, 1, n_sources))
        colors = np.vstack([[0,0,0,1], colors])
        cmap = ListedColormap(colors)
        
        im1 = ax1.imshow(merged_source_id, origin='lower', cmap=cmap,
                        vmin=0, vmax=n_sources, transform=ax1.get_transform('pixel'))
        
        # 标注源的中心
        for source in all_sources:
            x_center, y_center = source['centroid']
            # 转换到世界坐标
            world_coords = wcs_2d.pixel_to_world(x_center, y_center)
            ra_center = world_coords.ra.deg
            dec_center = world_coords.dec.deg
            
            # 计算偏移方向
            angle = source['merged_id'] * 0
            offset_ra = 0.15 * np.cos(np.radians(angle))
            offset_dec = 0.1 * np.sin(np.radians(angle))
            
            # 标注中心点
            ax1.plot(ra_center, dec_center, 'r+', markersize=8, markeredgewidth=2, 
                    transform=ax1.get_transform('world'))
            
            # 标注数字，位置偏移
            ax1.text(ra_center + offset_ra, dec_center + offset_dec, f'{source["merged_id"]}', 
                    color='white', fontsize=12, weight='bold',
                    transform=ax1.get_transform('world'),
                    bbox=dict(boxstyle='round', facecolor='black', alpha=0.3, pad=0.3))
        
        # 添加colorbar
        cbar1 = plt.colorbar(im1, ax=ax1, pad=0.15, shrink=1, aspect=30, location='bottom')
        cbar1.ax.tick_params(labelsize=16)
    else:
        im1 = ax1.imshow(merged_source_id, origin='lower', cmap='gray')
    
    # 图2：Moment 0图（积分强度）
    # 计算合适的colorbar范围（去掉异常值）
    moment0_data = merged_moment0[merged_moment0 > 0]
    if len(moment0_data) > 0:
        vmin = np.percentile(moment0_data, 2)  # 2%分位数
        vmax = np.percentile(moment0_data, 98)  # 98%分位数
    else:
        vmin, vmax = 0, 100
    
    im2 = ax2.imshow(merged_moment0, origin='lower', cmap='viridis',
                    vmin=0, vmax=40, transform=ax2.get_transform('pixel'))
    cbar2 = plt.colorbar(im2, ax=ax2, pad=0.15, shrink=1, aspect=30, location='bottom')
    cbar2.set_label('Moment 0 [K km s$^{-1}$]', fontsize=18)
    cbar2.ax.tick_params(labelsize=16)
    
    # 图3：位置图（速度）
    im3 = ax3.imshow(merged_position, origin='lower', cmap='coolwarm',
                    vmin=-300, vmax=-200, transform=ax3.get_transform('pixel'))
    cbar3 = plt.colorbar(im3, ax=ax3, pad=0.15, shrink=1, aspect=30, location='bottom')
    cbar3.set_label('Velocity [km s$^{-1}$]', fontsize=18)
    cbar3.ax.tick_params(labelsize=16)
    
    # 图4：FWHM图
    im4 = ax4.imshow(merged_fwhm, origin='lower', cmap='cubehelix',
                    vmin=0, vmax=50, transform=ax4.get_transform('pixel'))
    cbar4 = plt.colorbar(im4, ax=ax4, pad=0.15, shrink=1, aspect=30, location='bottom')
    cbar4.set_label('FWHM [km s$^{-1}$]', fontsize=18)
    cbar4.ax.tick_params(labelsize=16)
    
    # 配置所有子图的坐标轴
    for i, ax in enumerate(axes_list):
        # 设置标签
        ax.set_xlabel("Right Ascension [hours]", fontsize=18)
        
        # 根据子图位置决定是否显示Dec标签
        if i == 0 or i == 2:  # 第一列（子图1和3）显示Dec标签
            ax.set_ylabel("Declination [°]", fontsize=18)
            # 确保Dec坐标轴可见
            ax.coords[1].set_ticklabel_visible(True)
            ax.coords[1].set_ticks_visible(True)
            ax.coords[1].set_axislabel('Declination [°]')
        else:  # 第二列（子图2和4）不显示Dec标签
            ax.set_ylabel("")
            # 隐藏Dec坐标轴的标签和刻度
            ax.coords[1].set_ticklabel_visible(False)
            ax.coords[1].set_ticks_visible(False)
            ax.coords[1].set_axislabel('')
        
        ax.tick_params(axis='both', which='both', direction='out', labelsize=14)
        
        # 配置WCS坐标轴
        # RA坐标轴设置
        ax.coords[0].set_ticks_position('b')
        ax.coords[0].set_ticklabel_position('b')
        ax.coords[0].set_axislabel('Right Ascension [hours]')
        ax.coords[0].set_format_unit(u.hourangle)
        
        # Dec坐标轴设置
        if i == 0 or i == 2:
            ax.coords[1].set_ticks_position('l')
            ax.coords[1].set_ticklabel_position('l')
        else:
            ax.coords[1].set_ticks_position('')
            ax.coords[1].set_ticklabel_position('')
        
        ax.coords[1].set_format_unit(u.deg)
        
        # 创建上侧的RA角度副坐标轴
        try:
            overlay = ax.get_coords_overlay('fk5')
            overlay[0].set_axislabel('Right Ascension [°]', fontsize=18)
            overlay[0].tick_params(labelsize=14, direction='in')
            overlay[0].set_ticks_position('t')
            overlay[0].set_ticklabel_position('t')
            overlay[0].set_format_unit(u.deg)
            
            # 隐藏不需要的坐标轴
            overlay[1].set_axislabel('')
            overlay[1].set_ticklabel_visible(False)
            overlay[1].set_ticks_visible(False)
            
            # 添加网格
            overlay.grid(color='grey', ls='dotted', lw=0.5)
        except:
            pass
    
    all_file = os.path.join(output_dir, 'all_sources_overview.pdf')
    plt.savefig(all_file, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"All sources overview saved to: {all_file}")
    
    # 图2：每个源的组成图
    if n_sources > 0:
        n_cols = min(3, n_sources)
        n_rows = (n_sources + n_cols - 1) // n_cols
        fig = plt.figure(figsize=(5*n_cols, 4*n_rows))
        
        plt.subplots_adjust(left=0.08, right=0.92, bottom=0.08, top=0.95, 
                           wspace=0.2, hspace=0.2)
        
        for i, source in enumerate(all_sources):
            ax = fig.add_subplot(n_rows, n_cols, i+1, projection=wcs_2d)
            source_mask = merged_source_id == source['merged_id']
            
            ax.imshow(source_mask, origin='lower', cmap='Blues', alpha=0.3,
                     transform=ax.get_transform('pixel'))
            
            # 标注组成它的原始源
            colors = ['red', 'green', 'blue', 'purple', 'orange', 'brown']
            for j, comp_src in enumerate(source['component_sources']):
                comp_mask = np.zeros_like(source_mask)
                for x, y in comp_src['pixels']:
                    if 0 <= y < comp_mask.shape[0] and 0 <= x < comp_mask.shape[1]:
                        comp_mask[y, x] = True
                
                edges = ndimage.binary_dilation(comp_mask) & ~comp_mask
                y_edges, x_edges = np.where(edges)
                
                world_coords = wcs_2d.pixel_to_world(x_edges, y_edges)
                ra_edges = world_coords.ra.deg
                dec_edges = world_coords.dec.deg
                
                ax.scatter(ra_edges, dec_edges, c=colors[j % len(colors)], 
                          s=2, label=f'Comp{comp_src["component"]}-Src{comp_src["source_id"]}', 
                          alpha=0.8, transform=ax.get_transform('world'))
            
            # 显示源的类型
            if source.get('is_merged', False):
                source_type = f'Merged (Comp{source["components"]})'
            else:
                source_type = f'Comp{source["components"][0]}'
            
            # 计算标注偏移
            offset_x = 0.02
            offset_y = 0.02
            if source['merged_id'] % 2 == 0:
                offset_x = -0.02
            if source['merged_id'] % 3 == 0:
                offset_y = -0.02
            
            # 获取源的中心坐标
            x_center, y_center = source['centroid']
            world_coords = wcs_2d.pixel_to_world(x_center, y_center)
            ra_center = world_coords.ra.deg
            dec_center = world_coords.dec.deg
            
            # 在源中心附近添加信息文本
            info_text = f'{source["pixel_count"]}p\n{source["weighted_average"]["velocity"]:.0f}kms\nFWHM:{source["weighted_average"]["fwhm"]:.0f}'
            ax.text(ra_center + offset_x, dec_center + offset_y, info_text,
                   fontsize=6, transform=ax.get_transform('world'),
                   bbox=dict(boxstyle='round', facecolor='white', alpha=0.7, pad=0.2))
            
            ax.set_title(f'Source {source["merged_id"]}: {source_type}', fontsize=9, pad=3)
            
            ax.set_xlabel("RA [hours]", fontsize=8)
            ax.set_ylabel("Dec [°]", fontsize=8)
            ax.tick_params(axis='both', which='both', direction='out', labelsize=7)
            
            ax.coords[0].set_format_unit(u.hourangle)
            ax.coords[1].set_format_unit(u.deg)
            
            if len(source['component_sources']) <= 3:
                ax.legend(loc='upper right', fontsize=6, framealpha=0.8)
        
        plt.tight_layout()
        
        comp_file = os.path.join(output_dir, 'all_sources_composition.pdf')
        plt.savefig(comp_file, dpi=150, bbox_inches='tight')
        plt.close()
        print(f"All sources composition saved to: {comp_file}")

# ==================== 7. 保存每个源为独立的DAT文件 ====================
def save_individual_source_dat(results, source, output_dir, original_header_lines):
    """
    保存单个源为独立的DAT文件
    """
    n_x, n_y = results['shape']
    n_components = results['n_components']
    
    # 创建源专属的输出目录
    source_dir = os.path.join(output_dir, f'source_{source["merged_id"]:03d}')
    os.makedirs(source_dir, exist_ok=True)
    
    # 创建该源的数据数组（只包含该源的像素）
    data_lines = []
    
    for y in range(n_y):
        for x in range(n_x):
            # 检查该像素是否属于当前源
            pixel_belongs_to_source = False
            for comp_src in source['component_sources']:
                if (x, y) in comp_src['pixels']:
                    pixel_belongs_to_source = True
                    break
            
            if pixel_belongs_to_source:
                # 该像素属于当前源，写入所有分量的数据
                for comp in range(1, n_components + 1):
                    amp = 0
                    pos = 0
                    dis = 0
                    
                    # 检查该像素是否属于当前分量
                    for comp_src in source['component_sources']:
                        if comp_src['component'] == comp and (x, y) in comp_src['pixels']:
                            amp = results[f'comp{comp}']['amplitude_pixel'][y, x]
                            pos = results[f'comp{comp}']['position_pixel'][y, x]
                            dis = results[f'comp{comp}']['dispersion_pixel'][y, x]
                            break
                    
                    line = f"    {y:4d}    {x:4d}    {amp:20.16f}    {pos:20.16f}    {dis:20.16f}\n"
                    data_lines.append(line)
            else:
                # 该像素不属于当前源，所有分量写0
                for comp in range(1, n_components + 1):
                    line = f"    {y:4d}    {x:4d}    {0:20.16f}    {0:20.16f}    {0:20.16f}\n"
                    data_lines.append(line)
    
    # 保存DAT文件
    dat_filename = f'source_{source["merged_id"]:03d}.dat'
    dat_filepath = os.path.join(source_dir, dat_filename)
    
    with open(dat_filepath, 'w') as f:
        f.writelines(original_header_lines)
        f.writelines(data_lines)
    
    print(f"  Source {source['merged_id']} saved to: {dat_filepath}")
    
    # 同时保存源的参数信息
    info_filepath = os.path.join(source_dir, f'source_{source["merged_id"]:03d}_info.txt')
    with open(info_filepath, 'w') as f:
        f.write("="*60 + "\n")
        f.write(f"SOURCE {source['merged_id']} INFORMATION\n")
        f.write("="*60 + "\n\n")
        
        if source.get('is_merged', False):
            f.write(f"Type: MERGED\n")
            f.write(f"Components involved: {source['components']}\n")
        else:
            f.write(f"Type: SINGLE COMPONENT\n")
            f.write(f"Component: {source['components'][0]}\n")
        
        f.write(f"Number of pixels: {source['pixel_count']}\n")
        f.write(f"Centroid (X, Y): ({source['centroid'][0]:.2f}, {source['centroid'][1]:.2f})\n")
        f.write(f"Mean Velocity: {source['weighted_average']['velocity']:.6f} km/s\n")
        f.write(f"Mean FWHM: {source['weighted_average']['fwhm']:.6f} km/s\n")
        f.write(f"Mean Amplitude: {source['weighted_average']['amplitude']:.6f} K\n\n")
        
        f.write("Original Sources:\n")
        for comp_src in source['component_sources']:
            f.write(f"  Component {comp_src['component']}, Source {comp_src['source_id']}:\n")
            f.write(f"    Velocity: {comp_src['weighted_average']['velocity']:.6f} km/s\n")
            f.write(f"    FWHM: {comp_src['weighted_average']['fwhm']:.6f} km/s\n")
            f.write(f"    Pixels: {comp_src['pixel_count']}\n")
        
        f.write("\n" + "="*60 + "\n")
        f.write("PARAMETER STATISTICS\n")
        f.write("="*60 + "\n\n")
        
        # 计算该源的参数统计
        amplitudes = []
        positions = []
        dispersions = []
        
        for comp_src in source['component_sources']:
            comp = comp_src['component']
            for x, y in comp_src['pixels']:
                amp = results[f'comp{comp}']['amplitude_phys'][y, x]
                pos = results[f'comp{comp}']['position_phys'][y, x]
                dis = results[f'comp{comp}']['dispersion_phys'][y, x]
                if amp > 0:
                    amplitudes.append(amp)
                    positions.append(pos)
                    dispersions.append(dis)
        
        if amplitudes:
            f.write(f"Amplitude (K):\n")
            f.write(f"  Mean: {np.mean(amplitudes):.6f}\n")
            f.write(f"  Std:  {np.std(amplitudes):.6f}\n")
            f.write(f"  Min:  {np.min(amplitudes):.6f}\n")
            f.write(f"  Max:  {np.max(amplitudes):.6f}\n\n")
            
            f.write(f"Velocity (km/s):\n")
            f.write(f"  Mean: {np.mean(positions):.6f}\n")
            f.write(f"  Std:  {np.std(positions):.6f}\n")
            f.write(f"  Min:  {np.min(positions):.6f}\n")
            f.write(f"  Max:  {np.max(positions):.6f}\n\n")
            
            f.write(f"Dispersion (km/s):\n")
            f.write(f"  Mean: {np.mean(dispersions):.6f}\n")
            f.write(f"  Std:  {np.std(dispersions):.6f}\n")
            f.write(f"  Min:  {np.min(dispersions):.6f}\n")
            f.write(f"  Max:  {np.max(dispersions):.6f}\n\n")
    
    return source_dir

def save_all_individual_sources(results, all_sources, original_dat_file, output_dir):
    """
    保存所有源为独立的DAT文件
    """
    print("\n" + "="*60)
    print("SAVING INDIVIDUAL SOURCES TO .DAT FILES")
    print("="*60)
    
    # 读取原始DAT文件的头部
    try:
        with open(original_dat_file, 'r') as f:
            header_lines = []
            for i in range(27):
                line = f.readline()
                header_lines.append(line)
    except:
        # 如果无法读取原始文件，创建默认头部
        header_lines = [f"Header line {i+1}\n" for i in range(27)]
        print("Warning: Could not read original DAT header, using default header.")
    
    # 为每个源创建单独的目录和文件
    source_dirs = []
    for source in all_sources:
        source_dir = save_individual_source_dat(results, source, output_dir, header_lines)
        source_dirs.append(source_dir)
    
    # 创建总索引文件
    index_file = os.path.join(output_dir, 'sources_index.txt')
    with open(index_file, 'w') as f:
        f.write("="*60 + "\n")
        f.write("INDIVIDUAL SOURCES INDEX\n")
        f.write("="*60 + "\n\n")
        
        f.write(f"Total sources: {len(all_sources)}\n\n")
        
        for source in all_sources:
            f.write(f"\n{'='*50}\n")
            f.write(f"Source {source['merged_id']:03d}\n")
            f.write(f"{'='*50}\n")
            f.write(f"Directory: source_{source['merged_id']:03d}/\n")
            f.write(f"Data file: source_{source['merged_id']:03d}.dat\n")
            f.write(f"Info file: source_{source['merged_id']:03d}_info.txt\n\n")
            
            if source.get('is_merged', False):
                f.write(f"Type: MERGED (from Components {source['components']})\n")
            else:
                f.write(f"Type: SINGLE COMPONENT (Component {source['components'][0]})\n")
            
            f.write(f"Number of pixels: {source['pixel_count']}\n")
            f.write(f"Centroid: ({source['centroid'][0]:.2f}, {source['centroid'][1]:.2f})\n")
            f.write(f"Mean velocity: {source['weighted_average']['velocity']:.2f} km/s\n")
            f.write(f"Mean FWHM: {source['weighted_average']['fwhm']:.2f} km/s\n")
            f.write(f"Mean amplitude: {source['weighted_average']['amplitude']:.2f} K\n")
    
    print(f"\nAll individual source files saved to: {output_dir}")
    print(f"Index file saved to: {index_file}")
    print(f"Total {len(all_sources)} sources saved")

# ==================== 8. 保存DAT文件 ====================
def save_final_dat(results, all_sources, original_dat_file, output_file):
    """
    保存最终的DAT文件（像素单位）
    """
    print("\n" + "="*60)
    print("SAVING FINAL RESULTS TO .DAT FORMAT (PIXEL UNITS)")
    print("="*60)
    
    try:
        n_x, n_y = results['shape']
        n_components = results['n_components']
        
        with open(original_dat_file, 'r') as f:
            header_lines = []
            for i in range(27):
                line = f.readline()
                header_lines.append(line)
        
        source_id_map = {}
        for source in all_sources:
            for x, y in source['pixels']:
                source_id_map[(x, y)] = source['merged_id']
        
        data_lines = []
        
        for y in range(n_y):
            for x in range(n_x):
                for comp in range(1, n_components + 1):
                    if (x, y) in source_id_map:
                        for source in all_sources:
                            if source['merged_id'] == source_id_map[(x, y)]:
                                pixel_amp = 0
                                pixel_pos = 0
                                pixel_dis = 0
                                for comp_src in source['component_sources']:
                                    if comp_src['component'] == comp and (x, y) in comp_src['pixels']:
                                        amp = results[f'comp{comp}']['amplitude_pixel'][y, x]
                                        pos = results[f'comp{comp}']['position_pixel'][y, x]
                                        dis = results[f'comp{comp}']['dispersion_pixel'][y, x]
                                        if amp > 0:
                                            pixel_amp = amp
                                            pixel_pos = pos
                                            pixel_dis = dis
                                            break
                                line = f"    {y:4d}    {x:4d}    {pixel_amp:20.16f}    {pixel_pos:20.16f}    {pixel_dis:20.16f}\n"
                                data_lines.append(line)
                                break
                    else:
                        line = f"    {y:4d}    {x:4d}    {0:20.16f}    {0:20.16f}    {0:20.16f}\n"
                        data_lines.append(line)
        
        with open(output_file, 'w') as f:
            f.writelines(header_lines)
            f.writelines(data_lines)
        
        print(f"\nFinal .dat file saved to: {output_file}")
        
    except Exception as e:
        print(f"Error saving final .dat: {e}")

# ==================== 9. 保存源信息 ====================
def save_source_info(all_sources, output_dir):
    """
    保存源信息到文本文件
    """
    info_file = os.path.join(output_dir, 'final_source_information.txt')
    
    with open(info_file, 'w') as f:
        f.write("="*60 + "\n")
        f.write("FINAL SOURCE INFORMATION\n")
        f.write("="*60 + "\n\n")
        
        f.write(f"Total sources: {len(all_sources)}\n\n")
        
        for source in all_sources:
            f.write(f"\n{'='*50}\n")
            f.write(f"SOURCE {source['merged_id']}\n")
            f.write(f"{'='*50}\n\n")
            
            if source.get('is_merged', False):
                f.write(f"Type: MERGED\n")
                f.write(f"Components involved: {source['components']}\n")
            else:
                f.write(f"Type: SINGLE COMPONENT\n")
                f.write(f"Component: {source['components'][0]}\n")
            
            f.write(f"Number of pixels: {source['pixel_count']}\n")
            f.write(f"Centroid (X, Y): ({source['centroid'][0]:.2f}, {source['centroid'][1]:.2f})\n")
            f.write(f"Mean Velocity: {source['weighted_average']['velocity']:.6f} km/s\n")
            f.write(f"Mean FWHM: {source['weighted_average']['fwhm']:.6f} km/s\n")
            f.write(f"Mean Amplitude: {source['weighted_average']['amplitude']:.6f} K\n\n")
            
            f.write("Original Sources:\n")
            for comp_src in source['component_sources']:
                f.write(f"  Component {comp_src['component']}, Source {comp_src['source_id']}:\n")
                f.write(f"    Velocity: {comp_src['weighted_average']['velocity']:.6f} km/s\n")
                f.write(f"    FWHM: {comp_src['weighted_average']['fwhm']:.6f} km/s\n")
                f.write(f"    Pixels: {comp_src['pixel_count']}\n")
    
    print(f"Source information saved to: {info_file}")

# ==================== 10. 主函数 ====================
def main(dat_file, original_dat_file, output_dir, fits_file=None,
         min_pixels=10, max_gap=2, velocity_threshold_factor=0.7):
    """
    主函数 - 分别比较Component 2的第一个源和Component 3的第1、2个源
    """
    print("\n" + "="*60)
    print("SOURCE MERGING (Component 2 vs Component 3)")
    print("="*60)
    print(f"Input DAT file: {dat_file}")
    print(f"Output directory: {output_dir}")
    print(f"Min pixels per source: {min_pixels}")
    print(f"Max spatial gap: {max_gap}")
    print(f"Velocity threshold factor: {velocity_threshold_factor}")
    
    os.makedirs(output_dir, exist_ok=True)
    
    # 获取WCS坐标信息
    wcs_2d = None
    if fits_file is not None and os.path.exists(fits_file):
        print(f"\nReading WCS info from FITS: {fits_file}")
        wcs_2d, n_x, n_y = get_coordinate_info(fits_file)
        if wcs_2d is not None:
            print("Successfully loaded WCS coordinates")
        else:
            print("Could not load WCS coordinates")
    else:
        print("No FITS file provided")
    
    # 1. 读取DAT文件
    results = read_filtered_dat(dat_file)
    shape = results['shape']
    n_components = results['n_components']
    
    print(f"\n检测到 {n_components} 个高斯分量")
    
    # 2. 提取每个成分的源
    print("\n" + "="*60)
    print("EXTRACTING SOURCES FROM EACH COMPONENT")
    print("="*60)
    
    comp1_sources = extract_sources_from_component(results, 1, min_pixels, max_gap)
    comp2_sources = extract_sources_from_component(results, 2, min_pixels, max_gap)
    comp3_sources = extract_sources_from_component(results, 3, min_pixels, max_gap)
    
    print(f"\nComponent 1: {len(comp1_sources)} sources")
    for src in comp1_sources:
        print(f"  Source {src['source_id']}: {src['pixel_count']} pixels, v={src['weighted_average']['velocity']:.2f} km/s, FWHM={src['weighted_average']['fwhm']:.2f}")
    
    print(f"\nComponent 2: {len(comp2_sources)} sources")
    for src in comp2_sources:
        print(f"  Source {src['source_id']}: {src['pixel_count']} pixels, v={src['weighted_average']['velocity']:.2f} km/s, FWHM={src['weighted_average']['fwhm']:.2f}")
    
    print(f"\nComponent 3: {len(comp3_sources)} sources")
    for src in comp3_sources:
        print(f"  Source {src['source_id']}: {src['pixel_count']} pixels, v={src['weighted_average']['velocity']:.2f} km/s, FWHM={src['weighted_average']['fwhm']:.2f}")
    
    # 3. 分别比较Component 2的第一个源和Component 3的第1、2个源
    all_final_sources = []
    check_results = []
    used_sources = set()
    sources_to_merge = []
    
    if comp2_sources:
        comp2_source = comp2_sources[0]  # Component 2的第一个源
        print(f"\n{'='*60}")
        print(f"检查 Component 2 源 1 (v={comp2_source['weighted_average']['velocity']:.2f} km/s, FWHM={comp2_source['weighted_average']['fwhm']:.2f})")
        print(f"{'='*60}")
        
        # 先添加Component 2源到待合并列表
        sources_to_merge.append(comp2_source)
        
        # 检查Component 3的第1、2个源
        comp3_sources_to_check = []
        if len(comp3_sources) >= 1:
            comp3_sources_to_check.append(comp3_sources[0])
        if len(comp3_sources) >= 2:
            comp3_sources_to_check.append(comp3_sources[1])
        
        for comp3_src in comp3_sources_to_check:
            can_merge, dv, threshold = check_merge_condition(
                comp2_source, comp3_src, velocity_threshold_factor
            )
            
            check_results.append({
                'type': f'Comp2-Src1 vs Comp3-Src{comp3_src["source_id"]}',
                'v1': comp2_source['weighted_average']['velocity'],
                'v2': comp3_src['weighted_average']['velocity'],
                'fwhm1': comp2_source['weighted_average']['fwhm'],
                'fwhm2': comp3_src['weighted_average']['fwhm'],
                'dv': dv,
                'threshold': threshold,
                'can_merge': can_merge
            })
            
            print(f"\n  Comp2-Src1 vs Comp3-Src{comp3_src['source_id']}:")
            print(f"    v1={comp2_source['weighted_average']['velocity']:.2f}, v2={comp3_src['weighted_average']['velocity']:.2f}, dv={dv:.2f}")
            print(f"    fwhm1={comp2_source['weighted_average']['fwhm']:.2f}, fwhm2={comp3_src['weighted_average']['fwhm']:.2f}")
            print(f"    Wider FWHM: {max(comp2_source['weighted_average']['fwhm'], comp3_src['weighted_average']['fwhm']):.2f}")
            print(f"    Threshold (0.7 × wider FWHM): {threshold:.2f}")
            print(f"    Can merge: {can_merge}")
            
            if can_merge:
                sources_to_merge.append(comp3_src)
                used_sources.add(('comp3', comp3_src['source_id']))
                print(f"    ✓ 添加到合并列表")
            else:
                print(f"    ✗ 不满足合并条件")
        
        # 如果有可合并的源（除了Comp2源本身）
        if len(sources_to_merge) > 1:
            merged = merge_sources(sources_to_merge, results)
            if merged:
                all_final_sources.append(merged)
                used_sources.add(('comp2', comp2_source['source_id']))
                print(f"\n✓ 创建合并源，包含 {len(sources_to_merge)} 个源")
                print(f"  合并的源: Component 2-1 + Component 3 源 {[src['source_id'] for src in sources_to_merge[1:]]}")
        else:
            # 不能合并，保留为独立源
            all_final_sources.append({
                'merged_id': None,
                'pixel_count': comp2_source['pixel_count'],
                'pixels': comp2_source['pixels'],
                'centroid': comp2_source['centroid'],
                'component_sources': [comp2_source],
                'components': [2],
                'weighted_average': comp2_source['weighted_average'],
                'is_merged': False
            })
            used_sources.add(('comp2', comp2_source['source_id']))
            print(f"\n✗ 没有可合并的源，保留为独立源")
    else:
        print("\nComponent 2 没有源")
    
    # 4. 添加剩余的源（未使用的源）
    temp_sources = []
    
    # 添加Component 1的所有源
    for src in comp1_sources:
        temp_sources.append({
            'merged_id': None,
            'pixel_count': src['pixel_count'],
            'pixels': src['pixels'],
            'centroid': src['centroid'],
            'component_sources': [src],
            'components': [1],
            'weighted_average': src['weighted_average'],
            'is_merged': False
        })
    
    # 添加已经创建的合并源
    for src in all_final_sources:
        temp_sources.append(src)
    
    # 添加Component 2中未使用的源（除了第一个）
    for src in comp2_sources:
        if ('comp2', src['source_id']) not in used_sources:
            temp_sources.append({
                'merged_id': None,
                'pixel_count': src['pixel_count'],
                'pixels': src['pixels'],
                'centroid': src['centroid'],
                'component_sources': [src],
                'components': [2],
                'weighted_average': src['weighted_average'],
                'is_merged': False
            })
    
    # 添加Component 3中未使用的源（除了被合并的）
    for src in comp3_sources:
        if ('comp3', src['source_id']) not in used_sources:
            temp_sources.append({
                'merged_id': None,
                'pixel_count': src['pixel_count'],
                'pixels': src['pixels'],
                'centroid': src['centroid'],
                'component_sources': [src],
                'components': [3],
                'weighted_average': src['weighted_average'],
                'is_merged': False
            })
    
    # 重新分配ID
    for i, source in enumerate(temp_sources):
        source['merged_id'] = i + 1
    
    all_final_sources = temp_sources
    
    print(f"\n{'='*60}")
    print("最终源列表")
    print("="*60)
    print(f"总源数: {len(all_final_sources)}")
    for src in all_final_sources:
        if src.get('is_merged', False):
            print(f"  源 {src['merged_id']}: 合并源 (分量 {src['components']}), "
                  f"{src['pixel_count']} 像素, v={src['weighted_average']['velocity']:.2f} km/s, "
                  f"FWHM={src['weighted_average']['fwhm']:.2f} km/s")
        else:
            print(f"  源 {src['merged_id']}: 分量 {src['components'][0]}, "
                  f"{src['pixel_count']} 像素, v={src['weighted_average']['velocity']:.2f} km/s, "
                  f"FWHM={src['weighted_average']['fwhm']:.2f} km/s")
    
    # 5. 创建参数图（现在返回6个值）
    merged_amplitude, merged_position, merged_dispersion, merged_fwhm, merged_moment0, merged_source_id = \
        create_final_parameter_maps(all_final_sources, results, shape)
    
    # 6. 绘制所有源（使用WCS坐标）- 传入merged_moment0
    if wcs_2d is not None:
        plot_all_sources(all_final_sources, merged_amplitude, merged_position,
                        merged_fwhm, merged_moment0, merged_source_id, output_dir, wcs_2d)
    else:
        print("Warning: No WCS information available, cannot plot with celestial coordinates")
        # 如果没有WCS，可以创建简单的像素坐标图
        print("Creating simple pixel coordinate plot...")
        # 这里可以添加一个简单的像素坐标绘图函数
    
    # 7. 保存源信息
    save_source_info(all_final_sources, output_dir)
    
    # 8. 保存DAT文件
    final_dat = os.path.join(output_dir, 'final_sources.dat')
    save_final_dat(results, all_final_sources, original_dat_file, final_dat)
    
    # 9. 保存合并检查结果
    check_file = os.path.join(output_dir, 'merge_check_results.txt')
    with open(check_file, 'w') as f:
        f.write("="*60 + "\n")
        f.write("合并条件检查结果\n")
        f.write("="*60 + "\n\n")
        
        for check in check_results:
            f.write(f"\n{check['type']}:\n")
            f.write(f"  Component 2 速度: {check['v1']:.4f} km/s\n")
            f.write(f"  Component 2 FWHM: {check['fwhm1']:.4f} km/s\n")
            f.write(f"  Component 3 速度: {check['v2']:.4f} km/s\n")
            f.write(f"  Component 3 FWHM: {check['fwhm2']:.4f} km/s\n")
            f.write(f"  较宽的FWHM: {max(check['fwhm1'], check['fwhm2']):.4f} km/s\n")
            f.write(f"  速度差: {check['dv']:.4f} km/s\n")
            f.write(f"  阈值 (0.7 × 较宽FWHM): {check['threshold']:.4f} km/s\n")
            f.write(f"  是否合并: {check['can_merge']}\n")
    
    print(f"\n合并检查结果保存至: {check_file}")
    
    print("\n" + "="*60)
    print("处理完成")
    print("="*60)
    print(f"\n输出文件保存至: {output_dir}")
    print(f"  - all_sources_overview.pdf")
    print(f"  - all_sources_composition.pdf")
    print(f"  - final_source_information.txt")
    print(f"  - final_sources.dat")
    print(f"  - merge_check_results.txt")


# ==================== 11. 脚本执行 ====================
if __name__ == "__main__":
    # 请修改这些路径
    DAT_FILE = "./n_gauss=3/output_individual_source/source_filtered_rohsa_pixel.dat"
    ORIGINAL_DAT_FILE = "MS_ROHSA_3ngauss_1_3D_1.dat"
    FITS_FILE = "CRAFTS_-4.7_-350_-150_Original.fits"  # 你的FITS文件路径
    OUTPUT_DIR = "./n_gauss=3/output_merged_sources/picture"
    
    # 参数设置
    MIN_PIXELS = 10
    MAX_GAP = 2
    VELOCITY_THRESHOLD_FACTOR = 0.7
    
    # 运行
    main(
        dat_file=DAT_FILE,
        original_dat_file=ORIGINAL_DAT_FILE,
        output_dir=OUTPUT_DIR,
        fits_file=FITS_FILE,  # 传入FITS文件路径
        min_pixels=MIN_PIXELS,
        max_gap=MAX_GAP,
        velocity_threshold_factor=VELOCITY_THRESHOLD_FACTOR
    )


SOURCE MERGING (Component 2 vs Component 3)
Input DAT file: ./n_gauss=3/output_individual_source/source_filtered_rohsa_pixel.dat
Output directory: ./n_gauss=3/output_merged_sources/picture
Min pixels per source: 10
Max spatial gap: 2
Velocity threshold factor: 0.7

Reading WCS info from FITS: CRAFTS_-4.7_-350_-150_Original.fits
SpectralCube shape: (994, 89, 153)
Successfully loaded WCS coordinates

READING FILTERED ROHSA DAT FILE
Reading Gaussian parameters (pixel units)...
Opening data file
Gaussian pixel array shape: (9, 89, 153)
Data type: float64
Converting to physical units...
Spatial dimensions: X=153, Y=89
Number of Gaussian components: 3

Component 1:
  Non-zero pixels (phys): 4872
  Position range (phys): [-255.87, -149.83] km/s

Component 2:
  Non-zero pixels (phys): 291
  Position range (phys): [-282.77, -149.83] km/s

Component 3:
  Non-zero pixels (phys): 56
  Position range (phys): [-297.10, -149.83] km/s

检测到 3 个高斯分量

EXTRACTING SOURCES FROM EACH COMPONENT

Component 1:

### 提取每个源物理信息

In [19]:
import numpy as np
import os
from glob import glob

# 假设core模块已经导入，这里需要确保core模块可用
# import core

def read_individual_source_dat(source_dir, source_id=None):
    """
    读取单个独立源的.dat文件，并返回物理信息
    
    Parameters:
    -----------
    source_dir : str
        源目录的路径（例如：'./source_001'）
    source_id : int, optional
        源ID，用于标识
    
    Returns:
    --------
    dict : 包含源物理信息的字典
    """
    # 查找.dat文件
    dat_files = glob(os.path.join(source_dir, '*.dat'))
    if not dat_files:
        print(f"Warning: No .dat file found in {source_dir}")
        return None
    
    dat_file = dat_files[0]  # 取第一个.dat文件
    
    try:
        # 使用core模块读取像素单位的gaussian参数
        gaussian_pixel = core.read_gaussian(dat_file)
        
        # 转换为物理单位
        gaussian_physical = core.physical_gaussian(gaussian_pixel)
        
        # 解析维度
        n_params_times_comp, n_y, n_x = gaussian_pixel.shape
        n_components = n_params_times_comp // 3
        
        # 提取所有非零像素的物理参数
        all_pixels_info = []
        
        for comp in range(1, n_components + 1):
            # 获取物理单位的参数
            amp = gaussian_physical[3*(comp-1)]
            pos = gaussian_physical[3*(comp-1) + 1]
            sigma = gaussian_physical[3*(comp-1) + 2]
            
            # 找出非零像素
            non_zero_indices = np.where(amp > 0)
            y_indices, x_indices = non_zero_indices
            
            for y, x in zip(y_indices, x_indices):
                pixel_info = {
                    'i': y,  # y坐标 (行)
                    'j': x,  # x坐标 (列)
                    'component': comp,
                    'amplitude': amp[y, x],
                    'mean': pos[y, x],
                    'sigma': sigma[y, x]
                }
                all_pixels_info.append(pixel_info)
        
        # 计算源的统计信息
        if all_pixels_info:
            # 提取所有像素的振幅、均值和sigma
            amplitudes = [p['amplitude'] for p in all_pixels_info]
            means = [p['mean'] for p in all_pixels_info]
            sigmas = [p['sigma'] for p in all_pixels_info]
            
            # 计算加权平均（以振幅为权重）
            total_amp = sum(amplitudes)
            if total_amp > 0:
                weighted_mean = sum(m * a for m, a in zip(means, amplitudes)) / total_amp
                weighted_sigma = sum(s * a for s, a in zip(sigmas, amplitudes)) / total_amp
            else:
                weighted_mean = np.mean(means)
                weighted_sigma = np.mean(sigmas)
            
            # 按分量分组统计
            components_info = {}
            for comp in range(1, n_components + 1):
                comp_pixels = [p for p in all_pixels_info if p['component'] == comp]
                if comp_pixels:
                    comp_amplitudes = [p['amplitude'] for p in comp_pixels]
                    comp_means = [p['mean'] for p in comp_pixels]
                    comp_sigmas = [p['sigma'] for p in comp_pixels]
                    
                    comp_total_amp = sum(comp_amplitudes)
                    if comp_total_amp > 0:
                        comp_weighted_mean = sum(m * a for m, a in zip(comp_means, comp_amplitudes)) / comp_total_amp
                        comp_weighted_sigma = sum(s * a for s, a in zip(comp_sigmas, comp_amplitudes)) / comp_total_amp
                    else:
                        comp_weighted_mean = np.mean(comp_means)
                        comp_weighted_sigma = np.mean(comp_sigmas)
                    
                    components_info[f'component_{comp}'] = {
                        'n_pixels': len(comp_pixels),
                        'mean_velocity': comp_weighted_mean,
                        'mean_sigma': comp_weighted_sigma,
                        'mean_amplitude': np.mean(comp_amplitudes),
                        'total_amplitude': comp_total_amp
                    }
            
            # 保存源信息
            source_info = {
                'source_id': source_id if source_id is not None else os.path.basename(source_dir),
                'source_dir': source_dir,
                'dat_file': dat_file,
                'n_components': n_components,
                'n_pixels': len(all_pixels_info),
                'weighted_mean': weighted_mean,
                'weighted_sigma': weighted_sigma,
                'weighted_fwhm': 2.355 * weighted_sigma,
                'total_amplitude': total_amp,
                'mean_amplitude': np.mean(amplitudes),
                'components': components_info,
                'all_pixels': all_pixels_info
            }
            
            return source_info
        else:
            print(f"Warning: No non-zero pixels found in {source_dir}")
            return None
            
    except Exception as e:
        print(f"Error reading {dat_file}: {e}")
        return None


def read_all_individual_sources(output_dir):
    """
    读取所有独立源的.dat文件，并保存物理信息到txt文件
    
    Parameters:
    -----------
    output_dir : str
        包含所有源目录的根目录（例如：'./output_merged_sources'）
    
    Returns:
    --------
    list : 包含所有源信息的列表
    """
    print("\n" + "="*60)
    print("READING ALL INDIVIDUAL SOURCES")
    print("="*60)
    
    # 查找所有源目录
    source_dirs = sorted(glob(os.path.join(output_dir, 'source_*')))
    source_dirs = [d for d in source_dirs if os.path.isdir(d)]
    
    print(f"Found {len(source_dirs)} source directories")
    
    all_sources_info = []
    
    for source_dir in source_dirs:
        # 从目录名提取源ID
        source_name = os.path.basename(source_dir)
        source_id = source_name.split('_')[1] if '_' in source_name else source_name
        
        print(f"\nReading {source_name}...")
        
        source_info = read_individual_source_dat(source_dir, source_id)
        
        if source_info:
            all_sources_info.append(source_info)
            print(f"  ✓ Loaded: {source_info['n_pixels']} pixels, "
                  f"v={source_info['weighted_mean']:.2f} km/s")
        else:
            print(f"  ✗ Failed to load")
    
    return all_sources_info


def save_physical_info_to_txt(all_sources_info, output_dir, filename='all_sources_physical_info.txt'):
    """
    将所有源的物理信息保存到txt文件（注释行以#开头）
    
    Parameters:
    -----------
    all_sources_info : list
        包含所有源信息的列表
    output_dir : str
        输出目录
    filename : str
        输出文件名
    """
    output_file = os.path.join(output_dir, filename)
    
    with open(output_file, 'w') as f:
        f.write("# " + "="*78 + "\n")
        f.write("# PHYSICAL INFORMATION FOR ALL INDIVIDUAL SOURCES\n")
        f.write("# " + "="*78 + "\n\n")
        
        f.write(f"# Total sources: {len(all_sources_info)}\n")
        f.write(f"# Generated from core.read_gaussian() and core.physical_gaussian()\n")
        f.write(f"# Parameters: Amplitude (K), Mean Velocity (km/s), Sigma (km/s)\n\n")
        
        # 写入每个源的详细信息
        for source in all_sources_info:
            f.write("\n# " + "="*78 + "\n")
            f.write(f"# SOURCE {source['source_id']}\n")
            f.write("# " + "="*78 + "\n")
            f.write(f"# Source directory: {source['source_dir']}\n")
            f.write(f"# Data file: {source['dat_file']}\n")
            f.write(f"# Number of Gaussian components: {source['n_components']}\n")
            f.write(f"# Number of pixels: {source['n_pixels']}\n\n")
            
            f.write("# WEIGHTED AVERAGES (Amplitude-weighted):\n")
            f.write(f"#   Mean Velocity: {source['weighted_mean']:.6f} km/s\n")
            f.write(f"#   Mean Sigma:    {source['weighted_sigma']:.6f} km/s\n")
            f.write(f"#   Mean FWHM:     {source['weighted_fwhm']:.6f} km/s\n")
            f.write(f"#   Total Amplitude: {source['total_amplitude']:.6f} K\n")
            f.write(f"#   Mean Amplitude:  {source['mean_amplitude']:.6f} K\n\n")
            
            f.write("# COMPONENT STATISTICS:\n")
            for comp_name, comp_info in source['components'].items():
                f.write(f"\n#   {comp_name.upper()}:\n")
                f.write(f"#     Number of pixels: {comp_info['n_pixels']}\n")
                f.write(f"#     Mean velocity: {comp_info['mean_velocity']:.6f} km/s\n")
                f.write(f"#     Mean sigma: {comp_info['mean_sigma']:.6f} km/s\n")
                f.write(f"#     Mean amplitude: {comp_info['mean_amplitude']:.6f} K\n")
                f.write(f"#     Total amplitude: {comp_info['total_amplitude']:.6f} K\n")
            
            # 写入所有像素的详细信息
            f.write("\n# " + "-"*78 + "\n")
            f.write("# PIXEL-LEVEL INFORMATION:\n")
            f.write("# " + "-"*78 + "\n")
            f.write(f"# {'i':>6} {'j':>6} {'Comp':>6} {'Amplitude (K)':>15} {'Velocity (km/s)':>18} {'Sigma (km/s)':>15}\n")
            f.write("# " + "-"*78 + "\n")
            
            # 按分量排序像素
            for pixel in sorted(source['all_pixels'], key=lambda x: (x['component'], x['i'], x['j'])):
                f.write(f"{pixel['i']:6d} {pixel['j']:6d} {pixel['component']:6d} "
                       f"{pixel['amplitude']:15.6f} {pixel['mean']:18.6f} {pixel['sigma']:15.6f}\n")
    
    print(f"\nPhysical information saved to: {output_file}")


def save_each_source_physical_info(all_sources_info, output_dir):
    """
    为每个源单独保存物理信息到txt文件（注释行以#开头）
    保存在各自的源目录中
    
    Parameters:
    -----------
    all_sources_info : list
        包含所有源信息的列表
    output_dir : str
        输出目录
    """
    for source in all_sources_info:
        # 获取源目录
        source_dir = source['source_dir']
        info_file = os.path.join(source_dir, f'physical_info.txt')
        
        with open(info_file, 'w') as f:
            f.write("# " + "="*78 + "\n")
            f.write(f"# PHYSICAL INFORMATION FOR SOURCE {source['source_id']}\n")
            f.write("# " + "="*78 + "\n\n")
            
            f.write(f"# Source directory: {source['source_dir']}\n")
            f.write(f"# Data file: {source['dat_file']}\n")
            f.write(f"# Number of Gaussian components: {source['n_components']}\n")
            f.write(f"# Number of pixels: {source['n_pixels']}\n\n")
            
            f.write("# WEIGHTED AVERAGES (Amplitude-weighted):\n")
            f.write(f"#   Mean Velocity: {source['weighted_mean']:.6f} km/s\n")
            f.write(f"#   Mean Sigma:    {source['weighted_sigma']:.6f} km/s\n")
            f.write(f"#   Mean FWHM:     {source['weighted_fwhm']:.6f} km/s\n")
            f.write(f"#   Total Amplitude: {source['total_amplitude']:.6f} K\n")
            f.write(f"#   Mean Amplitude:  {source['mean_amplitude']:.6f} K\n\n")
            
            f.write("# COMPONENT STATISTICS:\n")
            for comp_name, comp_info in source['components'].items():
                f.write(f"\n#   {comp_name.upper()}:\n")
                f.write(f"#     Number of pixels: {comp_info['n_pixels']}\n")
                f.write(f"#     Mean velocity: {comp_info['mean_velocity']:.6f} km/s\n")
                f.write(f"#     Mean sigma: {comp_info['mean_sigma']:.6f} km/s\n")
                f.write(f"#     Mean amplitude: {comp_info['mean_amplitude']:.6f} K\n")
                f.write(f"#     Total amplitude: {comp_info['total_amplitude']:.6f} K\n")
            
            # 写入所有像素的详细信息
            f.write("\n# " + "-"*78 + "\n")
            f.write("# PIXEL-LEVEL INFORMATION:\n")
            f.write("# " + "-"*78 + "\n")
            f.write(f"# {'i':>6} {'j':>6} {'Comp':>6} {'Amplitude (K)':>15} {'Velocity (km/s)':>18} {'Sigma (km/s)':>15}\n")
            f.write("# " + "-"*78 + "\n")
            
            for pixel in sorted(source['all_pixels'], key=lambda x: (x['component'], x['i'], x['j'])):
                f.write(f"{pixel['i']:6d} {pixel['j']:6d} {pixel['component']:6d} "
                       f"{pixel['amplitude']:15.6f} {pixel['mean']:18.6f} {pixel['sigma']:15.6f}\n")
        
        print(f"  Physical info for source {source['source_id']} saved to: {info_file}")


def create_summary_statistics(all_sources_info, output_dir, filename='sources_summary_statistics.txt'):
    """
    创建所有源的汇总统计信息（注释行以#开头）
    
    Parameters:
    -----------
    all_sources_info : list
        包含所有源信息的列表
    output_dir : str
        输出目录
    filename : str
        输出文件名
    """
    output_file = os.path.join(output_dir, filename)
    
    with open(output_file, 'w') as f:
        f.write("# " + "="*78 + "\n")
        f.write("# SUMMARY STATISTICS FOR ALL SOURCES\n")
        f.write("# " + "="*78 + "\n\n")
        
        # 提取所有源的统计信息
        source_ids = []
        n_pixels_list = []
        velocities = []
        sigmas = []
        fwhms = []
        amplitudes = []
        
        for source in all_sources_info:
            source_ids.append(source['source_id'])
            n_pixels_list.append(source['n_pixels'])
            velocities.append(source['weighted_mean'])
            sigmas.append(source['weighted_sigma'])
            fwhms.append(source['weighted_fwhm'])
            amplitudes.append(source['mean_amplitude'])
        
        # 写入表格头
        f.write(f"# {'Source ID':>10} {'Pixels':>8} {'Velocity (km/s)':>18} {'Sigma (km/s)':>15} "
               f"{'FWHM (km/s)':>15} {'Amplitude (K)':>15}\n")
        f.write("# " + "-"*78 + "\n")
        
        # 写入数据（数据行不加#）
        for i, source_id in enumerate(source_ids):
            f.write(f"{source_id:>10} {n_pixels_list[i]:8d} {velocities[i]:18.6f} "
                   f"{sigmas[i]:15.6f} {fwhms[i]:15.6f} {amplitudes[i]:15.6f}\n")
        
        # 写入统计信息（统计信息加#）
        f.write("\n# " + "="*78 + "\n")
        f.write("# STATISTICS ACROSS ALL SOURCES\n")
        f.write("# " + "="*78 + "\n\n")
        
        f.write(f"# Number of sources: {len(all_sources_info)}\n")
        f.write(f"# Total pixels: {sum(n_pixels_list)}\n")
        f.write(f"# Average pixels per source: {np.mean(n_pixels_list):.2f}\n\n")
        
        f.write("# Velocity Statistics (km/s):\n")
        f.write(f"#   Mean: {np.mean(velocities):.6f}\n")
        f.write(f"#   Std:  {np.std(velocities):.6f}\n")
        f.write(f"#   Min:  {np.min(velocities):.6f}\n")
        f.write(f"#   Max:  {np.max(velocities):.6f}\n\n")
        
        f.write("# Sigma Statistics (km/s):\n")
        f.write(f"#   Mean: {np.mean(sigmas):.6f}\n")
        f.write(f"#   Std:  {np.std(sigmas):.6f}\n")
        f.write(f"#   Min:  {np.min(sigmas):.6f}\n")
        f.write(f"#   Max:  {np.max(sigmas):.6f}\n\n")
        
        f.write("# FWHM Statistics (km/s):\n")
        f.write(f"#   Mean: {np.mean(fwhms):.6f}\n")
        f.write(f"#   Std:  {np.std(fwhms):.6f}\n")
        f.write(f"#   Min:  {np.min(fwhms):.6f}\n")
        f.write(f"#   Max:  {np.max(fwhms):.6f}\n\n")
        
        f.write("# Amplitude Statistics (K):\n")
        f.write(f"#   Mean: {np.mean(amplitudes):.6f}\n")
        f.write(f"#   Std:  {np.std(amplitudes):.6f}\n")
        f.write(f"#   Min:  {np.min(amplitudes):.6f}\n")
        f.write(f"#   Max:  {np.max(amplitudes):.6f}\n")
    
    print(f"\nSummary statistics saved to: {output_file}")


# ==================== 主执行函数 ====================
def extract_physical_info_from_sources(output_dir):
    """
    主函数：从所有独立源的.dat文件中提取物理信息并保存
    
    Parameters:
    -----------
    output_dir : str
        包含所有源目录的根目录（例如：'./n_gauss=4/table'）
    """
    print("\n" + "="*80)
    print("EXTRACTING PHYSICAL INFORMATION FROM INDIVIDUAL SOURCES")
    print("="*80)
    print(f"Input directory: {output_dir}")
    
    # 1. 读取所有独立源的.dat文件
    all_sources_info = read_all_individual_sources(output_dir)
    
    if not all_sources_info:
        print("\nNo sources found to process!")
        return
    
    # 2. 保存所有源的物理信息到一个总文件
    save_physical_info_to_txt(all_sources_info, output_dir, 'all_sources_physical_info.txt')
    
    # 3. 为每个源单独保存物理信息
    print("\n" + "="*60)
    print("SAVING INDIVIDUAL SOURCE PHYSICAL INFO")
    print("="*60)
    save_each_source_physical_info(all_sources_info, output_dir)
    
    # 4. 创建汇总统计信息
    create_summary_statistics(all_sources_info, output_dir, 'sources_summary_statistics.txt')
    
    # 5. 打印总结
    print("\n" + "="*80)
    print("EXTRACTION COMPLETED")
    print("="*80)
    print(f"\nProcessed {len(all_sources_info)} sources")
    print(f"\nOutput files in {output_dir}:")
    print(f"  - all_sources_physical_info.txt (所有源的详细物理信息)")
    print(f"  - sources_summary_statistics.txt (所有源的汇总统计)")
    print(f"\nEach source directory now contains:")
    print(f"  - physical_info.txt (该源的详细物理信息)")


# ==================== 脚本执行 ====================
if __name__ == "__main__":
    # 请修改这个路径为你的输出目录
    OUTPUT_DIR = "./n_gauss=3/output_individual_source_table/individual_sources"
    
    # 运行提取
    extract_physical_info_from_sources(OUTPUT_DIR)


EXTRACTING PHYSICAL INFORMATION FROM INDIVIDUAL SOURCES
Input directory: ./n_gauss=3/output_individual_source_table/individual_sources

READING ALL INDIVIDUAL SOURCES
Found 9 source directories

Reading source_001...
Opening data file
  ✓ Loaded: 4660 pixels, v=-238.08 km/s

Reading source_002...
Opening data file
  ✓ Loaded: 106 pixels, v=-236.06 km/s

Reading source_003...
Opening data file
  ✓ Loaded: 13 pixels, v=-246.06 km/s

Reading source_004...
Opening data file
  ✓ Loaded: 75 pixels, v=-243.51 km/s

Reading source_005...
Opening data file
  ✓ Loaded: 18 pixels, v=-232.74 km/s

Reading source_006...
Opening data file
  ✓ Loaded: 291 pixels, v=-276.85 km/s

Reading source_007...
Opening data file
  ✓ Loaded: 10 pixels, v=-289.49 km/s

Reading source_008...
Opening data file
  ✓ Loaded: 33 pixels, v=-294.92 km/s

Reading source_009...
Opening data file
  ✓ Loaded: 13 pixels, v=-292.44 km/s

Physical information saved to: ./n_gauss=3/output_individual_source_table/individual_sour

### 数据处理

In [1]:
import pandas as pd
import os

def read_pixel_data(filepath):
    """
    读取像素级数据，跳过注释行
    返回 DataFrame
    """
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"文件不存在: {filepath}")
    
    # 读取数据，跳过以 # 开头的行
    df = pd.read_csv(filepath, 
                     comment='#',
                     delim_whitespace=True,
                     names=['i', 'j', 'Comp', 'Amplitude_K', 'Velocity_kms', 'Sigma_kms'])
    return df

# ========== 使用示例 ==========
x = 9

# 1. 读取原始数据
filepath = f'./n_gauss=3/output_individual_source_table/individual_sources/source_00{x}/physical_info.txt'  # 改成你的文件路径
df = read_pixel_data(filepath)

print("前5行数据:")
print(df.head())
print(f"\n总行数: {len(df)}")

# 2. 保存为 CSV 文件（方便后续调用）
df.to_csv(f'./n_gauss=3/output_individual_source_table/individual_sources/source_00{x}/pixel_data.csv', index=False)
print("\n已保存为 pixel_data.csv")

前5行数据:
    i   j  Comp  Amplitude_K  Velocity_kms  Sigma_kms
0  78  15     3     0.250337   -293.317009   8.904336
1  78  16     3     0.267240   -292.327069   8.774870
2  78  17     3     0.267558   -291.858393   8.339511
3  78  18     3     0.244007   -292.134986   7.828716
4  79  14     3     0.276257   -293.853572   9.593364

总行数: 13

已保存为 pixel_data.csv


/tmp/ipykernel_938/3536512314.py:13: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv(filepath,


In [3]:
import numpy as np
# 读取数据
df = pd.read_csv('./baseline/SNR=2/output_individual_source_10/output_merged_source_0.7/source_physical_parameters/source_001_physical_parameters.csv')

# 计算 Moment-0（理论公式）
moment0 = df['amplitude_K'] * df['sigma_kms'] * np.sqrt(2 * np.pi)

amplitudes = df['amplitude_K']
velocities = df['velocity_kms']
sigmas = df['sigma_kms']

# 添加到 DataFrame
df['Moment0'] = moment0

print(df.head())
print(f"\nMoment-0 范围: {moment0.min():.4f} ~ {moment0.max():.4f} K·km/s")

   final_source_id  component  original_component_source_id  x_pixel  y_pixel  \
0                1          1                             1       21        0   
1                1          1                             1       22        0   
2                1          1                             1       23        0   
3                1          1                             1       24        0   
4                1          1                             1       25        0   

     ra_deg  dec_deg  amplitude_pixel  position_pixel  dispersion_pixel  \
0  356.2625  -6.8875         0.210591      452.661999         65.726623   
1  356.2375  -6.8875         0.231671      456.077413         66.701504   
2  356.2125  -6.8875         0.245385      457.797127         67.834167   
3  356.1875  -6.8875         0.276904      458.011429         70.099264   
4  356.1625  -6.8875         0.325977      455.375251         70.709615   

   amplitude_K  velocity_kms  sigma_kms   fwhm_kms  moment0_K_

In [5]:
import numpy as np
import pandas as pd
from radio_beam import Beam
from astropy import units as u

# ========== 1. 读取数据并计算 Moment-0 ==========
df = pd.read_csv('./baseline/SNR=2/output_individual_source_10/output_merged_source_0.7/source_physical_parameters/source_001_physical_parameters.csv')

# 计算 Moment-0（理论公式）
df['Moment0'] = df['amplitude_K'] * df['sigma_kms'] * np.sqrt(2 * np.pi)

print("Moment-0 统计:")
print(f"  总和: {df['Moment0'].sum():.4f} K·km/s")
print(f"  范围: {df['Moment0'].min():.4f} ~ {df['Moment0'].max():.4f} K·km/s")

# ========== 2. 创建 2D Moment-0 图 ==========
ny = df['y_pixel'].max().astype(int) + 1
nx = df['x_pixel'].max().astype(int) + 1

moment0_map = np.zeros((ny, nx))
for _, row in df.iterrows():
    i, j = int(row['y_pixel']), int(row['x_pixel'])
    moment0_map[i, j] = row['Moment0']

print(f"\nMoment-0 图尺寸: {moment0_map.shape} (y, x)")

# ========== 3. 定义波束并计算转换因子 ==========
my_beam = Beam(240 * u.arcsec)  # 波束大小 240 角秒
print(f"\n波束信息: {my_beam}")

# 亮温度到流量密度的转换 (1 K -> Jy/beam at 1.42 GHz)
K_to_Jy = (1 * u.K).to(u.Jy / u.beam, my_beam.jtok_equiv(1.42 * u.GHz))
print(f"转换因子: 1 K = {K_to_Jy:.4f} Jy/beam")

# ========== 4. 计算流量 (flux) ==========
# flux 单位: Jy·km/s / beam
# moment0_map 单位: K·km/s
flux = moment0_map * K_to_Jy / u.K

print(f"\n流量统计:")
print(f"  流量总和: {np.nansum(flux):.4f} Jy·km/s/beam")
print(f"  流量范围: {np.nanmin(flux):.4f} ~ {np.nanmax(flux):.4f} Jy·km/s/beam")

# ========== 5. 计算平均谱线强度 ==========
# 对所有像素取平均
spectrum = np.mean(np.nan_to_num(flux))
print(f"\n平均谱线强度: {spectrum:.4f} Jy·km/s/beam")

# ========== 6. 波束面积计算 ==========


pixel=flux/((np.pi*240 *240/90**2)/(4*np.log(2)))
distance_mpc = 0.0501  #(+0.0014,-0.0012)  # Mpc
M_HI_your_method = np.nansum(pixel)*u.beam*2.356e5*distance_mpc**2/(u.Jy*u.km/u.s)*u.solMass *u.K
# print("Results:",a)

print(f"\nHI 质量（你的方法）: {M_HI_your_method:.2e} M_sun")

Moment-0 统计:
  总和: 98187.9708 K·km/s
  范围: 5.1169 ~ 46.4947 K·km/s

Moment-0 图尺寸: (88, 108) (y, x)

波束信息: 1.534039835497245e-06 sr
转换因子: 1 K = 0.0950 Jy / beam Jy/beam

流量统计:
  流量总和: 9331.3286 Jy / (K beam) Jy·km/s/beam
  流量范围: 0.0000 Jy / (K beam) ~ 4.4186 Jy / (K beam) Jy·km/s/beam

平均谱线强度: 0.9818 Jy / (K beam) Jy·km/s/beam

HI 质量（你的方法）: 6.85e+05 s solMass / km M_sun


In [7]:
import numpy as np
import pandas as pd
from radio_beam import Beam
from astropy import units as u
import os

# ===================== 基础参数 =====================
base_dir = "./baseline/SNR=2/output_individual_source_10/output_merged_source_0.7/source_physical_parameters"

# source_001 到 source_010
source_list = [f"source_{i:03d}" for i in range(1, 11)]
beam_size = 240 * u.arcsec
freq = 1.42 * u.GHz

# 距离
d_center = 0.0501
d_plus = 0.0014
d_minus = 0.0012

# 像素尺度：1 像素 = 0.0247 度
pixel_scale_deg = 0.0247  # deg/pixel
pixel_scale_arcsec = pixel_scale_deg * 3600  # arcsec/pixel

# 波束面积
beam_area = (np.pi * 240**2) / (4 * np.log(2)) / 90**2  # pixel/beam

# ===================== 定义椭圆拟合函数 =====================
def fit_ellipse(moment0_map, threshold=None):
    """
    对 Moment-0 图进行椭圆拟合，计算长轴、短轴和位置角
    
    Parameters:
    -----------
    moment0_map : 2D numpy array
        Moment-0 图
    threshold : float, optional
        阈值，低于此值的像素忽略。如果为 None，则使用最大值的 20%
    
    Returns:
    --------
    major_axis_deg : float
        长轴 (度)
    minor_axis_deg : float
        短轴 (度)
    pa : float
        位置角 (度，-90° 到 90°，0° 指向正北)
        正值为东偏北，负值为西偏北
    center_y, center_x : float
        中心坐标 (像素)
    """
    # 设置阈值
    if threshold is None:
        threshold = moment0_map.max() * 0
    
    # 创建掩码，只保留高于阈值的像素
    mask = moment0_map > threshold
    
    # 获取有效像素的坐标和权重
    y_idx, x_idx = np.where(mask)
    weights = moment0_map[mask]
    
    if len(y_idx) < 5:
        return np.nan, np.nan, np.nan, np.nan, np.nan
    
    # 计算加权质心
    total_weight = np.sum(weights)
    center_x = np.sum(x_idx * weights) / total_weight
    center_y = np.sum(y_idx * weights) / total_weight
    
    # 计算二阶矩
    dx = x_idx - center_x
    dy = y_idx - center_y
    
    # 加权协方差矩阵
    Ixx = np.sum(weights * dx * dx) / total_weight
    Iyy = np.sum(weights * dy * dy) / total_weight
    Ixy = np.sum(weights * dx * dy) / total_weight
    
    # 计算特征值（主轴长度）
    trace = Ixx + Iyy
    discriminant = np.sqrt((Ixx - Iyy)**2 + 4 * Ixy**2)
    
    lambda1 = (trace + discriminant) / 2  # 长轴方向方差
    lambda2 = (trace - discriminant) / 2  # 短轴方向方差
    
    # 半径（2.355 σ 转换为 FWHM）
    r_major = 2.355 * np.sqrt(lambda1)  # 像素单位
    r_minor = 2.355 * np.sqrt(lambda2)  # 像素单位
    
    # 计算主轴方向角（从 x 轴正方向逆时针测量）
    # 使用 arctan2 得到正确的象限
    if Ixx != Iyy:
        theta = 0.5 * np.arctan2(2 * Ixy, Ixx - Iyy)
    else:
        if Ixy > 0:
            theta = np.pi / 4
        elif Ixy < 0:
            theta = -np.pi / 4
        else:
            theta = 0
    
    # ========== 位置角转换（天文学定义）==========
    # 天文学位置角 (PA): 从北方向顺时针（向东）测量到长轴
    # 但在图像坐标系中：
    #   - x 轴指向东 (East)
    #   - y 轴指向北 (North)
    #   - theta 是从 x 轴（东）逆时针测量的角度
    # 
    # 转换公式：
    #   从北顺时针测量 = 90° - theta
    #   然后归一化到 [-90°, 90°] 范围
    #   正值表示东偏北，负值表示西偏北
    
    # 计算天文学 PA（从北顺时针）
    pa_astro = 90 - np.degrees(theta)
    
    # 归一化到 [-90°, 90°] 范围
    # 如果 PA > 90°，减去 180° 使其进入 [-90°, 90°]
    # 如果 PA < -90°，加上 180°
    if pa_astro > 90:
        pa_astro = pa_astro - 180
    elif pa_astro < -90:
        pa_astro = pa_astro + 180
    
    # 转换为度单位
    major_axis_deg = r_major * pixel_scale_deg
    minor_axis_deg = r_minor * pixel_scale_deg
    
    return major_axis_deg, minor_axis_deg, pa_astro, center_y, center_x

# ===================== 结果列表 =====================
results = []

# ===================== 循环处理每个源 =====================
# ===================== 循环处理每个源 =====================
for source in source_list:
    print(f"\n{'='*60}")
    print(f"处理：{source}")
    print(f"{'='*60}")

    csv_path = os.path.join(base_dir, f"{source}_physical_parameters.csv")

    if not os.path.exists(csv_path):
        print(f"⚠️ 文件不存在，跳过: {csv_path}")
        continue

    df = pd.read_csv(csv_path)

    # ------------------- 1. Moment-0 -------------------
    df['Moment0'] = df['amplitude_K'] * df['sigma_kms'] * np.sqrt(2 * np.pi)
    total_m0 = df['Moment0'].sum()

    # ------------------- 2. 创建 Moment-0 图 -------------------
    ny = df['y_pixel'].max().astype(int) + 1
    nx = df['x_pixel'].max().astype(int) + 1
    m0_map = np.zeros((ny, nx))
    for _, row in df.iterrows():
        i, j = int(row['y_pixel']), int(row['x_pixel'])
        m0_map[i, j] = row['Moment0']

    # ------------------- 3. 流量计算 -------------------
    beam = Beam(beam_size)
    K_to_Jy = (1*u.K).to(u.Jy/u.beam, beam.jtok_equiv(freq))
    flux_map = m0_map * K_to_Jy / u.K
    flux_pix = flux_map / beam_area
    total_flux = np.nansum(flux_pix)

    # ------------------- 4. HI 质量 -------------------
    def mass(d):
        return total_flux * 2.356e5 * d**2 * u.solMass
    M0 = mass(d_center)
    Mp = mass(d_center + d_plus)
    Mm = mass(d_center - d_minus)

    # ------------------- 5. 柱密度 N_HI [cm^-2] -------------------
    df['N_HI'] = 1.823e18 * df['Moment0']
    mean_NHI = df['N_HI'].mean()
    max_NHI = df['N_HI'].max()
    total_NHI = df['N_HI'].sum()

    # ------------------- 6. 椭圆拟合（长轴、短轴、位置角）-------------------
    major_axis_deg, minor_axis_deg, pa, center_y, center_x = fit_ellipse(m0_map)
    
    # 计算等效半径
    if not np.isnan(major_axis_deg):
        req_deg = np.sqrt(major_axis_deg * minor_axis_deg)
    else:
        req_deg = np.nan
    
    # 转换为弧秒（便于参考）
    major_axis_arcsec = major_axis_deg * 3600 if not np.isnan(major_axis_deg) else np.nan
    minor_axis_arcsec = minor_axis_deg * 3600 if not np.isnan(minor_axis_deg) else np.nan
    req_arcsec = req_deg * 3600 if not np.isnan(req_deg) else np.nan

    # ------------------- 打印结果 -------------------
    print(f"Moment-0 总和: {total_m0:.2f} K·km/s")
    print(f"总流量: {total_flux:.4f} Jy·km/s")
    print(f"HI 质量: {M0:.2e} M☉")
    print(f"平均柱密度: {mean_NHI:.2e} cm⁻²")
    print(f"最大柱密度: {max_NHI:.2e} cm⁻²")
    print(f"\n椭圆拟合结果 (1 pixel = {pixel_scale_deg:.4f} deg):")
    print(f"  长轴: {major_axis_deg:.4f} deg ({major_axis_arcsec:.1f} arcsec)")
    print(f"  短轴: {minor_axis_deg:.4f} deg ({minor_axis_arcsec:.1f} arcsec)")
    print(f"  等效半径: {req_deg:.4f} deg ({req_arcsec:.1f} arcsec)")
    print(f"  位置角 (PA): {pa:.1f}° (范围: -90° ~ 90°, 0°=正北)")
    print(f"    正值: 东偏北 (顺时针)")
    print(f"    负值: 西偏北 (逆时针)")
    print(f"  中心: (x={center_x:.1f}, y={center_y:.1f}) pixels")

    # ------------------- 存入结果 -------------------
    results.append([
        source,
        total_m0,
        total_flux.value,
        M0.value, Mp.value, Mm.value,
        mean_NHI, max_NHI, total_NHI,
        major_axis_deg, minor_axis_deg, req_deg,
        major_axis_arcsec, minor_axis_arcsec, req_arcsec,
        pa, center_x, center_y
    ])

# ===================== 保存到 TXT =====================
out_path = os.path.join(base_dir, "HI_mass_results.txt")
with open(out_path, 'w', encoding='utf-8') as f:
    f.write("="*220 + "\n")
    f.write("HI 质量及形态参数计算结果 (1 pixel = 0.0247 deg)\n")
    f.write("位置角 PA 范围: -90° ~ 90°, 0° = 正北\n")
    f.write("  正值: 东偏北 (顺时针)\n")
    f.write("  负值: 西偏北 (逆时针)\n")
    f.write("="*220 + "\n\n")
    
    f.write(
        f"{'Source':<10} {'M0':<12} {'Flux':<12} {'MHI':<14} "
        f"{'MHI_up':<14} {'MHI_low':<14} "
        f"{'<NHI>':<16} {'maxNHI':<16} {'totalNHI':<16} "
        f"{'Maj(deg)':<12} {'Min(deg)':<12} {'Req(deg)':<12} "
        f"{'Maj(arcsec)':<12} {'Min(arcsec)':<12} {'Req(arcsec)':<12} "
        f"{'PA(deg)':<10} {'X_center':<10} {'Y_center':<10}\n")
    f.write("-"*220 + "\n")

    for row in results:
        (src, m0, flux, m0v, mpv, mmv,
         mnhi, mxnhi, tnhi,
         maj_deg, min_deg, req_deg,
         maj_asec, min_asec, req_asec,
         pa, cx, cy) = row

        f.write(
            f"{src:<10} {m0:<12.2f} {flux:<12.4f} {m0v:<14.2e} "
            f"{mpv:<14.2e} {mmv:<14.2e} "
            f"{mnhi:<16.2e} {mxnhi:<16.2e} {tnhi:<16.2e} "
            f"{maj_deg:<12.4f} {min_deg:<12.4f} {req_deg:<12.4f} "
            f"{maj_asec:<12.1f} {min_asec:<12.1f} {req_asec:<12.1f} "
            f"{pa:<10.1f} {cx:<10.1f} {cy:<10.1f}\n")

print(f"\n✅ 全部完成！结果已保存到：\n{out_path}")

# ===================== 打印汇总 =====================
print("\n" + "="*100)
print("结果汇总")
print("="*100)
print(f"{'Source':<12} {'MHI (M☉)':<16} {'Major (deg)':<12} {'Minor (deg)':<12} {'PA (deg)':<12}")
print("-"*100)
for row in results:
    print(f"{row[0]:<12} {row[3]:<16.2e} {row[9]:<12.4f} {row[10]:<12.4f} {row[15]:<12.1f}")
print("="*100)

# ===================== PA 说明 =====================
print("\n" + "="*100)
print("位置角 (PA) 说明:")
print("="*100)
print("  PA = 0°   : 长轴指向正北")
print("  PA = 45°  : 长轴指向东北方向 (东偏北45°)")
print("  PA = 90°  : 长轴指向正东")
print("  PA = -45° : 长轴指向西北方向 (西偏北45°)")
print("  PA = -90° : 长轴指向正西")
print("="*100)


处理：source_001
Moment-0 总和: 98187.97 K·km/s
总流量: 1158.0881 Jy / (K beam) Jy·km/s
HI 质量: 6.85e+05 Jy solMass / (K beam) M☉
平均柱密度: 3.41e+19 cm⁻²
最大柱密度: 8.48e+19 cm⁻²

椭圆拟合结果 (1 pixel = 0.0247 deg):
  长轴: 1.5183 deg (5466.0 arcsec)
  短轴: 1.3265 deg (4775.3 arcsec)
  等效半径: 1.4192 deg (5109.0 arcsec)
  位置角 (PA): -11.2° (范围: -90° ~ 90°, 0°=正北)
    正值: 东偏北 (顺时针)
    负值: 西偏北 (逆时针)
  中心: (x=51.0, y=38.0) pixels

处理：source_002
Moment-0 总和: 2080.75 K·km/s
总流量: 24.5417 Jy / (K beam) Jy·km/s
HI 质量: 1.45e+04 Jy solMass / (K beam) M☉
平均柱密度: 2.65e+19 cm⁻²
最大柱密度: 3.56e+19 cm⁻²

椭圆拟合结果 (1 pixel = 0.0247 deg):
  长轴: 0.2580 deg (928.8 arcsec)
  短轴: 0.1497 deg (539.0 arcsec)
  等效半径: 0.1965 deg (707.6 arcsec)
  位置角 (PA): 37.5° (范围: -90° ~ 90°, 0°=正北)
    正值: 东偏北 (顺时针)
    负值: 西偏北 (逆时针)
  中心: (x=117.0, y=6.7) pixels

处理：source_003
Moment-0 总和: 204.03 K·km/s
总流量: 2.4064 Jy / (K beam) Jy·km/s
HI 质量: 1.42e+03 Jy solMass / (K beam) M☉
平均柱密度: 2.19e+19 cm⁻²
最大柱密度: 2.44e+19 cm⁻²

椭圆拟合结果 (1 pixel = 0.0247 deg):
  长轴

In [ ]:
./n_gauss=3/output_individual_source_table/individual_sources

In [36]:
import numpy as np
import pandas as pd
from radio_beam import Beam
from astropy import units as u
import os

# ===================== 基础参数 =====================
base_dir = "./n_gauss=3/output_individual_source_table/individual_sources"
source_list = [f"source_{i:03d}" for i in range(1, 10)]
beam_size = 240 * u.arcsec
freq = 1.42 * u.GHz

# 距离
d_center = 0.0501
d_plus = 0.0014
d_minus = 0.0012

# 像素尺度：1 像素 = 0.0247 度
pixel_scale_deg = 0.0247  # deg/pixel
pixel_scale_arcsec = pixel_scale_deg * 3600  # arcsec/pixel

# 波束面积
beam_area = (np.pi * 240**2) / (4 * np.log(2)) / 90**2  # pixel/beam

# ===================== 定义椭圆拟合函数 =====================
def fit_ellipse(moment0_map, threshold=None):
    """
    对 Moment-0 图进行椭圆拟合，计算长轴、短轴和位置角
    
    Parameters:
    -----------
    moment0_map : 2D numpy array
        Moment-0 图
    threshold : float, optional
        阈值，低于此值的像素忽略。如果为 None，则使用最大值的 20%
    
    Returns:
    --------
    major_axis_deg : float
        长轴 (度)
    minor_axis_deg : float
        短轴 (度)
    pa : float
        位置角 (度，-90° 到 90°，0° 指向正北)
        正值为东偏北，负值为西偏北
    center_y, center_x : float
        中心坐标 (像素)
    """
    # 设置阈值
    if threshold is None:
        threshold = moment0_map.max() * 0
    
    # 创建掩码，只保留高于阈值的像素
    mask = moment0_map > threshold
    
    # 获取有效像素的坐标和权重
    y_idx, x_idx = np.where(mask)
    weights = moment0_map[mask]
    
    if len(y_idx) < 5:
        return np.nan, np.nan, np.nan, np.nan, np.nan
    
    # 计算加权质心
    total_weight = np.sum(weights)
    center_x = np.sum(x_idx * weights) / total_weight
    center_y = np.sum(y_idx * weights) / total_weight
    
    # 计算二阶矩
    dx = x_idx - center_x
    dy = y_idx - center_y
    
    # 加权协方差矩阵
    Ixx = np.sum(weights * dx * dx) / total_weight
    Iyy = np.sum(weights * dy * dy) / total_weight
    Ixy = np.sum(weights * dx * dy) / total_weight
    
    # 计算特征值（主轴长度）
    trace = Ixx + Iyy
    discriminant = np.sqrt((Ixx - Iyy)**2 + 4 * Ixy**2)
    
    lambda1 = (trace + discriminant) / 2  # 长轴方向方差
    lambda2 = (trace - discriminant) / 2  # 短轴方向方差
    
    # 半径（2.355 σ 转换为 FWHM）
    r_major = 2.355 * np.sqrt(lambda1)  # 像素单位
    r_minor = 2.355 * np.sqrt(lambda2)  # 像素单位
    
    # 计算主轴方向角（从 x 轴正方向逆时针测量）
    # 使用 arctan2 得到正确的象限
    if Ixx != Iyy:
        theta = 0.5 * np.arctan2(2 * Ixy, Ixx - Iyy)
    else:
        if Ixy > 0:
            theta = np.pi / 4
        elif Ixy < 0:
            theta = -np.pi / 4
        else:
            theta = 0
    
    # ========== 位置角转换（天文学定义）==========
    # 天文学位置角 (PA): 从北方向顺时针（向东）测量到长轴
    # 但在图像坐标系中：
    #   - x 轴指向东 (East)
    #   - y 轴指向北 (North)
    #   - theta 是从 x 轴（东）逆时针测量的角度
    # 
    # 转换公式：
    #   从北顺时针测量 = 90° - theta
    #   然后归一化到 [-90°, 90°] 范围
    #   正值表示东偏北，负值表示西偏北
    
    # 计算天文学 PA（从北顺时针）
    pa_astro = 90 - np.degrees(theta)
    
    # 归一化到 [-90°, 90°] 范围
    # 如果 PA > 90°，减去 180° 使其进入 [-90°, 90°]
    # 如果 PA < -90°，加上 180°
    if pa_astro > 90:
        pa_astro = pa_astro - 180
    elif pa_astro < -90:
        pa_astro = pa_astro + 180
    
    # 转换为度单位
    major_axis_deg = r_major * pixel_scale_deg
    minor_axis_deg = r_minor * pixel_scale_deg
    
    return major_axis_deg, minor_axis_deg, pa_astro, center_y, center_x

# ===================== 结果列表 =====================
results = []

# ===================== 循环处理每个源 =====================
for source in source_list:
    print(f"\n{'='*60}")
    print(f"处理：{source}")
    print(f"{'='*60}")
    
    csv_path = os.path.join(base_dir, source, "pixel_data.csv")
    df = pd.read_csv(csv_path)

    # ------------------- 1. Moment-0 -------------------
    df['Moment0'] = df['Amplitude_K'] * df['Sigma_kms'] * np.sqrt(2 * np.pi)
    total_m0 = df['Moment0'].sum()

    # ------------------- 2. 创建 Moment-0 图 -------------------
    ny = df['i'].max().astype(int) + 1
    nx = df['j'].max().astype(int) + 1
    m0_map = np.zeros((ny, nx))
    for _, row in df.iterrows():
        i, j = int(row['i']), int(row['j'])
        m0_map[i, j] = row['Moment0']

    # ------------------- 3. 流量计算 -------------------
    beam = Beam(beam_size)
    K_to_Jy = (1*u.K).to(u.Jy/u.beam, beam.jtok_equiv(freq))
    flux_map = m0_map * K_to_Jy / u.K
    flux_pix = flux_map / beam_area
    total_flux = np.nansum(flux_pix)

    # ------------------- 4. HI 质量 -------------------
    def mass(d):
        return total_flux * 2.356e5 * d**2 * u.solMass
    M0 = mass(d_center)
    Mp = mass(d_center + d_plus)
    Mm = mass(d_center - d_minus)

    # ------------------- 5. 柱密度 N_HI [cm^-2] -------------------
    df['N_HI'] = 1.823e18 * df['Moment0']
    mean_NHI = df['N_HI'].mean()
    max_NHI = df['N_HI'].max()
    total_NHI = df['N_HI'].sum()

    # ------------------- 6. 椭圆拟合（长轴、短轴、位置角）-------------------
    major_axis_deg, minor_axis_deg, pa, center_y, center_x = fit_ellipse(m0_map)
    
    # 计算等效半径
    if not np.isnan(major_axis_deg):
        req_deg = np.sqrt(major_axis_deg * minor_axis_deg)
    else:
        req_deg = np.nan
    
    # 转换为弧秒（便于参考）
    major_axis_arcsec = major_axis_deg * 3600 if not np.isnan(major_axis_deg) else np.nan
    minor_axis_arcsec = minor_axis_deg * 3600 if not np.isnan(minor_axis_deg) else np.nan
    req_arcsec = req_deg * 3600 if not np.isnan(req_deg) else np.nan

    # ------------------- 打印结果 -------------------
    print(f"Moment-0 总和: {total_m0:.2f} K·km/s")
    print(f"总流量: {total_flux:.4f} Jy·km/s")
    print(f"HI 质量: {M0:.2e} M☉")
    print(f"平均柱密度: {mean_NHI:.2e} cm⁻²")
    print(f"最大柱密度: {max_NHI:.2e} cm⁻²")
    print(f"\n椭圆拟合结果 (1 pixel = {pixel_scale_deg:.4f} deg):")
    print(f"  长轴: {major_axis_deg:.4f} deg ({major_axis_arcsec:.1f} arcsec)")
    print(f"  短轴: {minor_axis_deg:.4f} deg ({minor_axis_arcsec:.1f} arcsec)")
    print(f"  等效半径: {req_deg:.4f} deg ({req_arcsec:.1f} arcsec)")
    print(f"  位置角 (PA): {pa:.1f}° (范围: -90° ~ 90°, 0°=正北)")
    print(f"    正值: 东偏北 (顺时针)")
    print(f"    负值: 西偏北 (逆时针)")
    print(f"  中心: (x={center_x:.1f}, y={center_y:.1f}) pixels")

    # ------------------- 存入结果 -------------------
    results.append([
        source,
        total_m0,
        total_flux.value,
        M0.value, Mp.value, Mm.value,
        mean_NHI, max_NHI, total_NHI,
        major_axis_deg, minor_axis_deg, req_deg,
        major_axis_arcsec, minor_axis_arcsec, req_arcsec,
        pa, center_x, center_y
    ])

# ===================== 保存到 TXT =====================
# ✅ 关键修改：全部输出完整小数，不使用科学计数法
out_path = os.path.join(base_dir, "HI_mass_results.txt")
with open(out_path, 'w', encoding='utf-8') as f:
    f.write("="*220 + "\n")
    f.write("HI 质量及形态参数计算结果 (1 pixel = 0.0247 deg)\n")
    f.write("位置角 PA 范围: -90° ~ 90°, 0° = 正北\n")
    f.write("  正值: 东偏北 (顺时针)\n")
    f.write("  负值: 西偏北 (逆时针)\n")
    f.write("="*220 + "\n\n")
    
    f.write(
        f"{'Source':<10} {'M0':<12} {'Flux':<14} {'MHI':<20} "
        f"{'MHI_up':<20} {'MHI_low':<20} "
        f"{'<NHI>':<22} {'maxNHI':<22} {'totalNHI':<22} "
        f"{'Maj(deg)':<12} {'Min(deg)':<12} {'Req(deg)':<12} "
        f"{'Maj(arcsec)':<12} {'Min(arcsec)':<12} {'Req(arcsec)':<12} "
        f"{'PA(deg)':<10} {'X_center':<10} {'Y_center':<10}\n")
    f.write("-"*220 + "\n")

    for row in results:
        (src, m0, flux, m0v, mpv, mmv,
         mnhi, mxnhi, tnhi,
         maj_deg, min_deg, req_deg,
         maj_asec, min_asec, req_asec,
         pa, cx, cy) = row

        # ✅ 全部输出完整小数，不使用科学计数法
        f.write(
            f"{src:<10} {m0:<12.2f} {flux:<14.4f} {m0v:<20.0f} "
            f"{mpv:<20.0f} {mmv:<20.0f} "
            f"{mnhi:<22.0f} {mxnhi:<22.0f} {tnhi:<22.0f} "
            f"{maj_deg:<12.4f} {min_deg:<12.4f} {req_deg:<12.4f} "
            f"{maj_asec:<12.1f} {min_asec:<12.1f} {req_asec:<12.1f} "
            f"{pa:<10.1f} {cx:<10.1f} {cy:<10.1f}\n")

print(f"\n✅ 全部完成！结果已保存到：\n{out_path}")

# ===================== 打印汇总 =====================
print("\n" + "="*100)
print("结果汇总")
print("="*100)
print(f"{'Source':<12} {'MHI (M☉)':<16} {'Major (deg)':<12} {'Minor (deg)':<12} {'PA (deg)':<12}")
print("-"*100)
for row in results:
    print(f"{row[0]:<12} {row[3]:<16.2e} {row[9]:<12.4f} {row[10]:<12.4f} {row[15]:<12.1f}")
print("="*100)

# ===================== PA 说明 =====================
print("\n" + "="*100)
print("位置角 (PA) 说明:")
print("="*100)
print("  PA = 0°   : 长轴指向正北")
print("  PA = 45°  : 长轴指向东北方向 (东偏北45°)")
print("  PA = 90°  : 长轴指向正东")
print("  PA = -45° : 长轴指向西北方向 (西偏北45°)")
print("  PA = -90° : 长轴指向正西")
print("="*100)


处理：source_001
Moment-0 总和: 80127.67 K·km/s
总流量: 945.0740 Jy / (K beam) Jy·km/s
HI 质量: 5.59e+05 Jy solMass / (K beam) M☉
平均柱密度: 3.13e+19 cm⁻²
最大柱密度: 8.11e+19 cm⁻²

椭圆拟合结果 (1 pixel = 0.0247 deg):
  长轴: 1.5289 deg (5503.9 arcsec)
  短轴: 1.2220 deg (4399.4 arcsec)
  等效半径: 1.3669 deg (4920.8 arcsec)
  位置角 (PA): -18.1° (范围: -90° ~ 90°, 0°=正北)
    正值: 东偏北 (顺时针)
    负值: 西偏北 (逆时针)
  中心: (x=48.4, y=36.7) pixels

处理：source_002
Moment-0 总和: 1364.99 K·km/s
总流量: 16.0995 Jy / (K beam) Jy·km/s
HI 质量: 9.52e+03 Jy solMass / (K beam) M☉
平均柱密度: 2.35e+19 cm⁻²
最大柱密度: 3.37e+19 cm⁻²

椭圆拟合结果 (1 pixel = 0.0247 deg):
  长轴: 0.2374 deg (854.5 arcsec)
  短轴: 0.1246 deg (448.4 arcsec)
  等效半径: 0.1720 deg (619.0 arcsec)
  位置角 (PA): 26.3° (范围: -90° ~ 90°, 0°=正北)
    正值: 东偏北 (顺时针)
    负值: 西偏北 (逆时针)
  中心: (x=117.2, y=7.1) pixels

处理：source_003
Moment-0 总和: 100.27 K·km/s
总流量: 1.1827 Jy / (K beam) Jy·km/s
HI 质量: 6.99e+02 Jy solMass / (K beam) M☉
平均柱密度: 1.41e+19 cm⁻²
最大柱密度: 1.71e+19 cm⁻²

椭圆拟合结果 (1 pixel = 0.0247 deg):
  长轴: